# 0.706 | GLM Fable Routing + 256-Token Likelihood

**Public LB 0.706** · lightningv08's 0.704 with the GLM likelihood stage reading the first 256 tokens of each
passage instead of 160 · T4 ×2 · internet off · every input is public

**In short.** lightningv08's [0.704](https://www.kaggle.com/code/lightningv08/0-704-glm-fable-routing-qwen3-8-27b-lora) ends with estopadilla's GLM likelihood stage: the ~15% of test
passages that look most like GLM (plus a capped share of claude-fable-5 probability) are scored under the real
GLM-5.3-Flash and GLM-5.3, and a logistic head turns per-token statistics of the **first 160 tokens** into class
log-probabilities. This notebook changes only that cut: the head is refitted with estopadilla's exact recipe on the
first **256 tokens**, and the stage reads 256 tokens per passage. Every other code cell is the 0.704's, and the code
cells below are byte for byte the version that scored 0.706.

## What changed (two cells)

* **The head** (the cell that writes `ll_model.json`): estopadilla's recipe, unchanged: the same 54 likelihood features
  (per model: token log-probability mean, spread and quantiles, next-token entropy, Fast-DetectGPT curvature, rank
  statistics, bits per character, ...; plus six Flash − GLM-5.3 differences), StandardScaler + multinomial logistic
  regression, C = 0.05, blend weight W = 1. Fitted on the 2,699 train passages cut at 256 tokens instead of 160, so
  `max_text_tokens` = 256. `v1_stage.py` passes that cut to the GLM runner and asserts that the test is cut exactly like
  the head's training features.
* **The token budget** (`v1_stage.py`): at most 130,000 scored token positions instead of 90,000, so that the 15% cover
  still binds with longer passages (about 100,000 positions expected on ~4,000 passages). lightningv08's earlier
  22% / 130,000 apex-style version ran within the time limit with the same budget.

## Why a longer cut

* 38% of the train passages are longer than 160 tokens (median 130; 11% are longer than 256), so the 160 cut drops the
  tail of more than a third of the passages the stage reads.
* Out of fold (5 folds grouped by prompt), the head alone gets better with each longer cut up to 320: 18-class accuracy 0.190 at
  160 tokens → 0.200 (192) → 0.210 (224) → 0.215 (256) → 0.222 (320); on passages longer than 160 tokens 0.204 →
  0.241 at 256.
* After the stage (rows chosen as in the 0.704, stage moves out of GLM reverted as its post-processing does), 256
  tokens scored +0.0015 [-0.0004, +0.0033] balanced accuracy over 160 on a stand-in stack. Small, because the stage only decides a few
  hundred rows.
* The test-time GLM scores are noisier than any train run: the GGUF mixture-of-experts passes are batch-dependent, and
  the same text can get per-token log-probabilities several nats apart in two runs. Statistics averaged over more tokens
  are steadier, a plausible reason why the leaderboard gain (+0.002) is larger than the out-of-fold one.

## Leaderboard A/B

| Notebook | Public LB |
|---|---:|
| lightningv08's [0.702 Soft Typography](https://www.kaggle.com/code/lightningv08/0-702-soft-typography-glm-qwen3-8-27b-lora): the shared base | 0.702 |
| lightningv08's [0.704 GLM Fable Routing](https://www.kaggle.com/code/lightningv08/0-704-glm-fable-routing-qwen3-8-27b-lora): the base of this notebook | 0.704 |
| bill123987's [ntl706w](https://www.kaggle.com/code/bill123987/ntl706w): the 0.702 + the GLM blend at 1.25 W | 0.704 |
| ours: the 0.704 + blend weight 1.25 W (bill123987's ntl706w change) | 0.704 |
| ours: the 0.704 + blend weight 1.5 W | 0.704 |
| ours: the 0.704 + 1.25 W + typography penalty 0.05 | 0.704 |
| ours: the 0.704 + 1.25 W without the Fable routing | 0.703 |
| ours: the 0.704 + a boosted-tree GLM head | 0.702 |
| **this notebook**: the 0.704 + the GLM stage reading 256 tokens | **0.706** |

Each row is one submission. On the 0.704 the stage's blend weight and the typography penalty did not move the public
score; the longer cut did. The public board scores part of the test set, and identical code has scored ±0.001 apart
here, so read single 0.001 steps with care.

## How the head was fitted

The train likelihoods come from running estopadilla's `ntl-segrun` (same prompt prefix, seq-batch 8) on all 2,699
train passages under both models, cut at 512 tokens, keeping the per-token scores. The head is then:

```python
from features import one            # the stage's own features.py (cell below)
f = [one(s[:256], len(t)) for s, t in zip(flash_scores, texts)]    # likewise g for GLM-5.3
X = [flash:<24 features>, glm53:<24 features>, d:<6 differences>]   # the column layout of ll_model.json
sc = StandardScaler().fit(X)
lr = LogisticRegression(C=0.05, max_iter=5000).fit(sc.transform(X), y)
```

At 160 tokens the same code reproduces estopadilla's published head (coefficient correlation 0.994; the remaining gap
is the GLM runs' batch noise), so the cut is the only change.

## How it works

1. **The stack** (lightningv08's pipeline): multi-depth LoRA read-outs of Qwen3-14B, Qwen3-8B, Llama-3.1-8B and
   Qwen2.5-14B, the Qwen3-14B adapters of nuridudak and Nishant Dahal, CHEN Han's five-fold Qwen3-Embedding-8B expert,
   lightningv08's Qwen3.8-27B classifier, a DeBERTa reader, soft Sinkhorn balancing.
2. **The GLM stage** (estopadilla's, with the 0.704's routing): passages ranked by GLM probability plus a capped share
   of claude-fable-5 probability are scored under the real GLM-5.3-Flash and GLM-5.3 (jamesliebel's IQ3_XXS GGUF
   segments, estopadilla's `ntl-segrun`), **first 256 tokens**. The refitted head turns the two models' likelihood
   statistics into class log-probabilities, blended in at W.
3. **Apex post-processing** (lumibee's sibling residual and typography rules, lightningv08's soft version and Fable
   routing): a within-family style residual, then three typographic habits that make some labels unlikely.

## Credits

* **lightningv08 (Vladislav Lavrentev)**: the [0.704](https://www.kaggle.com/code/lightningv08/0-704-glm-fable-routing-qwen3-8-27b-lora) this notebook runs, and the pipeline behind it.
* **José Padilla (estopadilla)**: the [GLM likelihood stage](https://www.kaggle.com/code/estopadilla/0-689-real-glm-5-3-likelihood-on-2xt4), `ntl-segrun`, the features and the
  [likelihood head recipe](https://www.kaggle.com/code/estopadilla/ntl-glm-likelihood-train) refitted here. **jamesliebel**: the public GLM-5.3 / GLM-5.3-Flash GGUF segment
  mirrors.
* **bill123987**: [ntl706w](https://www.kaggle.com/code/bill123987/ntl706w), whose stage change (1.25 W) is in the A/B table above.
* Everyone credited in lightningv08's credits section below; each input keeps its own licence. **Built with Llama.**
  Llama 3.1 is licensed under the [Llama 3.1 Community License](https://www.llama.com/llama3_1/license/)
  (Copyright © Meta Platforms, Inc.) and its [Acceptable Use Policy](https://www.llama.com/llama3_1/use-policy/).

## Running it

Attach the inputs below, choose **GPU T4 ×2** and Internet **off**, then *Save Version → Save & Run All* and submit
that version. The three visible test rows only run a fast format preview and the input checks; Kaggle's rerun on the
hidden rows runs everything.


# Apex and GLM likelihood calibration

This notebook builds on [the Claude Fable routing ensemble, V7](https://www.kaggle.com/code/lightningv08/0-702-apex-style-15-glm-qwen3-8-27b-lora?scriptVersionId=370826759). It retains the LoRA and encoder readers, the five-fold embedding expert, the Qwen3.8-27B adapter, and GLM-5.3 and GLM-5.3-Flash token likelihood. GLM work is capped at 15% of texts and 90,000 estimated token positions.

Use post-likelihood probabilities only for typography checks on GLM winners.


In [ ]:
%%writefile pinned_assets.py
"""Explicit paths from the pinned attachment inventory; no model discovery."""
from pathlib import Path
import os
MOUNT_ALIASES = {'/kaggle/input/glm53-iq3xxs-segs-a-mirror': '/kaggle/input/datasets/jamesliebel/glm53-iq3xxs-segs-a-mirror', '/kaggle/input/glm53-iq3xxs-segs-b-mirror': '/kaggle/input/datasets/jamesliebel/glm53-iq3xxs-segs-b-mirror', '/kaggle/input/glm53-flash-iq3xxs-segs-mirror': '/kaggle/input/datasets/jamesliebel/glm53-flash-iq3xxs-segs-mirror', '/kaggle/input/nlm-l24-adapters': '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters', '/kaggle/input/ntl-olmo2-native24-final-inference-assets': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets', '/kaggle/input/name-that-llm-open-pool': '/kaggle/input/datasets/crtproduction/name-that-llm-open-pool', '/kaggle/input/ntl-extra-api-passages': '/kaggle/input/datasets/seyitkaangunes/ntl-extra-api-passages', '/kaggle/input/ntl-generations': '/kaggle/input/datasets/abdullah1afify/ntl-generations', '/kaggle/input/ntl-proxy-generations': '/kaggle/input/datasets/dophuchung/ntl-proxy-generations', '/kaggle/input/ntl-qwen3-8b-lora-claude-head': '/kaggle/input/datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head', '/kaggle/input/nlm-matched-clean': '/kaggle/input/datasets/dhruvtibarewal/nlm-matched-clean', '/kaggle/input/nlm-offline-wheels': '/kaggle/input/datasets/dhruvtibarewal/nlm-offline-wheels', '/kaggle/input/ntl-atlas-qwen3-8b-v1': '/kaggle/input/datasets/josephadamski91/ntl-atlas-qwen3-8b-v1', '/kaggle/input/ntl-free-proxy-generations': '/kaggle/input/datasets/lucasboesen/ntl-free-proxy-generations', '/kaggle/input/ntl-q14-lora': '/kaggle/input/datasets/nuridudak/ntl-q14-lora', '/kaggle/input/ntl-q14-lora-v4': '/kaggle/input/datasets/nuridudak/ntl-q14-lora-v4', '/kaggle/input/ntl-spectral-style-heads-paired-4404': '/kaggle/input/datasets/kirilltym/ntl-spectral-style-heads-paired-4404', '/kaggle/input/ntl-svc-prefit-pool-v4n': '/kaggle/input/datasets/nuridudak/ntl-svc-prefit-pool-v4n', '/kaggle/input/nlm-enc-v5': '/kaggle/input/datasets/dhruvtibarewal/nlm-enc-v5', '/kaggle/input/ntl-two-stage-encoders': '/kaggle/input/datasets/nuridudak/ntl-two-stage-encoders', '/kaggle/input/ntl-two-stage-lora': '/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora', '/kaggle/input/ntl-handwriting-blend': '/kaggle/input/datasets/nishantdahal/ntl-handwriting-blend', '/kaggle/input/nlm-l24-adapters-v5': '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5', '/kaggle/input/prvsiyans-extended-name-that-llm-dataset': '/kaggle/input/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset', '/kaggle/input/qwen35-9b-safetensors': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors', '/kaggle/input/ultrabert-380m-ntl': '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl', '/kaggle/input/name-that-llm-glm4-9b-trunc25': '/kaggle/input/notebooks/lucasboesen/name-that-llm-glm4-9b-trunc25', '/kaggle/input/name-that-llm-extract-encoders2': '/kaggle/input/notebooks/lucasboesen/name-that-llm-extract-encoders2', '/kaggle/input/name-that-llm-electra-large-weights': '/kaggle/input/notebooks/lucasboesen/name-that-llm-electra-large-weights', '/kaggle/input/name-that-llm-desklib-weights': '/kaggle/input/notebooks/lucasboesen/name-that-llm-desklib-weights', '/kaggle/input/name-that-llm-train-lora24-s7': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-lora24-s7', '/kaggle/input/name-that-llm-train-tokattn': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn', '/kaggle/input/name-that-llm-train-tokattn2': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn2', '/kaggle/input/ntl-train-lora-qwen25-instyle': '/kaggle/input/notebooks/vnton37nguyn/ntl-train-lora-qwen25-instyle', '/kaggle/input/ntl-glm-likelihood-train': '/kaggle/input/notebooks/estopadilla/ntl-glm-likelihood-train', '/kaggle/input/llama-3-1-8b-base': '/kaggle/input/models/alexchilton/llama-3-1-8b-base', '/kaggle/input/qwen-3': '/kaggle/input/models/qwen-lm/qwen-3', '/kaggle/input/qwen-3-embedding': '/kaggle/input/models/qwen-lm/qwen-3-embedding', '/kaggle/input/qwen2.5': '/kaggle/input/models/qwen-lm/qwen2.5'}
PATHS = {'/kaggle/input/**/': [], '/kaggle/input/**/bitsandbytes-*.whl': ['/kaggle/input/nlm-offline-wheels/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl'], '/kaggle/input/**/config.json': ['/kaggle/input/models/alexchilton/llama-3-1-8b-base/transformers/default/1/config.json', '/kaggle/input/models/qwen-lm/qwen-3-embedding/transformers/8b/1/config.json', '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1/config.json', '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b/1/config.json', '/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1/config.json', '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1/config.json', '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1/config.json', '/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1/config.json', '/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b/1/config.json', '/kaggle/input/name-that-llm-desklib-weights/desklib-detector/config.json', '/kaggle/input/name-that-llm-electra-large-weights/electra-large-discriminator/config.json', '/kaggle/input/name-that-llm-extract-encoders2/deberta-v3-large/config.json', '/kaggle/input/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25/config.json', '/kaggle/input/nlm-enc-v5/encoder/config.json', '/kaggle/input/ntl-handwriting-blend/deberta_v3_large/encoder/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/document-deberta-07nov/document-deberta-07nov/final/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/document-modernbert-07nov/document-modernbert-07nov/final/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq3-base/qwen38-27b-gsq3/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/llama31-prefix24/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-tabular-offline-assets/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-27b-nf4/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/ntl-lora24-fivefold-training/lora24/final/base_config/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/olmo2-7b-fp16-offline-weights/olmo2-fp16-base/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679/base/1_Pooling/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679/base/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl-native-wemm4-fivefold-0679/base/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/config.json', '/kaggle/input/ntl-two-stage-encoders/desklib/config.json', '/kaggle/input/ntl-two-stage-encoders/modernbert/config.json', '/kaggle/input/qwen35-9b-safetensors/config.json', '/kaggle/input/ultrabert-380m-ntl/fold0/config.json', '/kaggle/input/ultrabert-380m-ntl/fold1/config.json', '/kaggle/input/ultrabert-380m-ntl/fold2/config.json', '/kaggle/input/ultrabert-380m-ntl/fold3/config.json', '/kaggle/input/ultrabert-380m-ntl/fold4/config.json'], '/kaggle/input/**/deberta-v3-large/config.json': ['/kaggle/input/name-that-llm-extract-encoders2/deberta-v3-large/config.json'], '/kaggle/input/**/deberta_v3_large/meta.json': ['/kaggle/input/ntl-handwriting-blend/deberta_v3_large/meta.json'], '/kaggle/input/**/desklib-detector/config.json': ['/kaggle/input/name-that-llm-desklib-weights/desklib-detector/config.json'], '/kaggle/input/**/electra-large-discriminator/config.json': ['/kaggle/input/name-that-llm-electra-large-weights/electra-large-discriminator/config.json'], '/kaggle/input/**/gen_passages.csv': ['/kaggle/input/datasets/lucasboesen/ntl-free-proxy-generations/gen_passages.csv'], '/kaggle/input/**/ll8v5/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/ll8v5/heads.pt'], '/kaggle/input/**/ll8v5_f0/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/ll8v5_f0/heads.pt'], '/kaggle/input/**/ll8v5_f1/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/ll8v5_f1/heads.pt'], '/kaggle/input/**/ll8v5_f2/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/ll8v5_f2/heads.pt'], '/kaggle/input/**/ll8v5_f3/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/ll8v5_f3/heads.pt'], '/kaggle/input/**/ll8v5_f4/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/ll8v5_f4/heads.pt'], '/kaggle/input/**/lora24/final/COMPLETE.json': ['/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/ntl-lora24-fivefold-training/lora24/final/COMPLETE.json'], '/kaggle/input/**/lora_qwen25_14b/meta.json': ['/kaggle/input/ntl-handwriting-blend/lora_qwen25_14b/meta.json'], '/kaggle/input/**/lora_qwen3_14b/meta.json': ['/kaggle/input/ntl-handwriting-blend/lora_qwen3_14b/meta.json'], '/kaggle/input/**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json': ['/kaggle/input/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json'], '/kaggle/input/**/nlm_matched_clean.parquet': ['/kaggle/input/datasets/dhruvtibarewal/nlm-matched-clean/nlm_matched_clean.parquet'], '/kaggle/input/**/ntl-native-q8-multidepth-fivefold-0679/manifest.json': ['/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679/manifest.json'], '/kaggle/input/**/ntl-q14-lora-v4/heads.safetensors': ['/kaggle/input/ntl-q14-lora-v4/heads.safetensors'], '/kaggle/input/**/ntl-q14-lora/heads.safetensors': ['/kaggle/input/ntl-q14-lora/heads.safetensors'], '/kaggle/input/**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json': ['/kaggle/input/ntl-qwen3-8b-lora-claude-head/final/COMPLETE.json'], '/kaggle/input/**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json': ['/kaggle/input/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json'], '/kaggle/input/**/ntl666-cache-parts.json': ['/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl666-cache-parts.json'], '/kaggle/input/**/ntl_extra_passages_claude.csv': ['/kaggle/input/datasets/seyitkaangunes/ntl-extra-api-passages/ntl_extra_passages_claude.csv'], '/kaggle/input/**/ntl_generated*.csv': [], '/kaggle/input/**/ntl_generations.csv': ['/kaggle/input/datasets/abdullah1afify/ntl-generations/ntl_generations.csv'], '/kaggle/input/**/ntl_open_pool.parquet': ['/kaggle/input/name-that-llm-open-pool/ntl_open_pool.parquet'], '/kaggle/input/**/ntl_proxy_generations.csv': ['/kaggle/input/datasets/dophuchung/ntl-proxy-generations/ntl_proxy_generations.csv'], '/kaggle/input/**/passages_clean.csv': ['/kaggle/input/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset/passages_clean.csv'], '/kaggle/input/**/q14v5/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q14v5/heads.pt'], '/kaggle/input/**/q14v5_f0/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q14v5_f0/heads.pt'], '/kaggle/input/**/q14v5_f1/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q14v5_f1/heads.pt'], '/kaggle/input/**/q14v5_f2/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q14v5_f2/heads.pt'], '/kaggle/input/**/q14v5_f3/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q14v5_f3/heads.pt'], '/kaggle/input/**/q14v5_f4/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q14v5_f4/heads.pt'], '/kaggle/input/**/q8v5/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q8v5/heads.pt'], '/kaggle/input/**/q8v5_f0/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q8v5_f0/heads.pt'], '/kaggle/input/**/q8v5_f1/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q8v5_f1/heads.pt'], '/kaggle/input/**/q8v5_f2/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q8v5_f2/heads.pt'], '/kaggle/input/**/q8v5_f3/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q8v5_f3/heads.pt'], '/kaggle/input/**/q8v5_f4/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q8v5_f4/heads.pt'], '/kaggle/input/**/tokattn_debl23_final.pt': ['/kaggle/input/name-that-llm-train-tokattn2/tokattn_debl23_final.pt'], '/kaggle/input/**/tokattn_final.pt': ['/kaggle/input/name-that-llm-train-tokattn/tokattn_final.pt'], '/kaggle/input/**/ultrabert-380m-ntl': ['/kaggle/input/ultrabert-380m-ntl'], '/kaggle/input/ultrabert-380m-ntl/fold*': ['/kaggle/input/ultrabert-380m-ntl/fold0', '/kaggle/input/ultrabert-380m-ntl/fold1', '/kaggle/input/ultrabert-380m-ntl/fold2', '/kaggle/input/ultrabert-380m-ntl/fold3', '/kaggle/input/ultrabert-380m-ntl/fold4']}
CHECKPOINTS = {('llama', None, None): '/kaggle/input/models/alexchilton/llama-3-1-8b-base/transformers/default/1', ('llama', 4096, None): '/kaggle/input/models/alexchilton/llama-3-1-8b-base/transformers/default/1', (None, None, '1'): '/kaggle/input/models/alexchilton/llama-3-1-8b-base/transformers/default/1', ('qwen3', None, None): '/kaggle/input/models/qwen-lm/qwen-3-embedding/transformers/8b/1', ('qwen3', 4096, None): '/kaggle/input/models/qwen-lm/qwen-3-embedding/transformers/8b/1', ('qwen3', 5120, None): '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1', ('qwen3', 2560, None): '/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1', ('qwen2', None, None): '/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1', ('qwen2', 5120, None): '/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1', ('qwen2', 3584, None): '/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b/1', ('deberta-v2', None, None): '/kaggle/input/name-that-llm-desklib-weights/desklib-detector', ('deberta-v2', 1024, None): '/kaggle/input/name-that-llm-desklib-weights/desklib-detector', (None, None, 'desklib-detector'): '/kaggle/input/name-that-llm-desklib-weights/desklib-detector', ('electra', None, None): '/kaggle/input/name-that-llm-electra-large-weights/electra-large-discriminator', ('electra', 1024, None): '/kaggle/input/name-that-llm-electra-large-weights/electra-large-discriminator', (None, None, 'electra-large-discriminator'): '/kaggle/input/name-that-llm-electra-large-weights/electra-large-discriminator', (None, None, 'deberta-v3-large'): '/kaggle/input/name-that-llm-extract-encoders2/deberta-v3-large', ('glm4', None, None): '/kaggle/input/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25', ('glm4', 4096, None): '/kaggle/input/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25', (None, None, 'glm4_9b_trunc25'): '/kaggle/input/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25', ('modernbert', None, None): '/kaggle/input/nlm-enc-v5/encoder', ('modernbert', 1024, None): '/kaggle/input/nlm-enc-v5/encoder', (None, None, 'encoder'): '/kaggle/input/nlm-enc-v5/encoder', ('deberta-v2', 768, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/document-deberta-07nov/document-deberta-07nov/final', (None, None, 'final'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/document-deberta-07nov/document-deberta-07nov/final', ('qwen3_5', None, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq3-base/qwen38-27b-gsq3', (None, None, 'qwen38-27b-gsq3'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq3-base/qwen38-27b-gsq3', (None, None, 'llama31-prefix24'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/llama31-prefix24', (None, None, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-tabular-offline-assets', (None, None, 'ntl-tabular-offline-assets'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-tabular-offline-assets', (None, None, 'qwen38-27b-nf4'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-27b-nf4', (None, None, 'base_config'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/ntl-lora24-fivefold-training/lora24/final/base_config', ('olmo2', None, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/olmo2-7b-fp16-offline-weights/olmo2-fp16-base', ('olmo2', 4096, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/olmo2-7b-fp16-offline-weights/olmo2-fp16-base', (None, None, 'olmo2-fp16-base'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/olmo2-7b-fp16-offline-weights/olmo2-fp16-base', ('smollm3', None, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base', ('smollm3', 2048, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base', (None, None, 'smollm3-base'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base', (None, None, '1_Pooling'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679/base/1_Pooling', (None, None, 'base'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679/base', (None, None, 'qwen38-27b-mixed-tail8-text'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text', (None, None, 'desklib'): '/kaggle/input/ntl-two-stage-encoders/desklib', (None, None, 'modernbert'): '/kaggle/input/ntl-two-stage-encoders/modernbert', (None, None, 'qwen35-9b-safetensors'): '/kaggle/input/qwen35-9b-safetensors', ('ultrabert', None, None): '/kaggle/input/ultrabert-380m-ntl/fold0', ('ultrabert', 1024, None): '/kaggle/input/ultrabert-380m-ntl/fold0', (None, None, 'fold0'): '/kaggle/input/ultrabert-380m-ntl/fold0', (None, None, 'fold1'): '/kaggle/input/ultrabert-380m-ntl/fold1', (None, None, 'fold2'): '/kaggle/input/ultrabert-380m-ntl/fold2', (None, None, 'fold3'): '/kaggle/input/ultrabert-380m-ntl/fold3', (None, None, 'fold4'): '/kaggle/input/ultrabert-380m-ntl/fold4'}

def asset_path(value):
    if Path(value).exists():
        return value
    for flat,namespaced in MOUNT_ALIASES.items():
        for origin,target in ((flat,namespaced),(namespaced,flat)):
            if value==origin or value.startswith(origin+'/'):
                resolved=target+value[len(origin):]
                assert Path(resolved).exists(), ('missing exact input',value,resolved)
                return resolved
    return value

def asset_paths(key, recursive=False):
    values=[asset_path(v) for v in PATHS[key]]
    for value in values:
        assert Path(value).exists(), value
    return values.copy()

def checkpoint(model_type, hidden=None, dir_name=None):
    path=asset_path(CHECKPOINTS[(model_type,hidden,dir_name)])
    assert Path(path,'config.json').is_file(), path
    return path


In [ ]:
from pinned_assets import asset_path,asset_paths
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd

NB_T0 = time.time()
FINAL_LOGITS = '/kaggle/temp/final-logits.npz'

wheels = [asset_path('/kaggle/input/nlm-offline-wheels/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl')]
if wheels:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '-q'] + wheels, capture_output=True, text=True, check=True)

from pinned_assets import asset_paths

def first(pattern):
    values=asset_paths(pattern)
    assert values, pattern
    return values[0]

def adapter_dir(name):
    return os.path.dirname(first(f'/kaggle/input/**/{name}/heads.pt'))

def meta_dir(name):
    filename=first(f'/kaggle/input/**/{name}/meta.json')
    return os.path.dirname(filename),json.load(open(filename))['cfg']


DATA = next(p for p in ['/kaggle/input/competitions/name-that-llm', '/kaggle/input/name-that-llm'] if os.path.exists(p + '/train.csv'))
FAST_COMMIT = os.getenv('KAGGLE_IS_COMPETITION_RERUN') is None and len(pd.read_csv(DATA + '/test.csv')) < 50   # ours: the scoring rerun never takes the fast path
print('FAST_COMMIT', FAST_COMMIT)

open_pool = pd.read_parquet(first('/kaggle/input/**/ntl_open_pool.parquet'), columns=['text', 'model', 'flag_train_prompt_regen', 'match_weight'])
open_pool = open_pool[~open_pool.flag_train_prompt_regen.astype(bool)]
open_pool[['text', 'model']].to_parquet('/kaggle/working/open_pool.parquet')
POOL = '/kaggle/working/open_pool.parquet'
print('svc pool rows', len(open_pool))


def draw(g, n):
    w = np.nan_to_num(g.match_weight.to_numpy(dtype=np.float64))
    return g.take(np.random.RandomState(0).choice(len(g), size=n, replace=False, p=w / w.sum()).astype(np.intp))


matched_counts = pd.read_parquet(first('/kaggle/input/**/nlm_matched_clean.parquet'), columns=['model']).model.value_counts()
extra = pd.concat([draw(g, min(len(g), int(matched_counts.get(m, 0)))) for m, g in open_pool.groupby('model')])
EXTRA = '/kaggle/working/extra_matched.parquet'
extra[['text', 'model']].to_parquet(EXTRA)
print('external rows', len(extra), 'matched_clean rows', int(matched_counts.sum()))
del open_pool, extra

B8 = asset_path('/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1')
B14 = asset_path('/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1')
B25_14 = asset_path('/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1')
BL = asset_path('/kaggle/input/models/alexchilton/llama-3-1-8b-base/transformers/default/1')
SHARDS = {'model-00001-of-00004.safetensors': 4976698672, 'model-00002-of-00004.safetensors': 4999802720,
          'model-00003-of-00004.safetensors': 4915916176, 'model-00004-of-00004.safetensors': 1168138808}
assert all(os.path.getsize(os.path.join(BL, f)) == n for f, n in SHARDS.items()), BL
if not FAST_COMMIT:
    shutil.copytree(BL, '/tmp/llama-3.1/8b', dirs_exist_ok=True)
    BL = '/tmp/llama-3.1/8b'

AQ8V5 = [adapter_dir(f'q8v5_f{k}') for k in range(5)] + [adapter_dir('q8v5')]
AQ14V5 = [adapter_dir(f'q14v5_f{k}') for k in range(5)] + [adapter_dir('q14v5')]
ALL8V5 = [adapter_dir(f'll8v5_f{k}') for k in range(5)] + [adapter_dir('ll8v5')]
N14B, N14BC = meta_dir('lora_qwen3_14b')
Q25_14, Q25_14C = meta_dir('lora_qwen25_14b')
OQ1 = os.path.dirname(first('/kaggle/input/**/ntl-q14-lora/heads.safetensors'))
OQ4 = os.path.dirname(first('/kaggle/input/**/ntl-q14-lora-v4/heads.safetensors'))

NATIVE = os.path.dirname(first('/kaggle/input/**/ntl-native-q8-multidepth-fivefold-0679/manifest.json'))
NATIVE_BASE = asset_path('/kaggle/input/models/qwen-lm/qwen-3-embedding/transformers/8b/1')
assert all(os.path.exists(f'{NATIVE}/fold{k}/trainables.safetensors') and os.path.exists(f'{NATIVE}/calibration/fold{k}.json') for k in range(5)), NATIVE
assert os.path.exists(f'{NATIVE}/base/model.safetensors.index.json') and os.path.exists(f'{NATIVE_BASE}/model.safetensors.index.json'), (NATIVE, NATIVE_BASE)

LB_ROOT = os.path.dirname(first('/kaggle/input/**/ntl666-cache-parts.json'))
GSQ_ASSETS = LB_ROOT
PREFIT = '/kaggle/temp/ntl666-prefit'
import zipfile
from pathlib import PurePosixPath
parts_meta = json.loads(open(f'{LB_ROOT}/ntl666-cache-parts.json').read())
assert parts_meta['format'] == 'ntl666-cache-parts-v1'
assert [x['name'] for x in parts_meta['parts']] == [f'ntl666-prefit.part{i:03d}.bin' for i in range(len(parts_meta['parts']))]
os.makedirs('/kaggle/temp', exist_ok=True)
shutil.rmtree(PREFIT, ignore_errors=True)
prefit_zip = '/kaggle/temp/ntl666-prefit.zip'
try:
    with open(prefit_zip, 'wb') as joined:
        for part in parts_meta['parts']:
            assert os.path.getsize(f"{LB_ROOT}/{part['name']}") == part['bytes'], part['name']
            with open(f"{LB_ROOT}/{part['name']}", 'rb') as stream:
                shutil.copyfileobj(stream, joined, 16 * 1024 * 1024)
    assert os.path.getsize(prefit_zip) == parts_meta['archive_bytes']
    with zipfile.ZipFile(prefit_zip) as archive:
        entries = archive.infolist()
        assert len(entries) == parts_meta['files'] and len({e.filename for e in entries}) == len(entries)
        for entry in entries:
            path = PurePosixPath(entry.filename)
            assert not path.is_absolute() and '..' not in path.parts and chr(92) not in entry.filename and path.parts[0] == 'ntl666-prefit' and not entry.is_dir()
        assert archive.read('ntl666-prefit/COMPLETE') == b'ntl666-v14-data-numeric-v1' + chr(10).encode()
        provenance = json.loads(archive.read('ntl666-prefit/provenance.json'))
        assert provenance['status'] == 'COMPLETE' and provenance['pool_rows'] == 190588 and provenance['dataset_version'] == 13
        archive.extractall('/kaggle/temp')
finally:
    if os.path.exists(prefit_zip):
        os.remove(prefit_zip)
for name in ('core/svc4', 'core/tfidf', 'core/distort'):
    assert os.path.isdir(f'{PREFIT}/{name}'), name
assert os.path.exists(f'{GSQ_ASSETS}/gsq3-base/qwen38-27b-gsq3/config.json') and os.path.exists(f'{GSQ_ASSETS}/gsq120k-trained/gsq120k-inference/gsq/final.safetensors')
assert len([asset_path(p) for p in ['/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/accelerate-1.15.0-py3-none-any.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/huggingface_hub-1.32.0-py3-none-any.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/transformers-5.14.1-py3-none-any.whl']]) >= 6

DEB = os.path.dirname(first('/kaggle/input/**/deberta_v3_large/meta.json'))
if not FAST_COMMIT:
    shutil.copytree(DEB, '/kaggle/working/deberta_v3_large', dirs_exist_ok=True)
    DEB = '/kaggle/working/deberta_v3_large'

for name, value in dict(B8=B8, B14=B14, B25_14=B25_14, BL=BL, N14B=N14B, Q25_14=Q25_14, OQ1=OQ1, OQ4=OQ4, NATIVE=NATIVE, NATIVE_BASE=NATIVE_BASE, PREFIT=PREFIT, GSQ_ASSETS=GSQ_ASSETS, DEB=DEB).items():
    print(f'{name:10s} {value}')
print('setup', round(time.time() - NB_T0), 's')


In [ ]:
%%writefile pngy_residual.py
"""Mass-preserving sibling log-ratio refinement; operates on post-blend logits."""
import re
import numpy as np
from scipy.special import logsumexp, softmax
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MARKERS = (",", ".", ";", ":", "!", "?", "—", "–", "’", "“", "”",
           "'", '"', "(", ")", "[", "]", "*", "", "**", " - ", "  ")
FUNCTIONS = ("the", "and", "but", "which", "however", "therefore", "moreover", "because", "perhaps",
             "might", "would", "could", "should", "also", "although", "overall", "finally", "we", "you", "i")
WORDS = re.compile(r"[A-Za-z]+(?:['’][A-Za-z]+)?")

def features(texts):
    rows = []
    for value in texts:
        if not isinstance(value, str):
            value = str(value)
        counts = np.array([value.count(m) for m in MARKERS], dtype=np.float64)
        tokens = WORDS.findall(value)
        n, chars = max(1, len(tokens)), max(1, len(value))
        lengths = np.array([len(t) for t in tokens] or [0], dtype=float)
        lower = [t.lower() for t in tokens]
        shape = [np.log1p(len(tokens)), np.log1p(len(value)), chars/n,
                 len(set(lower))/n, lengths.mean(), lengths.std(),
                 sum(c.isupper() for c in value)/chars, sum(c.isdigit() for c in value)/chars,
                 sum(ord(c) > 127 for c in value)/chars, sum(c.isspace() for c in value)/chars]
        rows.append(np.r_[shape, counts/n, [lower.count(w)/n for w in FUNCTIONS]])
    return np.asarray(rows, dtype=np.float64)

def family_columns(labels):
    return [np.array([i for i, label in enumerate(labels) if label.startswith(f + "-")])
            for f in ("claude", "gpt", "gemini", "glm")]

def refine(log_probability, evidence, families, strength=0.10):
    z, style = np.asarray(log_probability, dtype=float), np.asarray(evidence, dtype=float)
    result = z.copy()
    for columns in families:
        if len(columns) < 2:
            continue
        p = softmax(z[:, columns], axis=1)
        q = np.maximum(style[:, columns], 1e-15)
        q /= q.sum(1, keepdims=True)
        top = np.partition(p, -2, axis=1)[:, -2:]
        gate = np.clip((0.65 - (top[:, 1] - top[:, 0])) / 0.65, 0, 1) * strength
        mixed = (1 - gate[:, None]) * p + gate[:, None] * q
        result[:, columns] = logsumexp(z[:, columns], axis=1, keepdims=True) + np.log(mixed)
    return result

class Residual:
    def __init__(self, labels):
        self.labels = list(labels)
        self.families = family_columns(labels)

    def fit(self, texts, target):
        x = features(texts)
        target = np.asarray(target)
        self.models = []
        for columns in self.families:
            if len(columns) < 2:
                continue
            selected = np.isin(target, np.asarray(self.labels)[columns])
            model = make_pipeline(StandardScaler(), LogisticRegression(
                C=0.03, max_iter=500, class_weight="balanced", random_state=166))
            model.fit(x[selected], target[selected])
            self.models.append((columns, model))
        return self

    def predict_log_probability(self, base, texts):
        x = features(texts)
        evidence = np.ones((len(x), len(self.labels)), dtype=float)
        for columns, model in self.models:
            p = model.predict_proba(x)
            for j, label in enumerate(model.classes_):
                evidence[:, self.labels.index(label)] = p[:, j]
        return refine(base, evidence, self.families)



In [ ]:
%%writefile pipe.py
import gc
import os
import threading
import time
from collections import deque
import numpy as np

def _flat(x):
    if isinstance(x, dict):
        parts = [x[k] for k in sorted(x)]
    else:
        parts = x if isinstance(x, (tuple, list)) else [x]
    return np.concatenate([np.ravel(np.asarray(p, dtype=np.float64)) for p in parts])

def _agree(batches, work, workers, tag, log):
    probe = [batches[len(batches) // 2 + i] for i in range(workers)]
    try:
        ref = [work(b) for b in probe]
        if any((r is None for r in ref)):
            return False
        got = [None] * workers
        err = []

        def go(i):
            try:
                got[i] = work(probe[i])
            except Exception as e:
                err.append(e)
        threads = [threading.Thread(target=go, args=(i,)) for i in range(workers)]
        [t.start() for t in threads]
        [t.join() for t in threads]
        ok = not err and all((g is not None and np.abs(_flat(r) - _flat(g)).max() <= 0.02 * (np.abs(_flat(r)).mean() + 1e-06) for r, g in zip(ref, got)))
    except Exception as e:
        log(f'{tag}: concurrent check raised {e!r:.100}')
        ok = False
    log(f"{tag}: concurrent check {('passed' if ok else 'failed, single worker')}")
    return ok

def pipelined(batches, work, store, fallback, workers=2, tag='pipe', log=print):
    batches = list(batches)
    t0 = time.time()
    workers = min(workers, int(os.environ.get('NTL_PIPE_WORKERS', workers)))
    if workers < 2 or len(batches) < 6 or (not _agree(batches, work, workers, tag, log)):
        workers = 1
    queue = deque(batches)
    redo = []
    lock = threading.Lock()
    stop = threading.Event()

    def loop():
        while not stop.is_set():
            with lock:
                if not queue:
                    return
                b = queue.popleft()
            try:
                r = work(b)
            except Exception:
                with lock:
                    redo.append(b)
                stop.set()
                return
            with lock:
                if r is None:
                    redo.append(b)
                else:
                    store(b, r)
    if workers == 1:
        while queue:
            b = queue.popleft()
            r = work(b)
            if r is None:
                redo.append(b)
            else:
                store(b, r)
    else:
        threads = [threading.Thread(target=loop) for _ in range(workers)]
        [t.start() for t in threads]
        [t.join() for t in threads]
    rest = redo + list(queue)
    if rest:
        gc.collect()
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass
    for b in rest:
        fallback(b)
    log(f'{tag}: {len(batches)} batches, {workers} worker(s), {len(redo)} redone sequentially, {time.time() - t0:.0f}s')

def spans_devices(model):
    return len({p.device for p in model.parameters()}) > 1


In [ ]:
%%writefile gauss.py
import itertools
import numpy as np
K = 18
DEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,
           "iters": 5, "repel": True, "sp_scale": 1.0}

def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    from sklearn.covariance import ledoit_wolf_shrinkage
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]
    return mu, P, cnt_p

def fit(X, y, prompts, cfg=None):
    c = dict(DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    if c["red"] == "none":
        V = np.eye(D)
    else:
        if c["red"] == "wpca":
            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]
        elif c["red"] == "pca": R = Xs
        else: raise ValueError(c["red"])
        d = int(min(c["d"], N - G - 1, D))
        if N < D:
            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]
            V = (R.T @ U) / np.sqrt(w)
        else:
            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]
    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li
    W = V @ A.T
    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,
            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}

def transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

_MASK = {}
def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]
        _MASK[(n, C)] = bad
    return _MASK[(n, C)]

def singleton_ll(U, m, lam):
    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)
    return a, B

def joint_table(a, B):
    n, C = a.shape; Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    n = Q.ndim; C = Q.shape[0]; Qt = Q / T
    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt); out = np.empty((n, C))
    for i in range(n):
        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    n = len(U); v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v
        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:
            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)

LAM_DIFF = 1e6
def lam_g(prm, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})
    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)
    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg
    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)
    if n <= c["n_exact"]:
        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])
    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))
    sizes = np.bincount(lab); s1 = sizes[lab] == 1
    T1 = prm["T1"] if T1 is None else T1
    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out


In [ ]:
%%writefile atlas_infer.py
import gc
import json
import os
import re
import threading
import time

import numpy as np
import torch
from safetensors.torch import load_file
from torch import nn
from torch.nn import functional as F

from pipe import pipelined

TEMPERATURES = (1.0, 2.0, 4.0, 8.0)
HEAD_CHUNK_TOKENS = 64


class Route(threading.local):
    labels = None


class SourceLinear(nn.Module):
    def __init__(self, base, sources, rank, alpha, route):
        super().__init__()
        self.base = base
        self.route = route
        self.scaling = alpha / rank
        dev = base.weight.device
        self.a = nn.Parameter(torch.zeros(rank, base.in_features, device=dev, dtype=torch.float32), requires_grad=False)
        self.b = nn.Parameter(torch.zeros(sources, base.out_features, rank, device=dev, dtype=torch.float32), requires_grad=False)

    def forward(self, x):
        labels = self.route.labels
        if labels is None or labels.numel() != x.shape[0]:
            raise ValueError('source label missing for a batch row')
        out = self.base(x)
        delta = torch.bmm(F.linear(x.float(), self.a), self.b.index_select(0, labels.to(self.b.device)).transpose(1, 2))
        return out + (delta * self.scaling).to(out.dtype)


def score_columns(logp, mask):
    lp = logp.float()
    valid = mask.float().unsqueeze(1)
    mean_lp = (lp * valid).sum(-1) / valid.sum(-1).clamp_min(1)
    gap = (lp.unsqueeze(1) - lp.unsqueeze(2)).clamp_min(0).sum(2)
    cols = [mean_lp] + [(torch.exp(-gap / t) * valid).sum(-1) / valid.sum(-1).clamp_min(1) for t in TEMPERATURES]
    return torch.stack(cols, -1)


class AtlasModel(nn.Module):
    def __init__(self, base, labels, rank, alpha):
        super().__init__()
        self.base = base
        self.labels = list(labels)
        self.route = Route()
        self.base.requires_grad_(False)
        self.base.config.use_cache = False
        hit = [(n, m) for n, m in self.base.named_modules() if n.split('.')[-1] in ('q_proj', 'k_proj', 'v_proj', 'o_proj') and isinstance(m, nn.Linear)]
        if not hit:
            raise ValueError('no attention projections found')
        for name, m in hit:
            parent, leaf = name.rsplit('.', 1)
            setattr(self.base.get_submodule(parent), leaf, SourceLinear(m, len(labels), rank, alpha, self.route))
        emb = self.base.get_input_embeddings().weight
        self.control = nn.Parameter(torch.zeros(len(labels), 1, emb.shape[1], device=emb.device, dtype=torch.float32), requires_grad=False)

    @property
    def device(self):
        return self.control.device

    def load_state(self, state):
        own = {n: p for n, p in self.named_parameters() if n.endswith(('.a', '.b')) or n == 'control'}
        if set(own) != set(state):
            raise ValueError(f'adapter key mismatch: {sorted(set(own) ^ set(state))[:6]}')
        with torch.no_grad():
            for n, p in own.items():
                t = state[n]
                if tuple(t.shape) != tuple(p.shape) or not torch.isfinite(t).all():
                    raise ValueError(f'bad adapter tensor {n}')
                p.copy_(t.to(device=p.device, dtype=p.dtype))

    def token_logp(self, ids, mask, labels):
        self.route.labels = labels
        tok = self.base.get_input_embeddings()(ids)
        ctrl = self.control.index_select(0, labels).to(tok.dtype)
        embeds = torch.cat([ctrl, tok], 1)
        attn = torch.cat([mask.new_ones(len(ids), 1), mask], 1)
        hid = self.base.model(inputs_embeds=embeds, attention_mask=attn, use_cache=False).last_hidden_state[:, 1:-1]
        tgt = ids[:, 1:]
        head = self.base.get_output_embeddings()
        parts = []
        for s in range(0, hid.shape[1], HEAD_CHUNK_TOKENS):
            h, y = hid[:, s:s + HEAD_CHUNK_TOKENS], tgt[:, s:s + HEAD_CHUNK_TOKENS]
            logits = head(h.to(head.weight.device)).float()
            parts.append(-F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.to(logits.device).reshape(-1), reduction='none').reshape(y.shape))
        return torch.cat(parts, 1).to(ids.device)

    @torch.no_grad()
    def score(self, ids, mask, chunk=3):
        outs = []
        for first in range(0, len(self.labels), chunk):
            last = min(len(self.labels), first + chunk)
            src = torch.arange(first, last, device=ids.device).repeat_interleave(len(ids))
            lp = self.token_logp(ids.repeat(last - first, 1), mask.repeat(last - first, 1), src)
            outs.append(lp.reshape(last - first, len(ids), -1).transpose(0, 1))
        cols = score_columns(torch.cat(outs, 1), mask[:, 1:])
        return cols, torch.isfinite(cols).flatten(1).all(1)


def split_device_map(base_path, n_gpu, head_weight=2.4):
    from accelerate import init_empty_weights
    from transformers import AutoConfig, AutoModelForCausalLM
    if n_gpu < 2:
        return {'': 0}
    cfg = AutoConfig.from_pretrained(base_path)
    with init_empty_weights():
        meta = AutoModelForCausalLM.from_config(cfg, torch_dtype=torch.float16)
    first = int(round((cfg.num_hidden_layers + head_weight) / 2.0))
    dm = {'model.embed_tokens': 0, 'model.rotary_emb': 0, 'model.norm': 1, 'lm_head': 1}
    for i in range(cfg.num_hidden_layers):
        dm[f'model.layers.{i}'] = 0 if i < first else 1
    tensors = [n for n, _ in meta.named_parameters()] + [n for n, _ in meta.named_buffers()]
    missing = [n for n in tensors if not any(n == k or n.startswith(k + '.') for k in dm)]
    if missing:
        raise RuntimeError(f'device map does not cover {missing[:3]}')
    return dm


class Scorer:
    def __init__(self, base_path, bundle_dir, log=print):
        self.log = log
        meta = json.load(open(os.path.join(bundle_dir, 'atlas_bundle.json'), encoding='utf-8'))
        self.labels = list(meta['labels'])
        cfg = meta['config']
        n_gpu = torch.cuda.device_count()
        t0 = time.time()
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(base_path)
        self.tok.padding_side = 'right'
        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token
        try:
            dm = split_device_map(base_path, n_gpu)
        except Exception as e:
            self.log(f'explicit device map failed ({e!r:.100}), using device_map=auto')
            dm = 'auto'
        base = AutoModelForCausalLM.from_pretrained(base_path, dtype=torch.float16, attn_implementation='sdpa', device_map=dm)
        if any(p.device.type != 'cuda' for p in base.parameters()):
            raise RuntimeError('ATLAS base did not fit on the GPUs')
        self.model = AtlasModel(base, self.labels, cfg['rank'], cfg['alpha'])
        self.model.load_state(load_file(os.path.join(bundle_dir, 'adapters.safetensors')))
        self.model.eval()
        per_dev = {}
        for p in base.parameters():
            per_dev[str(p.device)] = per_dev.get(str(p.device), 0) + p.numel() * p.element_size()
        self.max_len = int(cfg['max_length'])
        head_path = os.path.join(bundle_dir, 'atlas_head.json')
        self.head = json.load(open(head_path)) if os.path.exists(head_path) else None
        lite_path = os.path.join(bundle_dir, 'atlas_head_meanlp.json')
        self.lite = json.load(open(lite_path)) if os.path.exists(lite_path) else None
        self.log(f'atlas loaded in {time.time() - t0:.0f}s, weights per device: ' + ', '.join(f'{k} {v / 2 ** 30:.1f} GiB' for k, v in sorted(per_dev.items())))
        self.n_gpu = n_gpu

    def encode(self, texts):
        return self.tok(list(texts), truncation=True, max_length=self.max_len)['input_ids']

    def _pack(self, enc, rows):
        pk = self.tok.pad({'input_ids': [enc[int(i)] for i in rows]}, padding=True, return_tensors='pt')
        return pk['input_ids'].to(self.model.device), pk['attention_mask'].to(self.model.device)

    def _chunks(self, enc, rows, batch, chunk, max_tokens):
        lens = np.array([len(enc[int(i)]) for i in rows])
        order = np.asarray(rows)[np.argsort(-lens, kind='stable')]
        out, a = [], 0
        while a < len(order):
            width = max(1, len(enc[int(order[a])]))
            size = max(1, min(batch, max_tokens // (chunk * width)))
            out.append(order[a:a + size])
            a += size
        return out

    def _score_rows(self, enc, rows, chunk):
        ids, mask = self._pack(enc, rows)
        p, ok = self.model.score(ids, mask, chunk)
        return p.float().cpu().numpy(), ok.cpu().numpy()

    def run(self, enc, rows, batch=4, chunk=3, max_tokens=14000, workers=2):
        out = np.zeros((len(enc), len(self.labels), 5), np.float32)
        done = np.zeros(len(enc), dtype=bool)
        chunks = self._chunks(enc, rows, batch, chunk, max_tokens)
        total = float(sum(len(enc[int(i)]) for i in rows))
        stat = {'invalid': 0, 'oom': 0, 'tokens': 0, 'rows': 0, 'next': 200, 't0': time.time()}

        def work(item):
            return self._score_rows(enc, item, chunk)

        def store(item, res):
            p, ok = res
            good = ok & np.isfinite(p).reshape(len(p), -1).all(1)
            out[item[good]] = p[good]
            done[item[good]] = True
            stat['invalid'] += int((~good).sum())
            stat['rows'] += len(item)
            stat['tokens'] += sum(len(enc[int(i)]) for i in item)
            if stat['rows'] >= stat['next']:
                stat['next'] += 200
                spent = time.time() - stat['t0']
                self.log(f"atlas {stat['rows']}/{len(rows)} rows, {spent:.0f}s elapsed, about {spent / stat['tokens'] * (total - stat['tokens']):.0f}s left")

        def fallback(item, c=chunk):
            try:
                store(item, self._score_rows(enc, item, c))
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                stat['oom'] += 1
                if len(item) > 1:
                    half = len(item) // 2
                    fallback(item[:half], c)
                    fallback(item[half:], c)
                elif c > 1:
                    fallback(item, 1)
                else:
                    stat['invalid'] += 1

        pipelined(chunks, work, store, fallback, workers if self.n_gpu >= 2 else 1, tag='atlas', log=self.log)
        self.log(f"atlas scored {int(done.sum())}/{len(rows)} rows in {time.time() - stat['t0']:.0f}s (invalid {stat['invalid']}, oom retries {stat['oom']})")
        return out, done

    def run_candidates(self, enc, cand, max_seqs=16, max_tokens=8192, workers=1):
        n, k = cand.shape
        rows = np.repeat(np.arange(n), k)
        labels = cand.ravel()
        lens = np.array([len(enc[int(r)]) for r in rows])
        order = np.argsort(-lens, kind='stable')
        batches, a = [], 0
        while a < len(order):
            size = max(1, min(max_seqs, max_tokens // max(1, int(lens[order[a]]))))
            batches.append(order[a:a + size])
            a += size
        out = np.full(n * k, np.nan, np.float32)
        stat = {'done': 0, 'next': 2000, 't0': time.time(), 'oom': 0}

        @torch.no_grad()
        def work(item):
            pk = self.tok.pad({'input_ids': [enc[int(rows[i])] for i in item]}, padding=True, return_tensors='pt')
            ids, mask = pk['input_ids'].to(self.model.device), pk['attention_mask'].to(self.model.device)
            lp = self.model.token_logp(ids, mask, torch.as_tensor(labels[item], device=ids.device))
            valid = mask[:, 1:].float()
            return ((lp * valid).sum(1) / valid.sum(1).clamp_min(1)).float().cpu().numpy()

        def store(item, vals):
            out[item] = np.where(np.isfinite(vals), vals, np.nan)
            stat['done'] += len(item)
            if stat['done'] >= stat['next']:
                stat['next'] += 2000
                spent = time.time() - stat['t0']
                self.log(f"atlas lite {stat['done']}/{len(order)} sequences, {spent:.0f}s elapsed, about {spent / stat['done'] * (len(order) - stat['done']):.0f}s left")

        def fallback(item):
            try:
                store(item, work(item))
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                stat['oom'] += 1
                if len(item) > 1:
                    half = len(item) // 2
                    fallback(item[:half])
                    fallback(item[half:])

        pipelined(batches, work, store, fallback, workers if self.n_gpu >= 2 else 1, tag='atlas lite', log=self.log)
        out = out.reshape(n, k)
        done = np.isfinite(out).all(1)
        self.log(f"atlas lite scored {int(done.sum())}/{n} rows ({n * k} sequences) in {time.time() - stat['t0']:.0f}s (oom retries {stat['oom']})")
        return out, done

    def candidate_logprobs(self, mean_lp, cand):
        z = self.lite['slope'] * mean_lp + np.asarray(self.lite['b'])[cand]
        z = z - z.max(1, keepdims=True)
        return z - np.log(np.exp(z).sum(1, keepdims=True))

    def calibrated(self, scores):
        feats = scores - scores.mean(1, keepdims=True)
        z = (feats / np.asarray(self.head['std'])) @ np.asarray(self.head['a']) + np.asarray(self.head['b'])
        z = z - z.max(1, keepdims=True)
        return z - np.log(np.exp(z).sum(1, keepdims=True))

    def close(self):
        del self.model
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
%%writefile native_q8.py
import argparse
import gc
import json
import os
import shutil
import time
from pathlib import Path
import numpy as np
import pandas as pd
INSTRUCTION = 'Instruct: Identify the language model that generated the text based on its writing style.\nQuery: '
DEPTHS = (3, 6, 9, 12, 18, 24, 30, 36)
SUFFIXES = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
RANK = 16
SCALE = 2.0
TOKEN_LIMIT = 512
TERMINAL = 151643

def log_softmax(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max(axis=-1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=-1, keepdims=True))

def calibrate(plain, centered, bias, groups, selections, use_centered=True):
    plain, centered, bias = (np.asarray(x, dtype=np.float64) for x in (plain, centered, bias))
    groups = np.asarray(groups)
    folds = plain.shape[0]
    assert plain.shape == centered.shape and plain.ndim == 3 and (plain.shape[-1] == 18)
    assert bias.shape == (folds, 18) and groups.shape == (plain.shape[1],) and (len(selections) == folds)
    assert all((np.isfinite(x).all() for x in (plain, centered, bias)))
    _, inverse = np.unique(groups, return_inverse=True)
    count = np.bincount(inverse)
    eligible = (count[inverse] >= 2) & (count[inverse] <= 18)
    out = []
    for k, state in enumerate(selections):
        assert not state['outer_labels_used'] and state['centered_head_trained']
        assert state['plain_scale'] > 0 and state['centered_scale'] > 0
        means = np.zeros((len(count), 18))
        np.add.at(means, inverse, centered[k])
        means /= count[:, None]
        relative = centered[k] - means[inverse] + bias[k]
        a = log_softmax(plain[k] * state['plain_scale'])
        b = log_softmax(relative * state['centered_scale'])
        weight = float(state['selected_centered_weight'])
        assert 0 <= weight <= 0.5
        fraction = np.where(eligible & use_centered, weight, 0.0)[:, None]
        out.append(log_softmax((1 - fraction) * a + fraction * b))
    return np.stack(out)

def windows(tokenizer, texts):
    prefix = tokenizer(INSTRUCTION, add_special_tokens=False)['input_ids']
    assert tokenizer('', add_special_tokens=True)['input_ids'] == [TERMINAL], 'unexpected end-of-text handling'
    capacity = TOKEN_LIMIT - len(prefix) - 1
    first_capacity = TOKEN_LIMIT - 1
    full = tokenizer([INSTRUCTION + str(t) for t in texts], truncation=False, add_special_tokens=False)['input_ids']
    seqs, owners, weights = ([], [], [])
    for row, tokens in enumerate(full):
        assert len(tokens) > 0
        seqs.append(tokens[:first_capacity] + [TERMINAL])
        owners.append(row)
        weights.append(max(1, min(len(tokens), first_capacity) - len(prefix)))
        pos = first_capacity
        while pos < len(tokens):
            chunk = tokens[pos:pos + capacity]
            seqs.append(prefix + chunk + [TERMINAL])
            owners.append(row)
            weights.append(len(chunk))
            pos += len(chunk)
    return (seqs, np.asarray(owners), np.asarray(weights, dtype=np.float64))

class NativeQ8:

    def __init__(self, base, assets, work, log=print, gpus=None):
        import torch
        from torch import nn
        from transformers import AutoModel, AutoTokenizer
        self.torch, self.nn, self.log = (torch, nn, log)
        torch.set_num_threads(2)
        for key in ('allow_tf32', 'allow_fp16_reduced_precision_reduction', 'allow_bf16_reduced_precision_reduction'):
            setattr(torch.backends.cuda.matmul, key, False)
        torch.backends.cudnn.allow_tf32 = False
        self.assets = Path(assets)
        self.labels = json.loads((self.assets / 'manifest.json').read_text())['classes']
        origin = Path(base)
        index = json.loads((self.assets / 'base' / 'model.safetensors.index.json').read_text())
        shards = sorted(set(index['weight_map'].values()))
        small = ['config.json', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json', 'merges.txt', 'vocab.json']
        self.staged = Path(work) / 'native-q8-base'
        self.staged.mkdir(parents=True, exist_ok=True)
        for name in small:
            src = self.assets / 'base' / name
            assert src.is_file(), str(src)
            dst = self.staged / name
            if not dst.exists():
                shutil.copyfile(src, dst)
        for name in shards:
            src, dst = (origin / name, self.staged / name)
            assert src.is_file(), str(src)
            if not dst.exists():
                try:
                    os.symlink(src, dst)
                except OSError:
                    shutil.copyfile(src, dst)
        for k in range(5):
            assert (self.assets / f'fold{k}' / 'trainables.safetensors').is_file(), k
        cfg = json.loads((self.staged / 'config.json').read_text())
        self.layers, self.hidden = (cfg['num_hidden_layers'], cfg['hidden_size'])
        assert self.layers == 36
        gpus = gpus if gpus is not None else torch.cuda.device_count()
        assert gpus >= 1
        self.head_device = 'cuda:0'
        place = {'embed_tokens': 0, 'norm': gpus - 1, 'rotary_emb': gpus - 1}
        cut = 16 if gpus >= 2 else self.layers
        place.update({f'layers.{i}': 0 if i < cut else 1 for i in range(self.layers)})
        self.tokenizer = AutoTokenizer.from_pretrained(self.staged, local_files_only=True)
        self.tokenizer.padding_side = 'right'
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        loaded = AutoModel.from_pretrained(self.staged, local_files_only=True, dtype=torch.float16, device_map=place, attn_implementation='sdpa', output_loading_info=True)
        if isinstance(loaded, tuple):
            self.model, info = loaded
            assert not any((info.get(k) for k in ('missing_keys', 'mismatched_keys', 'error_msgs', 'unexpected_keys'))), info
        else:
            self.model = loaded
        self.model.config.use_cache = False
        self.loras = {}
        for name, mod in list(self.model.named_modules()):
            if isinstance(mod, nn.Linear) and name.rsplit('.', 1)[-1] in SUFFIXES:
                wrapped = self._lora(mod)
                parent, attr = name.rsplit('.', 1)
                setattr(self.model.get_submodule(parent), attr, wrapped)
                self.loras[name] = wrapped
        assert len(self.loras) == self.layers * len(SUFFIXES), len(self.loras)
        dev = self.head_device
        self.norms = nn.ModuleList([nn.LayerNorm(self.hidden, device=dev, dtype=torch.float32) for _ in DEPTHS])
        self.plain = nn.Linear(self.hidden * len(DEPTHS), 18, device=dev, dtype=torch.float32)
        self.centered = nn.Linear(self.hidden * len(DEPTHS), 18, device=dev, dtype=torch.float32)
        self.model.requires_grad_(False).eval()
        for m in (self.norms, self.plain, self.centered):
            m.requires_grad_(False).eval()
        self.expected = {}
        for name, mod in self.loras.items():
            self.expected[f'encoder.backbone.base_model.model.{name}.lora_A.default.weight'] = mod.a.weight
            self.expected[f'encoder.backbone.base_model.model.{name}.lora_B.default.weight'] = mod.b.weight
        for prefix, m in (('norms', self.norms), ('plain', self.plain), ('centered', self.centered)):
            self.expected.update({f'{prefix}.{n}': p for n, p in m.named_parameters()})
        assert len(self.expected) == 2 * len(self.loras) + 2 * len(DEPTHS) + 4
        self.current = None

    def _lora(self, base):
        torch, nn = (self.torch, self.nn)

        class TaskLoRA(nn.Module):

            def __init__(self, layer):
                super().__init__()
                self.base_layer = layer
                dev = layer.weight.device
                self.a = nn.Linear(layer.in_features, RANK, bias=False, device=dev, dtype=torch.float32)
                self.b = nn.Linear(RANK, layer.out_features, bias=False, device=dev, dtype=torch.float32)

            def forward(self, x):
                result = self.base_layer(x)
                return (result + self.b(self.a(x.float())) * SCALE).to(result.dtype)
        return TaskLoRA(base)

    def load_fold(self, fold):
        if self.current == fold:
            return
        from safetensors.torch import load_file
        torch = self.torch
        state = load_file(str(self.assets / f'fold{fold}' / 'trainables.safetensors'))
        assert set(state) == set(self.expected), (sorted(set(state) - set(self.expected))[:3], sorted(set(self.expected) - set(state))[:3])
        with torch.inference_mode():
            for name, param in self.expected.items():
                assert state[name].shape == param.shape and state[name].dtype == torch.float32, name
                assert torch.isfinite(state[name]).all(), name
                param.copy_(state[name].to(param.device))
        del state
        gc.collect()
        self.current = fold

    def _features(self, seqs, idx):
        torch = self.torch
        batch = self.tokenizer.pad({'input_ids': [seqs[i] for i in idx]}, padding=True, return_tensors='pt').to('cuda:0')
        out = self.model(**batch, output_hidden_states=True, use_cache=False, return_dict=True)
        assert len(out.hidden_states) == self.layers + 1
        pos = batch['attention_mask'].sum(1) - 1
        rows = torch.arange(len(idx))
        parts = []
        for depth, norm in zip(DEPTHS, self.norms):
            hs = out.last_hidden_state if depth == self.layers else out.hidden_states[depth]
            pooled = hs[rows.to(hs.device), pos.to(hs.device)].float()
            if depth == self.layers:
                pooled = self.nn.functional.normalize(pooled, dim=-1)
            parts.append(norm(pooled.to(self.head_device)))
        return torch.cat(parts, dim=-1)

    def predict(self, texts, folds=range(5), batch_size=4, on_fold=None):
        torch = self.torch
        seqs, owners, weights = windows(self.tokenizer, texts)
        order = sorted(range(len(seqs)), key=lambda i: (len(seqs[i]), seqs[i]))
        n = len(texts)
        res = {'plain': [], 'centered': [], 'centered_bias': [], 'folds': []}
        finite = np.ones(n, dtype=bool)
        for fold in folds:
            self.load_fold(fold)
            t0 = time.time()
            vals = {'plain': np.zeros((n, 18)), 'centered': np.zeros((n, 18))}
            with torch.inference_mode():
                for s in range(0, len(order), batch_size):
                    idx = order[s:s + batch_size]
                    try:
                        feats = self._features(seqs, idx)
                    except torch.cuda.OutOfMemoryError:
                        torch.cuda.empty_cache()
                        feats = torch.cat([self._features(seqs, [i]) for i in idx], 0)
                    ok = torch.isfinite(feats).all(1).cpu().numpy()
                    if not ok.all():
                        for i, good in zip(idx, ok):
                            if not good:
                                finite[owners[i]] = False
                        feats = torch.nan_to_num(feats)
                    own = owners[idx]
                    for name, head in (('plain', self.plain), ('centered', self.centered)):
                        sc = head(feats).float().cpu().numpy().astype(np.float64)
                        np.add.at(vals[name], own, sc * weights[idx][:, None])
                    del feats
            denom = np.bincount(owners, weights=weights, minlength=n)[:, None]
            for name in ('plain', 'centered'):
                res[name].append(vals[name] / denom)
            res['centered_bias'].append(self.centered.bias.detach().cpu().numpy().astype(np.float64).copy())
            res['folds'].append(int(fold))
            self.log(f'native: fold {fold} done, {n} rows, {len(seqs)} windows, {time.time() - t0:.0f}s, non-finite rows {int((~finite).sum())}')
            if on_fold is not None:
                on_fold({k: np.stack(v) if k != 'folds' else np.asarray(v) for k, v in res.items()}, finite)
        return ({k: np.stack(v) if k != 'folds' else np.asarray(v) for k, v in res.items()}, finite)

def save(path, ids, labels, res, finite):
    tmp = str(path) + '.tmp.npz'
    np.savez(tmp, ids=np.asarray(ids, dtype=str), labels=np.asarray(labels), finite=finite, **res)
    os.replace(tmp, path)

def main():
    p = argparse.ArgumentParser()
    p.add_argument('--base', required=True)
    p.add_argument('--assets', required=True)
    p.add_argument('--query', required=True)
    p.add_argument('--out', required=True)
    p.add_argument('--work', required=True)
    p.add_argument('--batch', type=int, default=4)
    p.add_argument('--limit', type=int, default=0)
    a = p.parse_args()
    T0 = time.time()

    def log(*x):
        print(f'[{time.time() - T0:7.0f}s]', *x, flush=True)
    frame = pd.read_csv(a.query, dtype={'id': str})
    assert frame.id.is_unique and len(frame)
    if a.limit:
        frame = frame.iloc[:a.limit]
    texts = frame.text.fillna('').astype(str).tolist()
    model = NativeQ8(a.base, a.assets, a.work, log=log)
    log('native: backbone and heads built')
    res, finite = model.predict(texts, batch_size=a.batch, on_fold=lambda r, f: save(a.out, frame.id.values, model.labels, r, f))
    assert len(res['folds']), 'no fold finished'
    save(a.out, frame.id.values, model.labels, res, finite)
    log('native: saved', a.out, 'folds', res['folds'].tolist())
if __name__ == '__main__':
    main()


In [ ]:
%%writefile ntl660_readout.py
import json
import re
from pathlib import Path

import joblib
import numpy as np
from scipy.sparse import csr_matrix, diags, hstack

LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
                 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
                 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
                 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
FUNCTION_WORDS = frozenset('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
VIEWS = [('char', 3, 5, 250000, 1.), ('word', 1, 3, 180000, .7),
         ('shape', 2, 6, 60000, .5), ('function', 1, 3, 50000, .5)]


def shape_view(text):
    return re.sub(r'[^\W\d_]+', lambda m: 'X' if m[0].isupper() else 'x', text)


def function_view(text):
    return ' '.join(t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else 'CONTENT'
                    for t in re.findall(r'\w+|[^\w\s]', text))


def prepare(texts, view, keep=()):
    if view == 'shape':
        return [shape_view(t) for t in texts]
    if view == 'function':
        return [function_view(t) for t in texts]
    if view == 'distort':
        keep = set(keep)
        return [re.sub(r"[A-Za-z’']+", lambda m: m[0] if m[0].lower() in keep else '*', t) for t in texts]
    return texts


def log_softmax(z):
    z = z - z.max(1, keepdims=True)
    return z - np.log(np.exp(z).sum(1, keepdims=True))


def softmax(z):
    z = z - z.max(1, keepdims=True)
    z = np.exp(z)
    return z / z.sum(1, keepdims=True)


def sparse_group_centre(X, groups, lo=2, hi=18):
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1)
    col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()


def style_groups(texts):
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform
    from sklearn.feature_extraction.text import TfidfVectorizer
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    try:
        X = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=100000,
                            dtype=np.float32).fit_transform(texts)
        D = np.clip(1.0 - (X @ X.T).toarray(), 0., 1.)
    except ValueError as exc:
        if 'empty vocabulary' not in str(exc):
            raise
        D = np.ones((len(texts), len(texts)), dtype=np.float32) - np.eye(len(texts), dtype=np.float32)
    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)
    D = np.asarray(D / med, dtype=np.float64).copy()
    np.fill_diagonal(D, 0.)
    return fcluster(linkage(squareform(D, checks=False), method='average'), .90, criterion='distance')


class Readout:
    def __init__(self, folder):
        self.folder = Path(folder)
        self.meta = json.loads((self.folder / 'metadata.json').read_text())
        if self.meta['labels'] != LABELS:
            raise ValueError('Unexpected canonical class order')
        self.vectorizers = [joblib.load(self.folder / e['file']) for e in self.meta['views']]
        from sklearn.feature_extraction.text import TfidfVectorizer
        assert all(type(v) is TfidfVectorizer and not callable(v.analyzer) and
                   v.preprocessor is None and v.tokenizer is None for v in self.vectorizers)

    def matrix(self, texts):
        return hstack([v.transform(prepare(texts, e['view'], self.meta.get('keep', []))) * e['weight']
                       for v, e in zip(self.vectorizers, self.meta['views'])], format='csr')

    def raw(self, X, state='model'):
        with np.load(self.folder / (state + '.npz'), allow_pickle=False) as z:
            scores = np.asarray(X @ z['coef'].T) + z['intercept']
            classes = z['classes'].copy()
        if len(classes) == 2 and scores.shape[1] == 1:
            scores = np.column_stack([-scores[:, 0], scores[:, 0]])
        return scores, classes

    def scores(self, texts, state='model'):
        return self.raw(self.matrix(texts), state)


def core_votes(root, texts):
    votes = {}
    for key, weight in [('svc4', 12.0), ('tfidf', .25), ('distort', .10)]:
        r = Readout(Path(root) / 'core' / key)
        scores, classes = r.scores(texts)
        assert classes.tolist() == list(range(len(LABELS)))
        votes[key] = log_softmax(scores) * weight
    return votes


def complementary_style(root, texts):
    root = Path(root)
    r = Readout(root / 'comp' / 'stylegram')
    X = r.matrix(texts)
    d, classes = r.raw(X, 'plain')
    assert classes.tolist() == list(range(len(LABELS)))
    P = softmax(d)
    groups = style_groups(texts)
    sizes = np.bincount(groups)[groups]
    rows = (sizes >= 2) & (sizes <= 18)
    if rows.any():
        d, _ = r.raw(sparse_group_centre(X[rows], groups[rows]), 'twin')
        P[rows] = softmax(d)
    gates = {}
    for family in ['claude', 'gpt', 'gemini', 'glm']:
        g = Readout(root / 'comp' / 'gates' / family)
        d, classes = g.scores(texts)
        expected = [i for i, label in enumerate(LABELS) if label.startswith(family)]
        assert classes.tolist() == expected
        gates[family] = d
    return P, gates, groups


In [ ]:
%%writefile gsq_infer_model.py
import json
import math
import struct
import time
from pathlib import Path
import numpy as np
import torch
from torch import nn
from accelerate.utils import set_module_tensor_to_device
from transformers import AutoTokenizer
from transformers.models.qwen3_5.configuration_qwen3_5 import Qwen3_5TextConfig
from transformers.models.qwen3_5 import modeling_qwen3_5 as qm
from gsq_infer_layers import GSQLinear,DenseFrozenLinear
from delta_turing import triangular_chunk_gated_delta_rule

REVISION="b5ce0b76f60020a875dee4f6ec9d934cca4121e4"


class TensorReader:
    def __init__(self,base):
        self.base=Path(base)
        self.index=json.loads((self.base/"model.safetensors.index.json").read_text())["weight_map"]
        self.headers,self.maps={},{}
        for shard in sorted(set(self.index.values())):
            path=self.base/shard
            assert path.is_file(),f"Missing exact GSQ shard: {path}"
            with path.open("rb") as stream:
                size=struct.unpack("<Q",stream.read(8))[0];assert size<2**21
                self.headers[shard]=(8+size,json.loads(stream.read(size)))
            self.maps[shard]=np.memmap(path,dtype=np.uint8,mode="r")
        missing=[key for key,shard in self.index.items() if key not in self.headers[shard][1]]
        assert all(key.startswith("mtp.") for key in missing),missing
        self.stale_mtp_keys=missing

    def has(self,key):
        return key in self.index and key in self.headers[self.index[key]][1]

    def array(self,key):
        assert self.has(key),f"Required actual GSQ tensor is missing: {key}"
        shard=self.index[key];offset,header=self.headers[shard];item=header[key]
        lo,hi=item["data_offsets"]
        dtype={"I32":"<i4","I64":"<i8","BF16":"<u2","F16":"<f2","F32":"<f4"}[item["dtype"]]
        value=self.maps[shard][offset+lo:offset+hi].view(dtype).reshape(item["shape"])
        return value,item["dtype"]

    def tensor(self,key,device="cuda",bf16_dtype=None):
        value,dtype=self.array(key)
        if dtype=="BF16":
            assert bf16_dtype in (torch.float16,torch.float32)
            fp32=(value.astype(np.uint32)<<16).view(np.float32)
            result=torch.from_numpy(fp32).to(bf16_dtype)
        else:
            result=torch.from_numpy(value.copy())
        return result.to(device)


class QuantizedCPUEmbedding(nn.Module):
    def __init__(self,reader):
        super().__init__()
        prefix="model.language_model.embed_tokens."
        self.packed,dtype=reader.array(prefix+"weight_packed");assert dtype=="I32"
        self.scale_bits,dtype=reader.array(prefix+"weight_scale");assert dtype=="BF16"
        assert self.packed.shape==(248320,640) and self.scale_bits.shape==(248320,80)
        self.cpu_packed_bytes=self.packed.nbytes+self.scale_bits.nbytes
        self.last_lookup={}

    @torch.no_grad()
    def forward(self,ids):
        began=time.perf_counter()
        assert ids.device.type=="cpu"
        flat=ids.numpy().reshape(-1)
        unique,inverse=np.unique(flat,return_inverse=True)
        words=self.packed[unique].astype(np.uint32,copy=False)
        shifts=np.arange(8,dtype=np.uint32)*4
        q=((words[:,:,None]>>shifts)&15).reshape(len(unique),5120).astype(np.int16)-8
        scales=(self.scale_bits[unique].astype(np.uint32)<<16).view(np.float32)
        values=(q.reshape(len(unique),80,64).astype(np.float32)*scales[:,:,None]).reshape(len(unique),5120)
        decoded=torch.from_numpy(values).to(torch.float16)
        selected=decoded.index_select(0,torch.from_numpy(inverse)).reshape(*ids.shape,5120)
        result=selected.to("cuda",non_blocking=False)
        self.last_lookup=dict(unique_rows=len(unique),requested_rows=len(flat),
            transferred_bytes=selected.numel()*selected.element_size(),seconds=time.perf_counter()-began)
        return result


class GSQClassifier(nn.Module):
    def __init__(self,base,*,pad_multiple=8):
        super().__init__();began=time.perf_counter()
        assert torch.cuda.device_count()==1
        self.pad_multiple=pad_multiple
        self.depths=[8,16,24,32,40,48,56,64]
        config=json.loads((Path(base)/"config.json").read_text())
        quant=config["quantization_config"]
        assert quant["quant_method"]=="compressed-tensors" and quant["format"]=="pack-quantized"
        assert quant["config_groups"]["group_0"]["weights"]["num_bits"]==3
        assert quant["config_groups"]["group_0"]["weights"]["group_size"]==128
        cfg=Qwen3_5TextConfig(**config["text_config"])
        assert cfg.num_hidden_layers==64 and cfg.hidden_size==5120
        cfg._attn_implementation="sdpa";cfg.use_cache=False
        self.cfg=cfg
        delta=triangular_chunk_gated_delta_rule
        qm.FusedRMSNormGated=None
        with torch.device("meta"):self.model=qm.Qwen3_5TextModel(cfg)
        self.model.norm=nn.Identity()
        self.reader=TensorReader(base)
        prefix="model.language_model."
        packed_count=dense_count=packed_gpu_bytes=0
        for name,module in list(self.model.named_modules()):
            if not isinstance(module,nn.Linear):continue
            assert name.startswith("layers.") and module.bias is None
            layer=int(name.split(".")[1]);rank=16 if layer>=56 else 0
            key=prefix+name+"."
            if self.reader.has(key+"weight_packed"):
                shape,_=self.reader.array(key+"weight_shape")
                assert tuple(shape)==(module.out_features,module.in_features)
                packed=self.reader.tensor(key+"weight_packed")
                scales=self.reader.tensor(key+"weight_scale",bf16_dtype=torch.float32)
                replacement=GSQLinear(packed,scales,module.in_features,module.out_features,rank)
                packed_count+=1;packed_gpu_bytes+=packed.numel()*packed.element_size()+scales.numel()*scales.element_size()
            else:
                assert ".linear_attn.in_proj_a" in name or ".linear_attn.in_proj_b" in name,name
                replacement=DenseFrozenLinear(self.reader.tensor(key+"weight",bf16_dtype=torch.float16),module.in_features,module.out_features,rank)
                dense_count+=1
            parent,_,attribute=name.rpartition(".");setattr(self.model.get_submodule(parent),attribute,replacement)
        self.model.embed_tokens=QuantizedCPUEmbedding(self.reader)
        self.model.rotary_emb=qm.Qwen3_5TextRotaryEmbedding(cfg,device="cuda")
        for name,parameter in list(self.model.named_parameters()):
            if ".lora_A" in name or ".lora_B" in name:continue
            value=self.reader.tensor(prefix+name,bf16_dtype=torch.float16)
            set_module_tensor_to_device(self.model,name,"cuda",value=value,dtype=value.dtype)
        for name,parameter in self.model.named_parameters():parameter.requires_grad_(".lora_A" in name or ".lora_B" in name)
        assert not any(value.is_meta or value.device.type!="cuda" for value in self.model.parameters())
        assert not any(value.is_meta for value in self.model.buffers())
        for layer in self.model.layers:
            if hasattr(layer,"linear_attn"):
                layer.linear_attn.chunk_gated_delta_rule=delta
                layer.linear_attn.causal_conv1d_fn=None
        assert all(not p.requires_grad for layer in self.model.layers[:56] for p in layer.parameters())
        self.norms=nn.ModuleList([nn.LayerNorm(5120) for _ in self.depths]).to("cuda")
        self.plain=nn.Linear(40960,18).to("cuda");self.centered=nn.Linear(40960,18).to("cuda")
        self.tokenizer=AutoTokenizer.from_pretrained(base,local_files_only=True)
        self.tokenizer.pad_token_id=248044;self.tokenizer.padding_side="right"
        self.load_report=dict(revision=REVISION,seconds=time.perf_counter()-began,packed_linears=packed_count,
            ignored_original_BF16_linears=dense_count,packed_gpu_bytes=packed_gpu_bytes,
            CPU_packed_embedding_bytes=self.model.embed_tokens.cpu_packed_bytes,
            trainable_parameters=sum(p.numel() for p in self.parameters() if p.requires_grad),
            delta_backend=delta.__module__,causal_convolution="native PyTorch",pad_multiple=pad_multiple,
            all_decoder_layers=64,LoRA_target_layers=list(range(56,64)),
            dtype="FP16 compute/FP32 GSQ scales+LoRA+norms+heads; BF16 decoded on CPU",
            stale_unused_MTP_index_entries=len(self.reader.stale_mtp_keys))

    def forward(self,token_rows):
        assert all(ids and ids[-1]==248044 for ids in token_rows)
        batch=self.tokenizer.pad({"input_ids":token_rows},padding=True,pad_to_multiple_of=self.pad_multiple,return_tensors="pt")
        assert batch.attention_mask.sum(1).tolist()==[len(ids) for ids in token_rows]
        assert batch.input_ids.shape[1]<=self.cfg.max_position_embeddings
        raw=[]
        with torch.no_grad():
            hidden=self.model.embed_tokens(batch.input_ids)
            attention=batch.attention_mask.to("cuda")
            pos=torch.arange(hidden.shape[1],device="cuda")[None].expand(hidden.shape[0],-1)
            kwargs=dict(config=self.cfg,inputs_embeds=hidden,attention_mask=attention,past_key_values=None,position_ids=pos)
            masks={"full_attention":qm.create_causal_mask(**kwargs),"linear_attention":qm.create_recurrent_attention_mask(**kwargs)}
            rope=self.model.rotary_emb(hidden,pos[None].expand(3,-1,-1))
            mask=attention.unsqueeze(-1).float();denominator=mask.sum(1)
            for index in range(56):
                with torch.autocast("cuda",dtype=torch.float16):
                    hidden=self.model.layers[index](hidden,position_embeddings=rope,
                        attention_mask=masks[self.cfg.layer_types[index]],position_ids=pos,past_key_values=None,use_cache=False)
                if index+1 in self.depths:raw.append((hidden.float()*mask).sum(1)/denominator)
        assert len(raw)==7 and not hidden.requires_grad
        for index in range(56,64):
            layer=self.model.layers[index]
            def apply(value,block=layer,rotary=rope,position=pos,attention_mask=masks[self.cfg.layer_types[index]]):
                with torch.autocast("cuda",dtype=torch.float16):
                    return block(value,position_embeddings=rotary,attention_mask=attention_mask,
                                 position_ids=position,past_key_values=None,use_cache=False)
            hidden=apply(hidden)
        raw.append((hidden.float()*mask).sum(1)/denominator)
        features=torch.cat([norm(value) for norm,value in zip(self.norms,raw)],-1)
        assert features.shape==(len(token_rows),40960) and torch.isfinite(features).all()
        return features


In [ ]:
%%writefile gsq_infer_layers.py
import math
import torch
import triton
import triton.language as tl
from torch import nn
from torch.nn import functional as F


@triton.jit
def unpack3_kernel(P,S,W,N:tl.constexpr,K:tl.constexpr,PACKED_COLS:tl.constexpr,GROUPS:tl.constexpr,BLOCK:tl.constexpr):
    element=tl.program_id(0)*BLOCK+tl.arange(0,BLOCK)
    valid=element<N*K
    row=element//K;col=element%K
    bit=col*3;word=bit//32;shift=bit%32
    low=tl.load(P+row*PACKED_COLS+word,mask=valid,other=0).to(tl.uint32)
    crosses=shift>29
    high=tl.load(P+row*PACKED_COLS+word+1,mask=valid&crosses,other=0).to(tl.uint32)
    unsigned=((low>>shift)|(high<<((32-shift)&31)))&7
    signed=unsigned.to(tl.int32)-4
    scale=tl.load(S+row*GROUPS+col//128,mask=valid,other=0).to(tl.float32)
    tl.store(W+element,signed.to(tl.float32)*scale,mask=valid)


def dequantize(packed,scales):
    n,groups=scales.shape;k=groups*128
    output=torch.empty((n,k),device=packed.device,dtype=torch.float16)
    unpack3_kernel[(triton.cdiv(n*k,512),)](packed,scales,output,n,k,packed.shape[1],groups,512)
    return output


class GSQLinear(nn.Module):
    def __init__(self,packed,scales,in_features,out_features,rank):
        super().__init__()
        assert packed.dtype==torch.int32 and scales.dtype==torch.float32
        assert tuple(packed.shape)==(out_features,in_features*3//32)
        assert tuple(scales.shape)==(out_features,in_features//128)
        self.register_buffer("packed",packed,persistent=False)
        self.register_buffer("scales",scales,persistent=False)
        self.in_features,self.out_features=in_features,out_features
        self.bias=None
        initialize_lora(self,rank)
    def forward(self,value):
        base=F.linear(value,dequantize(self.packed,self.scales))
        return add_lora(self,value,base)


class DenseFrozenLinear(nn.Module):
    def __init__(self,weight,in_features,out_features,rank):
        super().__init__()
        assert tuple(weight.shape)==(out_features,in_features)
        self.register_buffer("base_weight",weight,persistent=False)
        self.in_features,self.out_features=in_features,out_features
        self.bias=None
        initialize_lora(self,rank)
    def forward(self,value):
        return add_lora(self,value,F.linear(value,self.base_weight))


def initialize_lora(module,rank):
    module.scale=2.
    module.dropout=nn.Identity()
    if rank:
        module.lora_A=nn.Parameter(torch.empty(rank,module.in_features,device="cuda",dtype=torch.float32))
        module.lora_B=nn.Parameter(torch.zeros(module.out_features,rank,device="cuda",dtype=torch.float32))
        nn.init.kaiming_uniform_(module.lora_A,a=math.sqrt(5))
    else:
        module.register_parameter("lora_A",None);module.register_parameter("lora_B",None)


def add_lora(module,value,base):
    if module.lora_A is None:return base
    with torch.autocast("cuda",dtype=torch.float16):
        extra=F.linear(F.linear(module.dropout(value),module.lora_A),module.lora_B)
    return base+module.scale*extra.to(base.dtype)


In [ ]:
%%writefile delta_turing.py
import torch
from torch.nn import functional as F
from transformers.models.qwen3_5.modeling_qwen3_5 import l2norm

def triangular_chunk_gated_delta_rule(
    query,
    key,
    value,
    g,
    beta,
    chunk_size=64,
    initial_state=None,
    output_final_state=False,
    use_qk_l2norm_in_kernel=False,
    **kwargs,
):
    initial_dtype = query.dtype
    if use_qk_l2norm_in_kernel:
        query = l2norm(query, dim=-1, eps=1e-6)
        key = l2norm(key, dim=-1, eps=1e-6)
    query, key, value, beta, g = [
        x.transpose(1, 2).contiguous().to(torch.float32) for x in (query, key, value, beta, g)
    ]

    batch_size, num_heads, sequence_length, k_head_dim = key.shape
    v_head_dim = value.shape[-1]
    pad_size = (chunk_size - sequence_length % chunk_size) % chunk_size
    query = F.pad(query, (0, 0, 0, pad_size))
    key = F.pad(key, (0, 0, 0, pad_size))
    value = F.pad(value, (0, 0, 0, pad_size))
    beta = F.pad(beta, (0, pad_size))
    g = F.pad(g, (0, pad_size))
    total_sequence_length = sequence_length + pad_size
    scale = 1 / (query.shape[-1] ** 0.5)
    query = query * scale

    v_beta = value * beta.unsqueeze(-1)
    k_beta = key * beta.unsqueeze(-1)
    query, key, value, k_beta, v_beta = [
        x.reshape(x.shape[0], x.shape[1], -1, chunk_size, x.shape[-1]) for x in (query, key, value, k_beta, v_beta)
    ]
    g = g.reshape(g.shape[0], g.shape[1], -1, chunk_size)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=0)

    g = g.cumsum(dim=-1)
    decay_mask = ((g.unsqueeze(-1) - g.unsqueeze(-2)).tril().exp().float()).tril()
    attn = -((k_beta @ key.transpose(-1, -2)) * decay_mask).masked_fill(mask, 0)
    with torch.autocast(device_type=query.device.type, enabled=False):
        identity = torch.eye(chunk_size, dtype=torch.float32, device=attn.device)
        coefficient = identity - attn.float()
        identity = identity.expand_as(coefficient)
        attn = torch.linalg.solve_triangular(coefficient, identity,
                                             upper=False, unitriangular=True)
    value = attn @ v_beta
    k_cumdecay = attn @ (k_beta * g.exp().unsqueeze(-1))
    last_recurrent_state = (
        torch.zeros(batch_size, num_heads, k_head_dim, v_head_dim, dtype=value.dtype, device=value.device)
        if initial_state is None
        else initial_state.to(value)
    )
    core_attn_out = torch.zeros_like(value)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=1)

    for i in range(0, total_sequence_length // chunk_size):
        q_i, k_i, v_i = query[:, :, i], key[:, :, i], value[:, :, i]
        attn = q_i @ k_i.transpose(-1, -2) * decay_mask[:, :, i]
        v_prime = (k_cumdecay[:, :, i]) @ last_recurrent_state
        v_new = v_i - v_prime
        attn_inter = (q_i * g[:, :, i, :, None].exp()) @ last_recurrent_state
        core_attn_out[:, :, i] = attn_inter + attn @ v_new
        last_recurrent_state = (
            last_recurrent_state * g[:, :, i, -1, None, None].exp()
            + (k_i * (g[:, :, i, -1, None] - g[:, :, i]).exp()[..., None]).transpose(-1, -2) @ v_new
        )

    if not output_final_state:
        last_recurrent_state = None
    core_attn_out = core_attn_out.reshape(core_attn_out.shape[0], core_attn_out.shape[1], -1, core_attn_out.shape[-1])
    core_attn_out = core_attn_out[:, :, :sequence_length]
    core_attn_out = core_attn_out.transpose(1, 2).contiguous().to(initial_dtype)
    return core_attn_out, last_recurrent_state


In [ ]:
%%writefile adapter.py
import json
from pathlib import Path

import torch
from safetensors import safe_open

REVISION = "b5ce0b76f60020a875dee4f6ec9d934cca4121e4"


def load_final(model, final_file, metadata_file, complete_file):
    final_file, metadata_file, complete_file = map(Path, (final_file, metadata_file, complete_file))
    for path in (final_file, metadata_file, complete_file):
        if not path.is_file():
            raise FileNotFoundError(f"Required final training artifact missing: {path}")
    metadata = json.loads(metadata_file.read_text(encoding="utf-8"))
    complete = json.loads(complete_file.read_text(encoding="utf-8"))
    assert complete["status"] == "TRAINED_FULLTEXT_GSQ3_PUBLIC119899_OFFICIAL2_EPOCHS"
    assert complete["rows_seen"] == {"external": 119899, "official": 5398}
    assert complete["final_artifact"]["path"] == final_file.name == "final.safetensors"
    assert complete["contract_hash"] == metadata["checkpoint_contract_hash"]
    assert metadata["source_revision"] == REVISION
    assert metadata["rank"] == 16 and metadata["alpha"] == 32
    assert metadata["depths"] == [8, 16, 24, 32, 40, 48, 56, 64]
    assert metadata["lora_target_layer_indices"] == list(range(56, 64))
    assert metadata["executed_decoder_layers"] == 64 and metadata["text_truncation"] is False
    assert len(metadata["labels"]) == 18
    assert final_file.stat().st_size == complete["final_artifact"]["bytes"]
    expected = {name: param for name, param in model.named_parameters() if param.requires_grad}
    export_keys = {"backbone." + name.removeprefix("model.") if name.startswith("model.") else name: name
                   for name in expected}
    with safe_open(str(final_file), framework="pt", device="cpu") as weights:
        assert set(weights.keys()) == set(export_keys), "Final adapter tensor names differ from model trainable parameters"
        with torch.no_grad():
            for key, name in export_keys.items():
                source = weights.get_tensor(key)
                target = expected[name]
                assert source.dtype == target.dtype and source.shape == target.shape and torch.isfinite(source).all(), key
                target.copy_(source.to(target.device))
    model.eval()
    return metadata


In [ ]:
%%writefile gsq_infer.py
import argparse
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def read_rows(path, text_column):
    frame = pd.read_parquet(path).reset_index(drop=True)
    assert text_column in frame and frame[text_column].notna().all()
    texts = frame[text_column].astype(str).tolist()
    assert all(texts)
    return frame, texts


def worker(args):
    assert os.environ.get("CUDA_VISIBLE_DEVICES") in ("0", "1")
    import torch
    from adapter import load_final
    from gsq_infer_model import GSQClassifier

    assert torch.cuda.device_count() == 1, "GSQ worker must see one isolated GPU"
    assert torch.cuda.get_device_capability(0) >= (7, 5), "GSQ T4 path requires SM 7.5 or newer"
    torch.set_grad_enabled(False)
    frame, texts = read_rows(args.input, args.text_column)
    indices = np.arange(len(texts), dtype=np.int64)[args.worker_id::2]
    assert len(indices)
    model = GSQClassifier(args.base)
    metadata = load_final(model, args.final, args.metadata, args.complete)
    tokenizer = model.tokenizer
    features = np.empty((len(indices), 40960), dtype=np.float32)
    plain = np.empty((len(indices), 18), dtype=np.float32)
    for begin in range(0, len(indices), args.batch_size):
        batch_ids = indices[begin:begin + args.batch_size]
        token_rows = tokenizer([texts[int(i)] for i in batch_ids], add_special_tokens=False,
                               truncation=False)["input_ids"]
        token_rows = [ids + [248044] for ids in token_rows]
        assert all(len(ids) <= model.cfg.max_position_embeddings for ids in token_rows), "Full text exceeds model context"
        with torch.inference_mode():
            value = model(token_rows)
            score = model.plain(value)
        stop = begin + len(batch_ids)
        features[begin:stop] = value.float().cpu().numpy()
        plain[begin:stop] = score.float().cpu().numpy()
        del value, score
    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez(args.output, row_index=indices, features=features, plain_logits=plain)
    print(json.dumps(dict(worker=args.worker_id, rows=len(indices), labels=metadata["labels"],
                          load_report=model.load_report)), flush=True)


def group_codes(args, frame):
    assert (args.groups is None) != (args.prompt_column is None), "Provide exactly one group source"
    if args.groups is not None:
        codes = np.load(args.groups, allow_pickle=False)
        assert codes.ndim == 1 and len(codes) == len(frame) and np.issubdtype(codes.dtype, np.integer)
        assert np.all(codes >= 0), "Explicit group codes must be nonnegative"
        return codes
    assert args.prompt_column in frame, "Requested prompt column is missing"
    assert frame[args.prompt_column].notna().all(), "Missing prompt group is ambiguous"
    return pd.factorize(frame[args.prompt_column], sort=False)[0]


def combine(args, outputs):
    from safetensors import safe_open

    frame, _ = read_rows(args.input, args.text_column)
    n = len(frame)
    features = np.empty((n, 40960), dtype=np.float32)
    plain = np.empty((n, 18), dtype=np.float32)
    seen = np.zeros(n, dtype=bool)
    for path in outputs:
        with np.load(path, allow_pickle=False) as item:
            ids = item["row_index"]
            assert ids.ndim == 1 and np.issubdtype(ids.dtype, np.integer)
            assert len(np.unique(ids)) == len(ids)
            assert np.all((0 <= ids) & (ids < n)) and not np.any(seen[ids])
            assert item["features"].shape == (len(ids), 40960)
            assert item["plain_logits"].shape == (len(ids), 18)
            features[ids] = item["features"]
            plain[ids] = item["plain_logits"]
            seen[ids] = True
    assert seen.all() and np.isfinite(features).all() and np.isfinite(plain).all()
    metadata = json.loads(args.metadata.read_text(encoding="utf-8"))
    with safe_open(str(args.final), framework="pt", device="cpu") as weights:
        kernel = weights.get_tensor("centered.weight").numpy().astype(np.float32)
        bias = weights.get_tensor("centered.bias").numpy().astype(np.float32)
    assert kernel.shape == (18, 40960) and bias.shape == (18,)
    groups = group_codes(args, frame)
    centered = plain.copy()
    valid_centered = np.zeros(n, dtype=bool)
    for group in np.unique(groups):
        ids = np.flatnonzero(groups == group)
        if len(ids) == 1:
            continue
        centered[ids] = (features[ids] - features[ids].mean(0, keepdims=True)) @ kernel.T + bias
        valid_centered[ids] = True
    assert np.isfinite(centered).all() and len(groups) == n
    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez(args.output, row_index=np.arange(n, dtype=np.int64), features=features,
             plain_logits=plain, centered_logits=centered, valid_centered=valid_centered,
             group_code=groups, labels=np.asarray(metadata["labels"]))


def parent(args):
    _, texts = read_rows(args.input, args.text_column)
    worker_count = min(2, len(texts))
    outputs = [args.work_dir / f"gsq-worker-{i}.npz" for i in range(worker_count)]
    args.work_dir.mkdir(parents=True, exist_ok=True)
    children = []
    for device in range(worker_count):
        command = [sys.executable, str(Path(__file__).resolve()), "--worker-id", str(device),
                   "--base", str(args.base), "--final", str(args.final), "--metadata", str(args.metadata),
                   "--complete", str(args.complete), "--input", str(args.input), "--output", str(outputs[device]),
                   "--text-column", args.text_column,
                   "--batch-size", str(args.batch_size)]
        env = os.environ.copy()
        env.update(CUDA_VISIBLE_DEVICES=str(device), HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                   TOKENIZERS_PARALLELISM="false")
        children.append(subprocess.Popen(command, env=env))
    codes = [child.wait() for child in children]
    if any(code != 0 for code in codes):
        raise RuntimeError(f"GSQ workers failed: exit codes {codes}; outputs were not combined")
    combine(args, outputs)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    for name in ("base", "final", "metadata", "complete", "input", "output"):
        p.add_argument("--" + name, type=Path, required=True)
    p.add_argument("--work-dir", type=Path)
    p.add_argument("--text-column", default="text")
    p.add_argument("--prompt-column")
    p.add_argument("--groups", type=Path)
    p.add_argument("--batch-size", type=int, default=1)
    p.add_argument("--worker-id", type=int, choices=(0, 1))
    args = p.parse_args()
    assert args.batch_size >= 1
    if args.worker_id is None:
        assert args.work_dir is not None and os.environ.get("CUDA_VISIBLE_DEVICES") in (None, "", "0,1")
        assert (args.groups is None) != (args.prompt_column is None), "Provide --groups or --prompt-column"
        parent(args)
    else:
        worker(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile gsq_stage.py
import json
import os
import shutil
import signal
import subprocess
import sys
import tempfile
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

BASE_FILES = {'config.json': 5038, 'merges.txt': 3353259, 'model.safetensors.index.json': 198539, 'tokenizer.json': 12809320,
              'tokenizer_config.json': 17928, 'vocab.json': 6722759, 'model-00001-of-00003.safetensors': 2272369784,
              'model-00002-of-00003.safetensors': 4777704120, 'model-00003-of-00003.safetensors': 4777704376}


@contextmanager
def staged_tree(source, files, label, temp_root='/tmp'):
    source, temp_root = Path(source), Path(temp_root)
    assert source.is_dir()
    for name, size in files.items():
        item = source / name
        assert item.is_file() and item.stat().st_size == size, (label, name, size)
    total = sum(files.values())
    free = shutil.disk_usage(temp_root).free
    assert free > total + 2 * 2 ** 30, ('not enough local disk to stage', label, total, free)
    destination = Path(tempfile.mkdtemp(prefix='ntl-stage-' + label + '-', dir=temp_root))
    start = time.monotonic()
    try:
        for name, size in files.items():
            with (source / name).open('rb') as reader, (destination / name).open('wb') as writer:
                while block := reader.read(16 * 1024 * 1024):
                    writer.write(block)
            assert (destination / name).stat().st_size == size
        print(json.dumps({'stage': label, 'staged_bytes': total, 'seconds': round(time.monotonic() - start)}), flush=True)
        yield destination
    finally:
        shutil.rmtree(destination, ignore_errors=True)


def run(assets, ids, texts, groups, work, labels, log=print):
    assets, work = Path(assets), Path(work)
    inference = assets / 'gsq120k-trained' / 'gsq120k-inference' / 'gsq'
    final, metadata, complete = inference / 'final.safetensors', inference / 'final.json', inference / 'COMPLETE.json'
    for path in (final, metadata, complete):
        assert path.is_file(), path
    wheels = sorted((assets / 'name-that-llm-artifacts' / 'qwen38-offline-linux').glob('*.whl'))
    assert len(wheels) >= 6, wheels
    work.mkdir(parents=True, exist_ok=True)
    target = work / 'python'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--disable-pip-version-check', '--target', str(target)] + [str(w) for w in wheels],
                   check=True, capture_output=True)
    query = work / 'query.parquet'
    pd.DataFrame({'id': list(ids), 'text': list(texts)}).to_parquet(query, index=False)
    group_file = work / 'groups.npy'
    np.save(group_file, np.asarray(groups, dtype=np.int64), allow_pickle=False)
    output = work / 'gsq.npz'
    if output.exists():
        output.unlink()
    started = time.monotonic()
    with staged_tree(assets / 'gsq3-base' / 'qwen38-27b-gsq3', BASE_FILES, 'gsq3-base') as base:
        env = dict(os.environ, PYTHONPATH=str(target), CUDA_VISIBLE_DEVICES='', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')
        command = [sys.executable, '-u', str(Path(__file__).resolve().parent / 'gsq_infer.py'), '--base', str(base), '--final', str(final), '--metadata', str(metadata),
                   '--complete', str(complete), '--input', str(query), '--groups', str(group_file), '--output', str(output), '--work-dir', str(work / 'workers'), '--batch-size', '1']
        process = subprocess.Popen(command, env=env, start_new_session=True)
        code = process.wait()
        if code != 0:
            raise RuntimeError(f'GSQ stage exited with code {code}')
    with np.load(output, allow_pickle=False) as z:
        saved = [str(x) for x in z['labels'].tolist()]
        assert sorted(saved) == sorted(labels) and len(saved) == 18
        logits = z['plain_logits'][:, [saved.index(l) for l in labels]].astype(np.float64)
        assert len(logits) == len(ids) and np.isfinite(logits).all()
    logits -= logits.max(1, keepdims=True)
    probs = np.exp(logits)
    probs /= probs.sum(1, keepdims=True)
    log(f'GSQ 27B scored {len(ids)} rows in {time.monotonic() - started:.0f} s')
    return probs


In [ ]:
%%writefile qwen_groups.py
import argparse
import json
import os
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer


def stage(base, assets, work):
    base, assets, staged = Path(base), Path(assets), Path(work) / 'qgroup-base'
    staged.mkdir(parents=True, exist_ok=True)
    index = json.loads((assets / 'base' / 'model.safetensors.index.json').read_text())
    for name in ('config.json', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json', 'merges.txt', 'vocab.json'):
        target = staged / name
        if not target.exists():
            shutil.copyfile(assets / 'base' / name, target)
    for name in sorted(set(index['weight_map'].values())):
        source, target = base / name, staged / name
        assert source.is_file(), str(source)
        if not target.exists():
            try:
                os.symlink(source, target)
            except OSError:
                shutil.copyfile(source, target)
    return staged


def main():
    p = argparse.ArgumentParser()
    for key in ('base', 'assets', 'query', 'out', 'work'):
        p.add_argument('--' + key, required=True)
    p.add_argument('--max_len', type=int, default=512)
    p.add_argument('--tok_budget', type=int, default=8192)
    p.add_argument('--mem_frac', type=float, default=0.88)
    a = p.parse_args()
    started = time.time()
    frame = pd.read_csv(a.query, dtype={'id': str})
    texts = frame.text.fillna('').astype(str).tolist()
    staged = stage(a.base, a.assets, a.work)
    tok = AutoTokenizer.from_pretrained(staged, local_files_only=True)
    auto, plain = tok('hello')['input_ids'], tok('hello', add_special_tokens=False)['input_ids']
    eos = auto[-1] if (len(auto) > len(plain) and auto[-1] != plain[-1]) else (tok.eos_token_id if tok.eos_token_id is not None else 151643)
    pad = tok.pad_token_id if tok.pad_token_id is not None else eos
    gpus = torch.cuda.device_count()
    assert gpus >= 1
    max_memory = {i: f'{int(torch.cuda.get_device_properties(i).total_memory * a.mem_frac / 2 ** 30)}GiB' for i in range(gpus)}
    model = AutoModel.from_pretrained(staged, local_files_only=True, dtype=torch.float16, device_map='auto', max_memory=max_memory).eval()
    first = next(model.parameters()).device
    ids = []
    for t in texts:
        x = tok(t, truncation=True, max_length=a.max_len - 1)['input_ids']
        ids.append(x if x[-1] == eos else x + [eos])
    order = np.argsort([len(x) for x in ids], kind='stable')
    emb = np.zeros((len(texts), model.config.get_text_config().hidden_size), np.float16)
    budget, done, ooms = a.tok_budget, 0, 0
    with torch.inference_mode():
        while done < len(order):
            end = done + 1
            while end < len(order) and (end - done + 1) * len(ids[order[end]]) <= budget:
                end += 1
            idx = order[done:end]
            try:
                width = max(len(ids[i]) for i in idx)
                inp = torch.full((len(idx), width), pad)
                att = torch.zeros((len(idx), width), dtype=torch.long)
                for r, i in enumerate(idx):
                    inp[r, :len(ids[i])] = torch.tensor(ids[i])
                    att[r, :len(ids[i])] = 1
                hidden = model(input_ids=inp.to(first), attention_mask=att.to(first), use_cache=False).last_hidden_state
                vec = hidden[torch.arange(len(idx), device=hidden.device), (att.sum(1) - 1).to(hidden.device)].float()
                emb[idx] = torch.nn.functional.normalize(vec, dim=-1).cpu().numpy().astype(np.float16)
                done = end
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                ooms += 1
                budget = max(len(ids[order[done]]), budget // 2)
                print(f'qwen_groups: OOM, token budget -> {budget}', flush=True)
    assert np.isfinite(emb).all()
    os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
    np.save(a.out, emb)
    print(f'qwen_groups: {len(texts)} rows, dim {emb.shape[1]}, {ooms} OOM retries, {time.time() - started:.0f} s', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile infer.py
import os, sys, time, gc, argparse, json, re
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np, pandas as pd, scipy.sparse as sp, torch
from pipe import pipelined, spans_devices
LABELS = ['claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max']
L2I = {l: i for i, l in enumerate(LABELS)}
p = argparse.ArgumentParser()
p.add_argument('--data', required=True)
p.add_argument('--models', required=True)
p.add_argument('--out', default='submission.parquet')
p.add_argument('--save_final_logits', default='')
p.add_argument('--comp_path', default='')
p.add_argument('--comp_w', type=float, default=0.0)
p.add_argument('--tfidf_w', type=float, default=1.0)
p.add_argument('--distort_w', type=float, default=0.0)
p.add_argument('--adapt_thr', type=float, default=0.15)
p.add_argument('--adapt_w', type=float, default=0.0)
p.add_argument('--extra', default='')
p.add_argument('--extra_w', type=float, default=0.5)
p.add_argument('--svc_w', type=float, default=0.0)
p.add_argument('--pool', default='')
p.add_argument('--sk_div', type=int, default=1)
p.add_argument('--sinkhorn', type=float, default=0.0)
p.add_argument('--sinkhorn_target_mix', type=float, default=0.35)
p.add_argument('--atlas_dir', default='')
p.add_argument('--atlas_base', default='')
p.add_argument('--atlas_mode', default='off', choices=['off', 'head', 'lite', 'mix'])
p.add_argument('--atlas_member_w', type=float, default=0.0)
p.add_argument('--atlas_k', type=int, default=6)
p.add_argument('--atlas_w', type=float, default=0.0)
p.add_argument('--atlas_margin', type=float, default=0.0)
p.add_argument('--atlas_bs', type=int, default=4)
p.add_argument('--atlas_chunk', type=int, default=3)
p.add_argument('--atlas_workers', type=int, default=2)
p.add_argument('--save_lp', default='')
p.add_argument('--native_dir', default='')
p.add_argument('--native_base', default='')
p.add_argument('--native_share', type=float, default=0.0)
p.add_argument('--native_cent', type=int, default=1)
p.add_argument('--native_work', default='/kaggle/temp/native')
p.add_argument('--native_min_agree', type=float, default=0.35)
p.add_argument('--prefit_core', default='')
p.add_argument('--gsq_dir', default='')
p.add_argument('--gsq_share', type=float, default=0.0)
p.add_argument('--gsq_work', default='/kaggle/temp/gsq')
p.add_argument('--gsq_min_agree', type=float, default=0.3)
p.add_argument('--group_embed_base', default='')
p.add_argument('--group_embed_assets', default='')
p.add_argument('--group_work', default='/kaggle/temp/qgroups')
p.add_argument('--group_sim', type=float, default=0.7)
a = p.parse_args()
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
T0 = time.time()

def log(*x):
    print(f'[{time.time() - T0:7.0f}s]', *x, flush=True)
train = pd.read_csv(os.path.join(a.data, 'train.csv'))
test = pd.read_csv(os.path.join(a.data, 'test.csv'))
train['text'] = train.text.fillna('')
test['text'] = test.text.fillna('')
y = train.model.map(L2I).values
Gtr = train.prompt_id.values
log('train', len(train), 'test', len(test))

def capped_agglo(S, cap=18, thr=0.02):
    n = len(S)
    A = S.astype(np.float32).copy()
    np.fill_diagonal(A, -np.inf)
    size = np.ones(n, np.int32)
    alive = np.ones(n, bool)
    lab = np.arange(n)
    while True:
        ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]
        M = np.where(ok, A, -np.inf)
        k = int(np.argmax(M))
        i, j = divmod(k, n)
        if not np.isfinite(M[i, j]) or M[i, j] < thr:
            break
        new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])
        A[i] = new
        A[:, i] = new
        A[i, i] = -np.inf
        A[j] = -np.inf
        A[:, j] = -np.inf
        size[i] += size[j]
        alive[j] = False
        lab[lab == j] = i
    return lab
from sklearn.feature_extraction.text import TfidfVectorizer
try:
    v = TfidfVectorizer(sublinear_tf=True, stop_words='english', min_df=2, max_df=0.05)
    Xc = v.fit_transform(test.text.values)
    Gte = capped_agglo((Xc @ Xc.T).toarray(), 18, a.adapt_thr)
except ValueError as e:
    log('clustering fallback:', e)
    Gte = np.arange(len(test))
log('test clusters', len(np.unique(Gte)), 'sizes', np.bincount(np.bincount(Gte)[np.bincount(Gte) > 0]).tolist())
if a.group_embed_base:
    try:
        import subprocess
        os.makedirs(a.group_work, exist_ok=True)
        emb_path = os.path.join(a.group_work, 'group_emb.npy')
        if os.path.exists(emb_path):
            os.remove(emb_path)
        gcmd = [sys.executable, '-u', os.path.join(os.path.dirname(os.path.abspath(__file__)), 'qwen_groups.py'), '--base', a.group_embed_base, '--assets', a.group_embed_assets, '--query', os.path.join(a.data, 'test.csv'), '--out', emb_path, '--work', a.group_work]
        gcode = subprocess.run(gcmd, env=dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')).returncode
        E_q = np.load(emb_path).astype(np.float32)
        assert gcode == 0 and E_q.shape[0] == len(test) and np.isfinite(E_q).all()
        G_q = capped_agglo(E_q @ E_q.T, 18, a.group_sim)
        _sz_old, _sz_new = (np.bincount(Gte)[Gte], np.bincount(G_q)[G_q])
        log('qwen groups', len(np.unique(G_q)), 'clusters (tf-idf', len(np.unique(Gte)), ') rows in groups of 2..18: qwen', round(float(((_sz_new >= 2) & (_sz_new <= 18)).mean()), 3), 'tf-idf', round(float(((_sz_old >= 2) & (_sz_old <= 18)).mean()), 3))
        assert np.bincount(G_q).max() <= 18
        Gte = G_q
        del E_q
    except Exception as _e:
        log('QWEN GROUPS SKIPPED, keeping the TF-IDF groups:', repr(_e))
LP = {}
EXT_T, EXT_Y = (np.array([], dtype=object), np.array([], dtype=int))
EXTRA_DF = pd.DataFrame(columns=['text', 'model'])

def _norm_text_key(value):
    import unicodedata
    return re.sub('\\s+', ' ', unicodedata.normalize('NFKC', str(value or ''))).strip()
if a.extra:
    _E = pd.read_parquet(a.extra) if a.extra.endswith('.parquet') else pd.read_csv(a.extra)
    if not {'text', 'model'}.issubset(_E.columns):
        raise ValueError(f'External file must contain text/model columns: {a.extra}')
    _E = _E[_E.model.isin(LABELS)].copy()
    _E['text'] = _E.text.fillna('').astype(str)
    _official_keys = {_norm_text_key(t) for t in train.text.values}
    _seen = set(_official_keys)
    _keep = []
    for _t in _E.text.values:
        _k = _norm_text_key(_t)
        _ok = bool(_k) and _k not in _seen
        _keep.append(_ok)
        if _ok:
            _seen.add(_k)
    EXTRA_DF = _E.loc[_keep, ['text', 'model']].reset_index(drop=True)
    EXT_T = EXTRA_DF.text.astype(str).values
    EXT_Y = EXTRA_DF.model.map(L2I).values.astype(int)
    log('matched-clean extra rows after exact de-dup', len(EXT_T))
FIT_T = np.concatenate([train.text.values, EXT_T])
FIT_Y = np.concatenate([y, EXT_Y])
FIT_W = np.concatenate([np.ones(len(train), dtype=np.float32), np.full(len(EXT_T), a.extra_w, dtype=np.float32)])
if a.tfidf_w > 0:
    from sklearn.linear_model import LogisticRegression
    mats_tr, mats_te = ([], [])
    for kw in [dict(analyzer='char_wb', ngram_range=(2, 5), min_df=3, max_features=60000), dict(token_pattern='\\w+|[^\\w\\s]', ngram_range=(1, 2), min_df=3, max_features=30000)]:
        vec = TfidfVectorizer(sublinear_tf=True, lowercase=False, **kw).fit(FIT_T)
        mats_tr.append(vec.transform(FIT_T))
        mats_te.append(vec.transform(test.text.values))
    LP['tfidf'] = LogisticRegression(C=20, max_iter=3000).fit(sp.hstack(mats_tr).tocsr(), FIT_Y, sample_weight=FIT_W).predict_log_proba(sp.hstack(mats_te).tocsr()) * a.tfidf_w
    gc.collect()
    log('tfidf done')
else:
    LP['tfidf'] = np.zeros((len(test), 18), dtype=np.float32)
    log('tfidf skipped (zero ensemble weight)')
if a.svc_w > 0:
    import re as _re
    from scipy.sparse import hstack as _hs
    from sklearn.svm import LinearSVC
    _FW = set('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
    _shape = lambda t: _re.sub('[^\\W\\d_]+', lambda m_: 'X' if m_[0].isupper() else 'x', t)
    _func = lambda t: ' '.join((w.lower() if w.lower() in _FW or not w.isalnum() else 'CONTENT' for w in _re.findall('\\w+|[^\\w\\s]', t)))
    _parts = []
    _official = train[['text', 'model']].copy()
    _official['text'] = _official.text.fillna('').astype(str)
    _official['_srcw'] = 1.0
    _parts.append(_official)
    _seen = {_norm_text_key(t) for t in _official.text.values}
    if len(EXTRA_DF):
        _e = EXTRA_DF[['text', 'model']].copy()
        _e['_srcw'] = float(a.extra_w)
        _keep = []
        for _t in _e.text.values:
            _k = _norm_text_key(_t)
            _ok = bool(_k) and _k not in _seen
            _keep.append(_ok)
            if _ok:
                _seen.add(_k)
        _e = _e.loc[_keep].copy()
        _parts.append(_e)
    if a.pool and os.path.exists(a.pool):
        _P = pd.read_parquet(a.pool)
        if not {'text', 'model'}.issubset(_P.columns):
            raise ValueError(f'Pool must contain text/model columns: {a.pool}')
        _P = _P[_P.model.isin(LABELS)][['text', 'model']].copy()
        _P['text'] = _P.text.fillna('').astype(str)
        _P['_srcw'] = 0.35
        _keep = []
        for _t in _P.text.values:
            _k = _norm_text_key(_t)
            _ok = bool(_k) and _k not in _seen
            _keep.append(_ok)
            if _ok:
                _seen.add(_k)
        _P = _P.loc[_keep].copy()
        _parts.append(_P)
    _fitdf = pd.concat(_parts, ignore_index=True)
    _fit = _fitdf.text.astype(str).tolist()
    _fy = _fitdf.model.map(L2I).to_numpy(dtype=int)
    _sw = _fitdf['_srcw'].to_numpy(dtype=np.float32)
    log(f'svc4 fit rows={len(_fit)} source_weighted; external clean={len(EXTRA_DF)}')
    _A, _B = ([], [])
    for view, lo, hi, mf, wv in [('char', 3, 5, 250000, 1.0), ('word', 1, 3, 180000, 0.7), ('shape', 2, 6, 60000, 0.5), ('function', 1, 3, 50000, 0.5)]:
        prep = _shape if view == 'shape' else _func if view == 'function' else lambda s_: s_
        an = 'char' if view == 'shape' else 'word' if view == 'function' else view
        o = dict(analyzer=an, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2, max_features=mf, dtype=np.float32)
        if an == 'word':
            o['token_pattern'] = '(?u)\\b\\w+\\b|[^\\w\\s]'
        _v = TfidfVectorizer(**o)
        _A.append(_v.fit_transform([prep(t) for t in _fit]) * wv)
        _B.append(_v.transform([prep(t) for t in test.text.values]) * wv)
    _svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight='balanced')
    _svc.fit(_hs(_A, format='csr'), _fy, sample_weight=_sw)
    _d = _svc.decision_function(_hs(_B, format='csr'))
    LP['svc4'] = (_d - _d.max(1, keepdims=True) - np.log(np.exp(_d - _d.max(1, keepdims=True)).sum(1, keepdims=True))) * a.svc_w
    del _A, _B, _parts, _fitdf, _svc
    gc.collect()
    log('svc4 done')
if a.distort_w > 0:
    import re, collections
    wre = re.compile("[A-Za-z’']+")
    cnt = collections.Counter((w.lower() for t in FIT_T for w in wre.findall(t)))
    keep = {w for w, _ in cnt.most_common(100)}
    dist = lambda arr: [wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in keep else '*', t) for t in arr]
    Dtr, Dte = (dist(FIT_T), dist(test.text.values))
    vs = [TfidfVectorizer(sublinear_tf=True, lowercase=False, analyzer='char', ngram_range=(2, 6), min_df=2), TfidfVectorizer(sublinear_tf=True, lowercase=True, token_pattern='[^\\s]+', ngram_range=(1, 3), min_df=2)]
    Xa = sp.hstack([v_.fit_transform(Dtr) for v_ in vs]).tocsr()
    Xb = sp.hstack([v_.transform(Dte) for v_ in vs]).tocsr()
    LP['distort'] = LogisticRegression(C=10, max_iter=3000).fit(Xa, FIT_Y, sample_weight=FIT_W).predict_log_proba(Xb) * a.distort_w
    log('distort done')
if a.prefit_core:
    from ntl660_readout import core_votes
    _pv = core_votes(a.prefit_core, test.text.tolist())
    assert set(_pv) == {'svc4', 'tfidf', 'distort'} and all((v.shape == (len(test), 18) and np.isfinite(v).all() for v in _pv.values()))
    LP.update(_pv)
    log('prefit core votes loaded', {k: round(float(abs(v).max()), 2) for k, v in _pv.items()})
from transformers import AutoTokenizer, AutoModel

@torch.no_grad()
def l24_predict(spec, texts, groups=None, train_texts=None):
    from transformers import AutoConfig
    from safetensors.torch import load_file
    from concurrent.futures import ThreadPoolExecutor
    adapters = spec.get('adapters') or [spec['adapter']]
    aw = spec.get('adapter_weights')
    if aw is None or len(aw) != len(adapters):
        aw = [1.0 / len(adapters)] * len(adapters)
    tok = AutoTokenizer.from_pretrained(spec['path'])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(spec['path'])
    tc = cfg.text_config if hasattr(cfg, 'text_config') else cfg
    tc.num_hidden_layers = spec['keep']
    tc.use_cache = False
    if getattr(tc, 'layer_types', None) is not None:
        tc.layer_types = tc.layer_types[:spec['keep']]
    _ms = []
    if spec.get('dp') and torch.cuda.device_count() >= 2:
        try:
            for _d in range(2):
                _ms.append(AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map={'': _d}).eval())
                for _n in ('vision_tower', 'multi_modal_projector'):
                    if getattr(_ms[-1], _n, None) is not None:
                        setattr(_ms[-1], _n, None)
                gc.collect()
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats(_d)
                _ms[-1](input_ids=torch.full((spec.get('bs', 16), spec.get('max_len', 384)), tok.pad_token_id, device=f'cuda:{_d}'), output_hidden_states=True, use_cache=False)
                assert torch.cuda.max_memory_allocated(_d) < torch.cuda.get_device_properties(_d).total_memory - (1 << 30), 'under 1 GiB to spare'
        except Exception as _e:
            log(f"  l24 {spec['name']}: one split copy instead of one per GPU ({_e!r:.80})")
            _ms = []
        gc.collect()
        torch.cuda.empty_cache()
    if not _ms:
        _ms = [AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map='auto').eval()]
    m = _ms[0]
    if spec.get('no_final_norm'):
        for _m in _ms:
            _m.norm = torch.nn.Identity()
    mods = dict(m.named_modules())
    first = next(m.parameters()).device
    L = spec['layers']
    _mods = [dict(_m.named_modules()) for _m in _ms]
    _eos = spec.get('eos_id')
    _tk = lambda tx: [x + ([_eos] if _eos is not None else []) for x in tok(list(tx), truncation=True, max_length=spec.get('max_len', 384) - (1 if _eos is not None else 0))['input_ids']]
    enc = _tk(texts)
    order = np.argsort([len(x) for x in enc])
    bs = spec.get('bs', 16)

    def lsm(z):
        return z - z.max(1, keepdims=True) - np.log(np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True))
    P_acc, C_acc = (0.0, 0.0)
    FEATS = None
    OVF = [0, 0]
    if train_texts is not None:
        enc_tr = _tk(train_texts)

    def featurize(enc_, norms, hid):
        order_ = np.argsort([len(x) for x in enc_])
        Z_ = np.zeros((len(enc_), hid * len(L)), np.float32)
        batches = [order_[i:i + bs] for i in range(0, len(order_), bs)]

        def feats(mm, b, check=True):
            dev = next(mm.parameters()).device
            width = max((len(enc_[j]) for j in b))
            ids = torch.full((len(b), width), tok.pad_token_id)
            att = torch.zeros((len(b), width), dtype=torch.long)
            for r, j in enumerate(b):
                ids[r, :len(enc_[j])] = torch.tensor(enc_[j])
                att[r, :len(enc_[j])] = 1
            with torch.no_grad():
                hs = mm(input_ids=ids.to(dev), attention_mask=att.to(dev), output_hidden_states=True, use_cache=False).hidden_states
                if check and (not all((torch.isfinite(hs[l]).all() for l in L))):
                    return None
                mask = att.unsqueeze(-1).float()
                mask[:, 0] = 0
                mg = {d_: mask.to(d_) for d_ in {hs[l].device for l in L}}
                return torch.cat([nm(((hs[l].float() * mg[hs[l].device]).sum(1) / mg[hs[l].device].sum(1)).cpu()) for l, nm in zip(L, norms)], -1).numpy()

        def put(b, r):
            Z_[b] = r

        def checked(k, b):
            mm = _ms[k]
            r = feats(mm, b)
            if r is None:
                mm.to(torch.bfloat16)
                r = feats(mm, b, check=False)
                mm.to(torch.float16)
                OVF[k] += 1
            put(b, r)

        def run_copy(k, n):
            for b in batches[k::n]:
                checked(k, b)
        if len(_ms) == 1:
            if spans_devices(_ms[0]):
                pipelined(batches, lambda b: feats(_ms[0], b), put, lambda b: checked(0, b), workers=2, tag='l24 ' + spec['name'], log=log)
            else:
                for b in batches:
                    checked(0, b)
        else:
            try:
                with ThreadPoolExecutor(len(_ms)) as _ex:
                    for _f in [_ex.submit(run_copy, k, len(_ms)) for k in range(len(_ms))]:
                        _f.result()
            except Exception as _e:
                log(f"  l24 {spec['name']}: two-copy pass failed ({_e!r:.80}), one copy from here on")
                del _ms[1:], _mods[1:]
                gc.collect()
                torch.cuda.empty_cache()
                for b in batches:
                    checked(0, b)
        return Z_
    for ai, ad in enumerate(adapters):
        adp = os.path.join(ad, 'adapter') if os.path.exists(os.path.join(ad, 'adapter', 'adapter_config.json')) else ad
        acfg = json.load(open(os.path.join(adp, 'adapter_config.json')))
        scale = acfg['lora_alpha'] / acfg['r']
        sd = load_file(os.path.join(adp, 'adapter_model.safetensors'))
        deltas = []
        for k, v in sd.items():
            if '.lora_A.' not in k:
                continue
            name = k.replace('base_model.model.', '', 1).split('.lora_A.')[0]
            if name not in mods and 'model.' + name in mods:
                name = 'model.' + name
            B = sd[k.replace('.lora_A.', '.lora_B.')]
            for _md in _mods:
                W = _md[name].weight
                W += (B.to(W.device, torch.float32) @ v.to(W.device, torch.float32) * scale).to(W.dtype)
                deltas.append((W, B, v))
        if os.path.exists(os.path.join(ad, 'heads.pt')):
            H = torch.load(os.path.join(ad, 'heads.pt'), map_location='cpu')
        else:
            H = dict(load_file(os.path.join(ad, 'heads.safetensors')))
        if os.path.exists(os.path.join(ad, 'meta.json')):
            _lab = json.load(open(os.path.join(ad, 'meta.json')))['labels']
            _perm = [_lab.index(l) for l in LABELS]
            for _p in ('plain', 'centered'):
                H[f'{_p}.weight'] = H[f'{_p}.weight'][_perm]
                H[f'{_p}.bias'] = H[f'{_p}.bias'][_perm]
        hid = H['norms.0.weight'].shape[0]
        norms = [torch.nn.LayerNorm(hid) for _ in L]
        for i, nm in enumerate(norms):
            nm.load_state_dict({'weight': H[f'norms.{i}.weight'], 'bias': H[f'norms.{i}.bias']})
        Z = featurize(enc, norms, hid)
        if train_texts is not None and ai == len(adapters) - 1:
            FEATS = (Z.copy(), featurize(enc_tr, norms, hid))
        for W, B, A_ in deltas:
            W -= (B.to(W.device, torch.float32) @ A_.to(W.device, torch.float32) * scale).to(W.dtype)
        torch.cuda.empty_cache()
        del deltas, sd
        gc.collect()
        hd = lambda p, X: lsm(X @ H[f'{p}.weight'].float().numpy().T + H[f'{p}.bias'].float().numpy())
        P_acc = P_acc + hd('plain', Z) * aw[ai]
        if groups is not None:
            Zc = Z.copy()
            for g in np.unique(groups):
                ii = groups == g
                Zc[ii] -= Zc[ii].mean(0)
            C_acc = C_acc + hd('centered', Zc) * aw[ai]
        log(f"  l24 {spec['name']}: adapter {os.path.basename(ad)} done (bf16 redo batches {sum(OVF)})")
    del m, mods, _ms, _mods
    gc.collect()
    torch.cuda.empty_cache()
    if train_texts is not None:
        return (P_acc, C_acc if groups is not None else None, FEATS)
    return (P_acc, C_acc if groups is not None else None)

def gpu_mem():
    return ' '.join((f'gpu{i}:{torch.cuda.memory_allocated(i) / 1000000000.0:.1f}GB' for i in range(torch.cuda.device_count())))
specs = json.loads(a.models)
failed = []

def run_l24(spec):
    log('l24', spec['name'])
    out = {}
    if spec.get('gauss', 0) > 0:
        plain, cent, (Zte_g, Ztr_g) = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None, train_texts=train.text.values)
        import gauss as _ga
        prm = _ga.fit(Ztr_g, y, Gtr, dict(red='wpca', d=128, se_shrink='lw', sp_shrink='lw', lam_floor=0.0001, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0))
        prm['T1'] = prm['Tg'] = 3.0
        _sz = np.bincount(Gte)[Gte]
        _rows = (_sz >= 2) & (_sz <= 18)
        _q = _ga.scores(Ztr_g[[0, 1]], np.zeros(2, dtype=int), prm)
        assert np.allclose(np.exp(_q).sum(1), 1)
        _G = np.zeros((len(test), 18))
        if _rows.any():
            _G[_rows] = _ga.scores(Zte_g[_rows], np.unique(Gte[_rows], return_inverse=True)[1], prm)
        out[spec['name'] + '_gauss'] = _G * spec['gauss']
        log(f"  gaussian head {spec['name']}: {int(_rows.sum())} grouped test rows, weight {spec['gauss']}")
    else:
        plain, cent = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None)
    out[spec['name']] = plain * spec.get('w', 1.0)
    if cent is not None:
        rows = np.bincount(Gte)[Gte] >= 2
        out[spec['name'] + '_twin'] = np.where(rows[:, None], cent, 0.0) * a.adapt_w * spec.get('w', 1.0) * spec.get('twin', 1.0)
    LP.update(out)

def run_enc(spec):
    log('enc', spec['name'])
    from safetensors.torch import load_file as _lf
    _ck = spec['ckpt']
    _meta = json.load(open(os.path.join(_ck, 'meta.json')))
    _perm = [_meta['labels'].index(l) for l in LABELS]
    _hd = _lf(os.path.join(_ck, 'head.safetensors'))
    _W = _hd['weight'].float().numpy()[_perm]
    _b = _hd['bias'].float().numpy()[_perm]
    _etok = AutoTokenizer.from_pretrained(os.path.join(_ck, 'encoder'))
    _enc = AutoModel.from_pretrained(os.path.join(_ck, 'encoder'), dtype=torch.float32).to('cuda:0').eval()
    _ids = _etok(list(test.text.values), truncation=True, max_length=_meta['max_len'])['input_ids']
    _ord = np.argsort([len(x) for x in _ids], kind='stable')
    Z_ = np.zeros((len(_ids), _enc.config.hidden_size), np.float32)
    with torch.inference_mode():
        for _s in range(0, len(_ord), 32):
            _idx = _ord[_s:_s + 32]
            _bt = _etok.pad({'input_ids': [_ids[i] for i in _idx]}, return_tensors='pt').to('cuda:0')
            _m = _bt['attention_mask'].unsqueeze(-1).float()
            Z_[_idx] = ((_enc(**_bt).last_hidden_state * _m).sum(1) / _m.sum(1).clamp(min=1)).cpu().numpy()
    del _enc
    _zz = Z_ @ _W.T + _b
    LP[spec['name']] = (_zz - _zz.max(1, keepdims=True) - np.log(np.exp(_zz - _zz.max(1, keepdims=True)).sum(1, keepdims=True))) * spec.get('w', 1.0)
RUNNERS = {'l24': run_l24, 'enc': run_enc}
for spec in specs:
    try:
        RUNNERS[spec['type']](spec)
    except Exception as _e:
        failed.append(spec['name'])
        log(f"MEMBER FAILED {spec['name']}: {_e!r}")
    gc.collect()
    torch.cuda.empty_cache()
    log(spec['name'], 'done', gpu_mem())
log('members failed:', failed if failed else 'none')
assert not failed, failed
tot = sum(LP.values())
if a.save_lp:
    np.save(a.save_lp, tot)
COMP_P = None
if a.comp_w > 0 and a.comp_path and os.path.exists(a.comp_path):
    try:
        _cm = json.load(open(a.comp_path.replace('comp_logit.npy', 'comp_meta.json')))
        assert _cm['ids'] == test['id'].astype(str).tolist(), 'comp stack row order differs'
        _cl = np.load(a.comp_path)[:, [_cm['labels'].index(l) for l in LABELS]]
        COMP_P = np.exp(_cl - _cl.max(1, keepdims=True))
        COMP_P /= COMP_P.sum(1, keepdims=True)
        assert np.isfinite(COMP_P).all()
        log(f"comp stack loaded ({_cm['seconds']:.0f}s)")
    except Exception as _e:
        log('COMP SKIPPED:', repr(_e))
        raise
        COMP_P = None
elif a.comp_w > 0:
    log('COMP SKIPPED: no stack output')

def blend(terms):
    total = sum(terms.values())
    active = sum((abs(v).max() > 0 for v in terms.values()))
    lpn = total / max(1, active) if a.sk_div else total
    P = np.exp(lpn - lpn.max(1, keepdims=True))
    P /= P.sum(1, keepdims=True)
    if COMP_P is not None:
        P = (1 - a.comp_w) * P + a.comp_w * COMP_P
        P /= P.sum(1, keepdims=True)
    return P

def decide_scores(P):
    Q = P.copy()
    rho = float(np.clip(a.sinkhorn_target_mix, 0.0, 1.0))
    marg = np.clip(P.mean(0), 1e-06, None)
    marg /= marg.sum()
    prior = np.bincount(y, minlength=18).astype(np.float64) + 1.0
    prior /= prior.sum()
    target = np.clip((1.0 - rho) * marg + rho * prior, 1e-06, None)
    target /= target.sum()
    for _ in range(100):
        Q *= len(Q) * target[None, :] / np.clip(Q.sum(0, keepdims=True), 1e-12, None)
        Q /= np.clip(Q.sum(1, keepdims=True), 1e-12, None)
    return (1 - a.sinkhorn) * np.log(P + 1e-12) + a.sinkhorn * np.log(Q + 1e-12)

def decide(P):
    return decide_scores(P).argmax(1)

def save_final_logits(z):
    if a.save_final_logits:
        np.savez(a.save_final_logits, ids=test.id.astype(str).values.astype('U'), labels=np.array(LABELS), logits=np.asarray(z, dtype=np.float64))

def write_pred(pred, tag):
    sub = pd.DataFrame({'id': test.id.values, 'model': [LABELS[i] for i in pred]})
    assert sub.model.isin(LABELS).all() and sub.id.is_unique and (len(sub) == len(test))
    sub.to_parquet(a.out, index=False)
    log(f'wrote {tag}', sub.model.value_counts().to_dict())

def write(P, tag):
    save_final_logits(decide_scores(P))
    write_pred(decide(P), tag)
P = blend(LP)
write(P, 'blend without ATLAS')

def atlas_lite(k):
    import atlas_infer
    scorer = atlas_infer.Scorer(a.atlas_base, a.atlas_dir, log=log)
    try:
        enc = scorer.encode(test.text.values)
        to_b = np.array([scorer.labels.index(l) for l in LABELS])
        cand = np.argsort(-P, 1)[:, :k]
        mean_lp, done = scorer.run_candidates(enc, to_b[cand], workers=a.atlas_workers)
        lp = scorer.candidate_logprobs(mean_lp[done], to_b[cand][done])
    finally:
        scorer.close()
    rows = np.repeat(lp.min(1, keepdims=True), 18, 1)
    np.put_along_axis(rows, cand[done], lp, 1)
    term = np.zeros((len(test), 18))
    term[done] = rows
    return (term, done)

def atlas_scores(rows, calibrate):
    import atlas_infer
    scorer = atlas_infer.Scorer(a.atlas_base, a.atlas_dir, log=log)
    try:
        enc = scorer.encode(test.text.values)
        out, done = scorer.run(enc, rows, batch=a.atlas_bs, chunk=a.atlas_chunk, workers=a.atlas_workers)
        order = [scorer.labels.index(l) for l in LABELS]
        if calibrate:
            res = np.zeros((len(test), 18))
            res[done] = scorer.calibrated(out[done])
        else:
            raw = out[..., 1:].mean(-1)
            res = np.divide(raw, raw.sum(1, keepdims=True), out=np.zeros_like(raw), where=raw.sum(1, keepdims=True) > 0)
    finally:
        scorer.close()
    return (res[:, order], done)
if a.atlas_mode != 'off' and a.atlas_dir:
    try:
        if a.atlas_mode == 'lite':
            term, done = atlas_lite(a.atlas_k)
            P = blend({**LP, 'atlas': a.atlas_member_w * term})
            log(f'atlas lite top-{a.atlas_k} on {int(done.sum())}/{len(test)} rows at weight {a.atlas_member_w}', gpu_mem())
        elif a.atlas_mode == 'head':
            lp, done = atlas_scores(np.arange(len(test)), True)
            P = blend({**LP, 'atlas': a.atlas_member_w * lp})
            log(f'atlas head on {int(done.sum())}/{len(test)} rows at weight {a.atlas_member_w}', gpu_mem())
        else:
            top2 = np.sort(P, 1)
            margin = top2[:, -1] - top2[:, -2]
            rows = np.where(margin < a.atlas_margin)[0] if a.atlas_margin > 0 else np.arange(len(test))
            A_, done = atlas_scores(rows, False)
            use = rows[done[rows]]
            mixed = P.copy()
            mixed[use] += a.atlas_w * (A_[use] - P[use])
            P = mixed / mixed.sum(1, keepdims=True)
            log(f'atlas mixed on {len(use)}/{len(rows)} rows at weight {a.atlas_w}', gpu_mem())
    except Exception as _e:
        log('ATLAS SKIPPED:', repr(_e))
    write(P, 'final blend')
if a.gsq_share > 0 and a.gsq_dir:
    try:
        import gsq_stage as _gs
        import shutil as _sh
        _sh.rmtree('/tmp/llama-3.1', ignore_errors=True)
        gc.collect()
        torch.cuda.empty_cache()
        Q27 = _gs.run(a.gsq_dir, test['id'].astype(str).tolist(), test.text.tolist(), Gte, a.gsq_work, LABELS, log)
        _agree = float((Q27.argmax(1) == P.argmax(1)).mean())
        log(f'GSQ 27B agreement with the blend {_agree:.3f}')
        if _agree < a.gsq_min_agree:
            raise RuntimeError(f'GSQ agreement {_agree:.3f} below {a.gsq_min_agree}: refusing to use it')
        P = (1 - a.gsq_share) * P + a.gsq_share * Q27
        P /= P.sum(1, keepdims=True)
        write(P, 'blend with GSQ 27B')
    except Exception as _e:
        log('GSQ SKIPPED:', repr(_e))
        raise

def rowsoftmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def native_stage(P):
    import subprocess
    import native_q8
    os.makedirs(a.native_work, exist_ok=True)
    out = os.path.join(a.native_work, 'native.npz')
    if os.path.exists(out):
        os.remove(out)
    gc.collect()
    torch.cuda.empty_cache()
    cmd = [sys.executable, '-u', os.path.join(os.path.dirname(os.path.abspath(__file__)), 'native_q8.py'), '--base', a.native_base, '--assets', a.native_dir, '--query', os.path.join(a.data, 'test.csv'), '--out', out, '--work', a.native_work]
    env = dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')
    code = subprocess.run(cmd, env=env).returncode
    if not os.path.exists(out):
        raise RuntimeError(f'native worker produced nothing (exit code {code})')
    z = np.load(out, allow_pickle=False)
    folds = z['folds'].tolist()
    assert z['ids'].astype(str).tolist() == test['id'].astype(str).tolist(), 'native row order differs'
    labels = z['labels'].astype(str).tolist()
    order = [labels.index(l) for l in LABELS]
    states = [json.load(open(os.path.join(a.native_dir, 'calibration', f'fold{k}.json')))['models']['qwen8'] for k in folds]
    lp = native_q8.calibrate(z['plain'][:, :, order], z['centered'][:, :, order], z['centered_bias'][:, order], Gte, states, bool(a.native_cent))
    share = (a.native_share * z['finite'])[:, None]
    logP = np.log(np.maximum(P, 1e-15))
    agree = [float((lp[i].argmax(1) == P.argmax(1)).mean()) for i in range(len(folds))]
    log(f'native folds {folds} (worker exit {code}); agreement with the blend per fold', [round(v, 3) for v in agree], 'rows without native', int((~z['finite']).sum()))
    if min(agree) < a.native_min_agree:
        raise RuntimeError(f'native agreement {min(agree):.3f} below {a.native_min_agree}: refusing to use it')
    probs = np.zeros_like(P)
    for i in range(len(folds)):
        probs += rowsoftmax(decide_scores(rowsoftmax((1 - share) * logP + share * lp[i]))) / len(folds)
    assert np.isfinite(probs).all() and np.allclose(probs.sum(1), 1)
    return probs
if a.native_share > 0 and a.native_dir and a.native_base:
    try:
        probs = native_stage(P)
        changed = int((probs.argmax(1) != decide(P)).sum())
        write_pred(probs.argmax(1), 'final blend with native q8')
        save_final_logits(np.log(np.maximum(probs, 1e-15)))
        log(f'native q8 at share {a.native_share}: {changed}/{len(test)} decisions differ from the blend-only decision', gpu_mem())
    except Exception as _e:
        log('NATIVE SKIPPED:', repr(_e))
        raise


In [ ]:
def run(cmd, env=None):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    for line in proc.stdout:
        if not re.search(r'Warning|warn', line):
            print(line, end='')
    return proc.wait()



# The reference complementary worker resolves a config-only directory.
# Its output is absent; make the effective zero contribution explicit.
COMP_SEC = 0.0


In [ ]:
LAYERS = [2, 4, 6, 8, 12, 16, 20, 24]
MODELS = [
    {"name": "q14v5", "type": "l24", "path": B14, "adapters": AQ14V5, "keep": 27, "layers": LAYERS, "w": 3.0, "bs": 8, "max_len": 384, "twin": 0.35, "gauss": 0.70},
    {"name": "q8v5", "type": "l24", "path": B8, "adapters": AQ8V5, "keep": 24, "layers": LAYERS, "w": 2.0, "bs": 16, "max_len": 384, "twin": 0.35, "gauss": 0.44, "dp": True},
    {"name": "ll8v5", "type": "l24", "path": BL, "adapters": ALL8V5, "keep": 24, "layers": LAYERS, "w": 1.0, "bs": 16, "max_len": 384, "twin": 0.35, "dp": True},
    {"name": "n14b_clean", "type": "l24", "path": B14, "adapters": [OQ4, N14B], "adapter_weights": [0.72, 0.28], "keep": N14BC['keep'], "layers": N14BC['layers'],
     "w": 2.5, "bs": 8, "max_len": N14BC['max_len'], "twin": 0.35, "no_final_norm": True, "eos_id": N14BC['eos_id']},
    {"name": "q25_14", "type": "l24", "path": B25_14, "adapters": [Q25_14], "keep": Q25_14C['keep'], "layers": Q25_14C['layers'], "w": 1.0, "bs": 8,
     "max_len": Q25_14C['max_len'], "twin": 0.35, "no_final_norm": True, "eos_id": Q25_14C['eos_id']},
    {"name": "deb", "type": "enc", "ckpt": DEB, "w": 1.0},
]
OUT = '/kaggle/working/submission.parquet'
cmd = [sys.executable, 'infer.py', '--data', DATA, '--models', json.dumps(MODELS),
       '--prefit_core', PREFIT, '--tfidf_w', '0', '--distort_w', '0', '--svc_w', '0',
       '--adapt_w', '1.0', '--adapt_thr', '0.15', '--sinkhorn', '0.30', '--sinkhorn_target_mix', '0.35',
       '--native_dir', NATIVE, '--native_base', NATIVE_BASE, '--native_share', '0.40', '--native_cent', '1',
       '--group_embed_base', NATIVE_BASE, '--group_embed_assets', NATIVE, '--group_sim', '0.70',
       '--gsq_dir', GSQ_ASSETS, '--gsq_share', '0.03',
       '--comp_path', '/kaggle/working/comp_logit.npy', '--comp_w', '0.0', '--out', OUT, '--save_final_logits', FINAL_LOGITS]
if FAST_COMMIT:
    ids = pd.read_csv(os.path.join(DATA, 'test.csv')).id
    pd.DataFrame({'id': ids, 'model': 'gpt-5.5'}).to_parquet(OUT, index=False)
    sys.path.insert(0, os.getcwd())
    from ntl660_readout import core_votes
    smoke = core_votes(PREFIT, pd.read_csv(os.path.join(DATA, 'test.csv')).text.fillna('').tolist())
    assert set(smoke) == {'svc4', 'tfidf', 'distort'} and all(len(v) == len(ids) and np.isfinite(v).all() for v in smoke.values())
    print('prefit core votes smoke ok', {k: v.shape for k, v in smoke.items()})
else:
    assert run(cmd, env=dict(os.environ, NTL_PIPE_WORKERS='1')) == 0, 'ensemble worker failed'


In [ ]:
submission = pd.read_parquet(OUT)
test_ids = pd.read_csv(os.path.join(DATA, 'test.csv')).id
assert len(submission) == len(test_ids) and set(submission.id) == set(test_ids)
assert list(submission.columns) == ['id', 'model'] and submission.id.is_unique and submission.model.notna().all()
print(submission.model.value_counts().to_string())
for path in ['/kaggle/working/open_pool.parquet', '/kaggle/working/extra_matched.parquet']:
    if os.path.exists(path):
        os.remove(path)
shutil.rmtree('/kaggle/working/deberta_v3_large', ignore_errors=True)
print('total', round(time.time() - NB_T0), 's')


## The GLM likelihood stage, at 256 tokens

The next cells are the 0.704's GLM stage unchanged except two: the cell writing `ll_model.json` (estopadilla's recipe
refitted on the first 256 tokens, `max_text_tokens` = 256) and, in `v1_stage.py`, the token budget (130,000 scored
positions, was 90,000).


In [ ]:
from pathlib import Path
Path('/kaggle/temp/v1').mkdir(parents=True,exist_ok=True)


In [ ]:
%%writefile /kaggle/temp/v1/segio.py
"""Read and write the binary files of ntl-segrun (see the header of tools/ntl-segrun/ntl-segrun.cpp)."""
import struct

import numpy as np


def write_texts(path, texts):
    with open(path, 'wb') as f:
        f.write(b'NTLT')
        f.write(struct.pack('<I', len(texts)))
        for t in texts:
            b = t.encode('utf-8')
            f.write(struct.pack('<I', len(b)))
            f.write(b)


def write_tokens(path, seqs):
    """seqs: list of (ids, n_prefix)."""
    with open(path, 'wb') as f:
        f.write(b'NTLK')
        f.write(struct.pack('<I', len(seqs)))
        for ids, n_prefix in seqs:
            ids = np.asarray(ids, dtype='<i4')
            f.write(struct.pack('<II', len(ids), n_prefix))
            f.write(ids.tobytes())


def read_tokens(path):
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLK'
        (n,) = struct.unpack('<I', f.read(4))
        for _ in range(n):
            nt, npre = struct.unpack('<II', f.read(8))
            ids = np.frombuffer(f.read(4 * nt), dtype='<i4')
            out.append((ids, npre))
    return out


def read_scores(path):
    """List of float32 arrays [n_scored, 4]: logprob, entropy, E[logp^2], rank."""
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLP'
        n, nf = struct.unpack('<II', f.read(8))
        for _ in range(n):
            (ns,) = struct.unpack('<I', f.read(4))
            out.append(np.frombuffer(f.read(4 * ns * nf), dtype='<f4').reshape(ns, nf))
    return out


def read_state(path):
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLS'
        dt, w, nt, nr = struct.unpack('<IIIQ', f.read(20))
        dtype = {0: '<f4', 1: '<f2'}.get(dt)
        if dtype is None:  # bf16
            raw = np.frombuffer(f.read(2 * w * nr), dtype='<u2').astype('<u4') << 16
            return raw.view('<f4').reshape(nr, w)
        return np.frombuffer(f.read(np.dtype(dtype).itemsize * w * nr), dtype=dtype).reshape(nr, w).astype('f4')


In [ ]:
%%writefile /kaggle/temp/v1/features.py
"""Per-text likelihood features from the per-token scores of ntl-segrun.

Per scored token: lp = log-prob of the true token, H = entropy of the next-token distribution,
m2 = E_p[logp^2] (so Var_p[logp] = m2 - H^2), rank = how many tokens beat the true one (0 = top-1).
"""
import numpy as np
import pandas as pd


def one(s, n_chars):
    lp, h, m2, rank = s[:, 0].astype(np.float64), s[:, 1].astype(np.float64), s[:, 2].astype(np.float64), s[:, 3]
    n = len(lp)
    var = np.maximum(m2 - h * h, 1e-9)
    half = max(n // 2, 1)
    q = np.quantile(lp, [0.05, 0.10, 0.25, 0.50, 0.75, 0.90]) if n else np.zeros(6)
    return {
        'n_tok': n,
        'lp_mean': lp.mean(),
        'lp_std': lp.std(),
        'lp_q05': q[0], 'lp_q10': q[1], 'lp_q25': q[2], 'lp_q50': q[3], 'lp_q75': q[4], 'lp_q90': q[5],
        'lp_min': lp.min(),
        'ent_mean': h.mean(),
        'ent_std': h.std(),
        # Fast-DetectGPT analytic curvature: how far the text's log-likelihood sits above the model's own expectation
        'fdg': (lp + h).sum() / np.sqrt(var.sum()),
        'surprise_mean': (-lp - h).mean(),          # observed surprisal minus expected surprisal
        'z_mean': ((lp + h) / np.sqrt(var)).mean(),
        'logrank_mean': np.log1p(rank).mean(),
        'top1': (rank == 0).mean(),
        'top5': (rank < 5).mean(),
        'top20': (rank < 20).mean(),
        'rank_ge100': (rank >= 100).mean(),
        'lp_mean_first': lp[:half].mean(),
        'lp_mean_second': lp[half:].mean() if n > half else lp.mean(),
        'bpc': -lp.sum() / max(n_chars, 1) / np.log(2),
        'tok_per_char': n / max(n_chars, 1),
    }


def text_features(scores, texts):
    return pd.DataFrame([one(s, len(t)) for s, t in zip(scores, texts)])


In [ ]:
%%writefile /kaggle/temp/v1/run_pipeline.py
"""Score text likelihood by streaming each model segment once per state chunk."""

import argparse
import glob
import json
import os
import re
import shutil
import struct
import subprocess
import sys
import threading
import time

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import features  # noqa: E402
import segio  # noqa: E402

CHAT_PREFIX = '[gMASK]<sop><|user|><|assistant|><think></think>'


def seg_index(path):
    m = re.search(r'seg(\d+)-of-(\d+)', os.path.basename(path))
    assert m, path
    return int(m.group(1)), int(m.group(2))


def gguf_kv_u32(path, key):
    """Read one uint32 metadata value from a GGUF header without the gguf package."""
    with open(path, 'rb') as f:
        assert f.read(4) == b'GGUF'
        _ver, _nt, nkv = struct.unpack('<IQQ', f.read(20))

        def rstr():
            (n,) = struct.unpack('<Q', f.read(8))
            return f.read(n).decode('utf-8', 'replace')

        sizes = {0: 1, 1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 4, 7: 1, 10: 8, 11: 8, 12: 8}

        def skip(t):
            if t == 8:
                rstr()
            elif t == 9:
                it, n = struct.unpack('<IQ', f.read(12))
                if it in sizes:
                    f.seek(sizes[it] * n, 1)
                else:
                    for _ in range(n):
                        skip(it)
            else:
                f.seek(sizes[t], 1)

        for _ in range(nkv):
            k = rstr()
            (t,) = struct.unpack('<I', f.read(4))
            if k == key:
                assert t == 4, (key, t)
                return struct.unpack('<I', f.read(4))[0]
            skip(t)
    return None


class EarlyFailure(RuntimeError):
    """ntl-segrun failed before writing any output (model load or context reservation, e.g. out of GPU memory)."""


_PREFETCHED = {}


def prefetch(path):
    """Read a file once in a daemon thread (page cache warm-up for the next mmap); each file at most once."""
    path = os.path.realpath(path)
    if path in _PREFETCHED:
        return
    info = _PREFETCHED[path] = {'s': None}

    def read():
        t0 = time.time()
        with open(path, 'rb', buffering=0) as f:
            try:
                os.posix_fadvise(f.fileno(), 0, 0, os.POSIX_FADV_SEQUENTIAL)
            except (AttributeError, OSError):
                pass
            buf = bytearray(16 << 20)
            while f.readinto(buf):
                pass
        info['s'] = time.time() - t0
    threading.Thread(target=read, daemon=True).start()


def run(cmd, log, on_ready=None):
    """Run one ntl-segrun call; on_ready() is called when it prints "context ready" (the model is loaded)."""
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
    lines = []
    for line in p.stdout:
        lines.append(line)
        if on_ready is not None and 'context ready' in line:
            on_ready()
            on_ready = None
    p.wait()
    out = ''.join(lines)
    with open(log, 'a') as lf:
        lf.write('$ ' + ' '.join(cmd) + '\n' + out)
    dt = time.time() - t0
    if p.returncode != 0:
        print('\n'.join(out.splitlines()[-40:]), flush=True)
        err = EarlyFailure if 'context ready' not in out else RuntimeError
        raise err(f'failed ({p.returncode}): {" ".join(cmd)}')
    return dt


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--bin', required=True)
    ap.add_argument('--segs', required=True, help='JSON list of the pinned segment paths')
    ap.add_argument('--csv', required=True)
    ap.add_argument('--text-col', default='text')
    ap.add_argument('--id-col', default='id')
    ap.add_argument('--out', required=True)
    ap.add_argument('--tag', required=True)
    ap.add_argument('--work', default='/tmp/ntl')
    ap.add_argument('--prefix', default=CHAT_PREFIX)
    ap.add_argument('--max-text-tokens', type=int, default=320)
    ap.add_argument('--seq-batch', type=int, default=8)
    ap.add_argument('--state-dtype', default='auto', help='auto | f32 | f16 | bf16')
    ap.add_argument('--threads', type=int, default=min(4, os.cpu_count() or 4))
    ap.add_argument('--limit', type=int, default=-1)
    ap.add_argument('--disk-frac', type=float, default=0.8, help='share of the free work disk the state may use')
    ap.add_argument('--pack', action='store_true', help='sort by length and pack padded decodes (ntl-segrun --pack)')
    ap.add_argument('--ubatch-div', type=int, default=1, help='n_ubatch = n_batch / P (pipeline parallel on 2 GPUs)')
    ap.add_argument('--prefetch', action='store_true', help='read the next segment while the current one computes')
    ap.add_argument('--prefetch-after', default='', help='with --prefetch: file to read during the last segment')
    a = ap.parse_args()
    main_run(a)


def main_run(a):

    t_start = time.time()

    os.makedirs(a.out, exist_ok=True)
    os.makedirs(a.work, exist_ok=True)
    log = os.path.join(a.out, f'{a.tag}_run.log')
    segrun = os.path.join(a.bin, 'ntl-segrun')

    segs = sorted(json.loads(a.segs), key=lambda p: seg_index(p)[0])
    assert segs, a.segs
    n_seg = seg_index(segs[0])[1]
    assert [seg_index(p)[0] for p in segs] == list(range(n_seg)), segs
    ranges = [(gguf_kv_u32(p, 'ntl.layer_start'), gguf_kv_u32(p, 'ntl.layer_end')) for p in segs]
    print(f'[{a.tag}] {n_seg} segments, layers {ranges}', flush=True)
    for (s0, e0), (s1, _e1) in zip(ranges, ranges[1:]):
        assert e0 == s1, ranges

    df = pd.read_csv(a.csv)
    if a.limit > 0:
        df = df.iloc[:a.limit]
    texts = df[a.text_col].astype(str).tolist()
    tx = os.path.join(a.work, f'{a.tag}_texts.bin')
    tk = os.path.join(a.work, f'{a.tag}_tok.bin')
    segio.write_texts(tx, texts)
    run([segrun, 'tokenize', '--model', segs[0], '--texts', tx, '--out', tk, '--prefix', a.prefix,
         '--max-text-tokens', str(a.max_text_tokens)], log)
    seqs = segio.read_tokens(tk)
    n_rows = np.array([len(ids) for ids, _ in seqs])
    # processing order: longest first when packing (similar lengths share a decode, little padding)
    order = np.argsort(-n_rows, kind='stable') if a.pack else np.arange(len(seqs))
    print(f'[{a.tag}] {len(seqs)} texts, {n_rows.sum()} rows, max {n_rows.max()}, prefix {seqs[0][1]}', flush=True)

    # state width: glm5-next hc*n_embd, glm-dsa n_embd (read from the first segment's metadata)
    n_embd = gguf_kv_u32(segs[0], 'glm5-next.embedding_length')
    if n_embd is not None:
        width = gguf_kv_u32(segs[0], 'glm5-next.hyper_connection.count') * n_embd
    else:
        width = gguf_kv_u32(segs[0], 'glm-dsa.embedding_length')
    free = shutil.disk_usage(a.work).free
    if a.state_dtype == 'auto':
        # exact f32 if the whole state fits in one chunk, else bf16 (half the disk, no overflow risk)
        a.state_dtype = 'f32' if n_rows.sum() * width * 4 <= free * a.disk_frac else 'bf16'
        print(f'[{a.tag}] state dtype auto -> {a.state_dtype}', flush=True)
    el = 2 if a.state_dtype in ('f16', 'bf16') else 4
    max_rows = int(free * a.disk_frac / (width * el))
    chunks, cur, cur_rows = [], [], 0
    for i in order:
        r = n_rows[i]
        if cur and cur_rows + r > max_rows:
            chunks.append(cur)
            cur, cur_rows = [], 0
        cur.append(i)
        cur_rows += r
    chunks.append(cur)
    print(f'[{a.tag}] state width {width}, {el} B/value, {free/2**30:.1f} GiB free on {a.work} -> '
          f'{len(chunks)} chunk(s)', flush=True)

    multi_gpu = False
    try:
        out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
        multi_gpu = out.count('GPU ') > 1
    except FileNotFoundError:
        pass

    timing = {'tag': a.tag, 'n_texts': len(seqs), 'n_rows': int(n_rows.sum()), 'chunks': [], 'segments': segs,
              'ranges': ranges, 'seq_batch': a.seq_batch, 'state_dtype': a.state_dtype, 'prefix': a.prefix,
              'pack': a.pack, 'ubatch_div': a.ubatch_div}
    all_scores = [None] * len(seqs)
    st = os.path.join(a.work, f'{a.tag}_state.bin')
    done = 0
    for ci, idx in enumerate(chunks):
        ctk = os.path.join(a.work, f'{a.tag}_tok_c{ci}.bin')
        segio.write_tokens(ctk, [seqs[i] for i in idx])
        sc = os.path.join(a.work, f'{a.tag}_scores_c{ci}.bin')
        seg_t = []
        for k, p in enumerate(segs):
            while True:
                cmd = [segrun, 'run', '--model', p, '--tokens', ctk, '--threads', str(a.threads),
                       '--seq-batch', str(a.seq_batch), '--state-dtype', a.state_dtype]
                if multi_gpu:
                    cmd.append('--split-mid')
                if a.pack:
                    cmd.append('--pack')
                if a.ubatch_div > 1:
                    cmd += ['--ubatch-div', str(a.ubatch_div)]
                if k > 0:
                    cmd += ['--state-in', st]
                cmd += ['--out', sc if k == n_seg - 1 else st]
                nxt = (segs[k + 1] if k + 1 < n_seg else segs[0] if ci + 1 < len(chunks) else a.prefetch_after)
                ready = (lambda f=nxt: prefetch(f)) if a.prefetch and nxt else None
                try:
                    dt = run(cmd, log, on_ready=ready)
                    break
                except EarlyFailure:
                    # nothing written yet (the input state is intact): retry with half the texts per decode
                    if a.seq_batch == 1:
                        raise
                    a.seq_batch //= 2
                    timing['seq_batch_reduced_to'] = a.seq_batch
                    print(f'[{a.tag}] segment {k+1} failed before writing: retrying with seq-batch {a.seq_batch}', flush=True)
            seg_t.append(dt)
            print(f'[{a.tag}] chunk {ci+1}/{len(chunks)} segment {k+1}/{n_seg} {os.path.basename(p)}: {dt:.0f} s '
                  f'(total {time.time()-t_start:.0f} s)', flush=True)
        for i, s in zip(idx, segio.read_scores(sc)):
            all_scores[i] = s
        done += len(idx)
        timing['chunks'].append({'n_texts': len(idx), 'seg_seconds': seg_t})
        for f in (ctk, sc, st):
            if os.path.exists(f):
                os.remove(f)

    ok = [i for i, s in enumerate(all_scores) if s is not None]
    off = np.zeros(len(ok) + 1, dtype=np.int64)
    off[1:] = np.cumsum([len(all_scores[i]) for i in ok])
    np.savez_compressed(os.path.join(a.out, f'{a.tag}_scores.npz'),
                        ids=df[a.id_col].astype(str).values[ok],
                        scores=np.concatenate([all_scores[i] for i in ok]).astype(np.float32),
                        offsets=off,
                        n_prefix=np.array([seqs[i][1] for i in ok]),
                        tokens=np.concatenate([seqs[i][0][seqs[i][1]:] for i in ok]).astype(np.int32))
    feats = features.text_features([all_scores[i] for i in ok], [texts[i] for i in ok])
    feats.insert(0, a.id_col, df[a.id_col].astype(str).values[ok])
    feats.to_csv(os.path.join(a.out, f'{a.tag}_features.csv'), index=False)
    timing['total_s'] = time.time() - t_start
    if a.prefetch:
        timing['prefetch_s'] = {os.path.basename(k): v['s'] for k, v in _PREFETCHED.items()}
    timing['n_done'] = len(ok)
    with open(os.path.join(a.out, f'{a.tag}_timing.json'), 'w') as f:
        json.dump(timing, f, indent=1)
    print(f'[{a.tag}] done: {len(ok)}/{len(seqs)} texts in {timing["total_s"]:.0f} s', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/temp/v1/v1_stage.py
"""Rank by GLM probability and add two-model likelihood evidence on selected texts."""

import argparse
import glob
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import log_softmax

PREFIX = '[gMASK]<sop><|system|>Reasoning Effort: Max<|user|><|assistant|><think></think>'


def entropy(z):
    return float(-(np.exp(z) * z).sum(1).mean())


def match_temperature(z, h_ref):
    """T' with mean entropy of log_softmax(T' z) equal to h_ref (entropy decreases with T')."""
    lo, hi = 1e-3, 1e3
    if entropy(log_softmax(z * hi, 1)) > h_ref or entropy(log_softmax(z * lo, 1)) < h_ref:
        raise ValueError('entropy_ref out of reach')
    for _ in range(100):
        mid = np.sqrt(lo * hi)
        if entropy(log_softmax(z * mid, 1)) > h_ref:
            lo = mid
        else:
            hi = mid
    return float(np.sqrt(lo * hi))


def member_logp(model, feats, ids):
    """feats: {'flash': DataFrame, 'glm53': DataFrame} indexed by id -> (n, 18) log-probs, model['classes'] order."""
    blocks = [feats[t].add_prefix(f'{t}:') for t in ('flash', 'glm53')]
    F = pd.concat(blocks, axis=1).reindex(ids)
    for c in model['diff_features']:
        F[f'd:{c}'] = F[f'flash:{c}'] - F[f'glm53:{c}']
    X = F[model['features']].to_numpy(np.float64)
    assert np.isfinite(X).all(), 'non-finite likelihood features'
    z = ((X - np.asarray(model['mean'])) / np.asarray(model['scale'])) @ np.asarray(model['coef']).T + np.asarray(model['intercept'])
    return log_softmax(z, 1)


def main():
    ap = argparse.ArgumentParser()
    for k in ('--test-csv', '--final-logits', '--out', '--report', '--bin', '--pipeline', '--flash-segs', '--glm53-segs',
              '--model', '--work', '--outdir'):
        ap.add_argument(k, required=True)
    ap.add_argument('--pack', action='store_true')
    ap.add_argument('--ubatch-div', type=int, default=1)
    ap.add_argument('--seq-batch', type=int, default=8)
    ap.add_argument('--max-text-tokens', type=int, default=0,
                    help='0 = the cut the member was trained with (model max_text_tokens); any other value must match it')
    ap.add_argument('--select-classes', default='glm-5.3,glm-5.3-flash',
                    help='texts are scored in decreasing order of the S10 probability of these classes')
    ap.add_argument('--max-cover', type=float, default=0.15, help='at most this fraction of the test is scored')
    ap.add_argument('--min-texts', type=int, default=50, help='fewer texts than this fit: keep S10')
    ap.add_argument('--prefetch', action='store_true', help='run_pipeline --prefetch (page-cache warm-up of segments)')
    ap.add_argument('--max-changed', type=float, default=0.25,
                    help='sanity guard: the blend changes ~11%% of the predictions on train OOF (W=1); far more means '
                         'something is broken (e.g. features unlike train): fail instead of submitting it')
    a = ap.parse_args()

    t0 = time.monotonic()
    rep = {'status': 'started'}
    out_report = Path(a.report)

    def save(**kw):
        rep.update(kw)
        out_report.write_text(json.dumps(rep, indent=1))
        print('V1', json.dumps({k: v for k, v in kw.items() if not isinstance(v, dict)}), flush=True)

    model = json.loads(Path(a.model).read_text())
    assert model['format'] == 'ntl-v1-ll-model-1'
    a.max_text_tokens = a.max_text_tokens or model['max_text_tokens']
    assert a.max_text_tokens == model['max_text_tokens'], 'the test must be cut like the train features'
    frame = pd.read_csv(a.test_csv, dtype={'id': str})
    frame['text'] = frame.text.fillna('').astype(str)
    ids = frame.id.tolist()
    with np.load(a.final_logits, allow_pickle=False) as f:
        assert f['ids'].astype(str).tolist() == ids, 'final-logits ids differ from the test order'
        labels = f['labels'].astype(str).tolist()
        z = f['logits'].astype(np.float64)
    assert sorted(labels) == sorted(model['classes']) and z.shape == (len(ids), 18) and np.isfinite(z).all()

    zl = log_softmax(z, 1)
    sel_cls = a.select_classes.split(',')
    assert all(c in labels for c in sel_cls), sel_cls
    mass = np.exp(zl[:, [labels.index(c) for c in sel_cls]]).sum(1)
    fable_mass = np.exp(zl[:, labels.index('claude-fable-5')])
    priority = mass + 0.35 * np.minimum(fable_mass, mass)
    order = np.argsort(-priority, kind='stable')

    # Bound work before starting, below the measured 160k-token pass.
    tpc = model.get('tok_per_char', 0.235)
    rows_i = np.minimum(frame.text.str.len().to_numpy() * tpc * 1.05, a.max_text_tokens) + 13
    cum = np.cumsum(rows_i[order])
    n_sel = min(int((cum <= 130000).sum()), int(a.max_cover * len(ids)))  # ours: room for 256-token rows
    save(status='selected', n_texts=len(ids), n_selected=n_sel,
         estimated_token_rows=int(cum[n_sel-1]) if n_sel else 0)
    if n_sel < a.min_texts:
        np.savez(Path(a.outdir)/'post-glm.npz', ids=np.asarray(ids), labels=np.asarray(labels), logits=zl)
        save(status='fallback_s10', reason='Insufficient selected texts')
        return
    sel = np.sort(order[:n_sel])
    sel_ids = [ids[i] for i in sel]

    Path(a.work).mkdir(parents=True, exist_ok=True)
    Path(a.outdir).mkdir(parents=True, exist_ok=True)
    times = {}
    for tag, segs in (('flash', a.flash_segs), ('glm53', a.glm53_segs)):
        query = Path(a.work) / f'v1_query_{tag}.csv'
        frame.iloc[sel][['id', 'text']].to_csv(query, index=False)
        cmd = [sys.executable, a.pipeline, '--bin', a.bin, '--segs', segs, '--csv', str(query), '--out', a.outdir,
               '--tag', tag, '--work', str(Path(a.work) / 'state'), '--seq-batch', str(a.seq_batch),
               '--state-dtype', 'auto', '--max-text-tokens', str(a.max_text_tokens), '--prefix', PREFIX]
        if a.pack:
            cmd.append('--pack')
        if a.ubatch_div > 1:
            cmd += ['--ubatch-div', str(a.ubatch_div)]
        if a.prefetch:
            cmd.append('--prefetch')
            if tag == 'flash':
                cmd += ['--prefetch-after', json.loads(a.glm53_segs)[0]]
        t = time.monotonic()
        with open(Path(a.outdir) / f'{tag}_pipeline.txt', 'w') as log:
            code = subprocess.call(cmd, stdout=log, stderr=subprocess.STDOUT)
        times[tag] = time.monotonic() - t
        if code != 0:
            raise RuntimeError(f'V1 {tag} pipeline failed with exit code {code}; see {tag}_pipeline.txt')
        save(status=f'{tag}_done', times=times)

    feats = {t: pd.read_csv(Path(a.outdir) / f'{t}_features.csv', dtype={'id': str}).set_index('id')
             for t in ('flash', 'glm53')}
    for t, f in feats.items():
        assert set(sel_ids) <= set(f.index), f'{t}: features missing for some texts'
        feats[t] = f.loc[sel_ids]
    ll = member_logp(model, feats, sel_ids)[:, [model['classes'].index(c) for c in labels]]

    tp = match_temperature(zl, model['entropy_ref'])
    w = model['W'] / tp
    before = zl.argmax(1)
    post_ll = zl.copy()
    post_ll[sel] += w * ll
    after = post_ll.argmax(1)
    np.savez(Path(a.outdir)/'post-glm.npz', ids=np.asarray(ids), labels=np.asarray(labels), logits=post_ll)
    changed = float((before != after).mean())
    changed_sel = float((before[sel] != after[sel]).mean())
    agrees = float((ll.argmax(1) == before[sel]).mean())
    if changed > a.max_changed:
        save(status='error_sanity', changed=changed, changed_selected=changed_sel, temperature=tp, weight=w,
             member_argmax_agrees_s10=agrees)
        raise RuntimeError(f'V1 blend changes {changed:.1%} of the predictions (> {a.max_changed:.0%}): not submitted')
    sub = pd.DataFrame({'id': ids, 'model': np.asarray(labels)[after]})
    assert len(sub) == len(frame) and sub.id.is_unique and sub.model.isin(labels).all()
    sub.to_parquet(a.out, index=False)
    assert pd.read_parquet(a.out).equals(sub)
    count = lambda p: {labels[i]: int(n) for i, n in zip(*np.unique(p, return_counts=True))}
    save(status='applied', n_blended=len(sel_ids), entropy_s10=entropy(zl), temperature=tp, weight=w, changed=changed,
         changed_selected=changed_sel, member_argmax_agrees_s10=agrees,
         counts_s10=count(before), counts_v1=count(after), times=times, total_s=time.monotonic() - t0)


if __name__ == '__main__':
    main()


In [ ]:
# ours (exp060): estopadilla's recipe refitted on the first 256 tokens (his: 160)
Path('/kaggle/temp/v1/ll_model.json').write_text('{"format": "ntl-v1-ll-model-1", "features": ["flash:n_tok", "flash:lp_mean", "flash:lp_std", "flash:lp_q05", "flash:lp_q10", "flash:lp_q25", "flash:lp_q50", "flash:lp_q75", "flash:lp_q90", "flash:lp_min", "flash:ent_mean", "flash:ent_std", "flash:fdg", "flash:surprise_mean", "flash:z_mean", "flash:logrank_mean", "flash:top1", "flash:top5", "flash:top20", "flash:rank_ge100", "flash:lp_mean_first", "flash:lp_mean_second", "flash:bpc", "flash:tok_per_char", "glm53:n_tok", "glm53:lp_mean", "glm53:lp_std", "glm53:lp_q05", "glm53:lp_q10", "glm53:lp_q25", "glm53:lp_q50", "glm53:lp_q75", "glm53:lp_q90", "glm53:lp_min", "glm53:ent_mean", "glm53:ent_std", "glm53:fdg", "glm53:surprise_mean", "glm53:z_mean", "glm53:logrank_mean", "glm53:top1", "glm53:top5", "glm53:top20", "glm53:rank_ge100", "glm53:lp_mean_first", "glm53:lp_mean_second", "glm53:bpc", "glm53:tok_per_char", "d:lp_mean", "d:lp_q50", "d:ent_mean", "d:top1", "d:logrank_mean", "d:fdg"], "diff_features": ["lp_mean", "lp_q50", "ent_mean", "top1", "logrank_mean", "fdg"], "mean": [143.4942571322712, -2.7692889704619112, 3.0811616330999563, -8.998225405252434, -7.300399023172988, -4.358984172211704, -1.614914514211349, -0.30412007728078283, -0.03360296263390142, -13.587119730016221, 1.6514205775048167, 1.2328850771047561, -7.0541035326366055, 1.1178683929570943, -0.6463722096816437, 1.306881561170243, 0.4510668339149418, 0.7113314745377359, 0.8434471700437572, 0.06857922094967255, -2.957147819243623, -2.5833207663172812, 0.7732457002401327, 0.1932137772659812, 143.4942571322712, -2.796258105180564, 3.046951533681022, -8.922277576908355, -7.241702206605625, -4.392754349455917, -1.7019232843773768, -0.33704813000524453, -0.03854291351721504, -13.434241818692694, 1.8369970738197667, 1.3755185885520542, -5.75770463027778, 0.9592610313607977, -0.5446435082093042, 1.3578542278724461, 0.4467821959926888, 0.7023185343354684, 0.8358182089345989, 0.07394616453294502, -3.0021249163136736, -2.592449064618259, 0.7809880612844654, 0.1932137772659812, 0.026969134718653465, 0.08700877016602776, -0.18557649631495005, 0.004284637922253058, -0.05097266670220322, -1.296398902358825], "scale": [66.52493330333779, 0.6000917390135948, 0.3919399576531433, 1.2640758631733466, 1.250907442175144, 1.1211864995474472, 0.6736264622535302, 0.2543733235734339, 0.06346921899573942, 1.864618533277407, 0.363983066008332, 0.1600137724660138, 2.325057961794335, 0.3480945338561963, 0.20294788907576836, 0.3625587361966399, 0.07794190284738825, 0.08325717071180681, 0.06731823599189385, 0.04148542569614112, 0.7009652119736901, 0.6248319606586527, 0.18611688290475717, 0.017444411249817417, 66.52493330333779, 0.6014583614767404, 0.4140393919193309, 1.3600659648436992, 1.2750536871047795, 1.075880925888507, 0.6773109085038506, 0.2715788293556719, 0.06801915299716463, 1.964273842291717, 0.39772588133409975, 0.1628603518128719, 2.1233218650084025, 0.35713954949245563, 0.2375375763915939, 0.3745237455812487, 0.07618027679361748, 0.08266446742035992, 0.06817188608754908, 0.04356849508076967, 0.7176473287505805, 0.6386021803528715, 0.188586251323954, 0.017444411249817417, 0.30536438106693636, 0.3516630087724693, 0.15839009668590556, 0.042202102684828616, 0.1769998997164424, 2.0143783275795224], "coef": [[-0.10001914519901488, 0.09113964832588761, -0.1081012704485882, 0.08468668096404673, 0.0732126225445589, 0.22457852419365168, 0.030987679536597985, 0.09325319542126428, 0.055666691486554724, -0.031365138933571485, -0.2098648630922391, 0.04145421994662328, 0.311441218967152, 0.06232532875027907, 0.07690590940480001, -0.006847963861688462, 0.02925646956674203, 0.1502839229465678, -0.008618296436837447, 0.13746841698203385, -0.050797584383270644, 0.2319884491716611, 0.027261600114791684, 0.22002426027906613, -0.10001914519901488, 0.05485259350576765, -0.002470999406442335, -0.13582483849142338, 0.12031591421128042, 0.13357603181712263, 0.09286645681830828, 0.07347204151248898, 0.1534481465121828, -0.16547470075052292, -0.17384848552383134, -0.08665639414900557, 0.24261948201109448, 0.10122791262466822, -0.030522488496267684, 0.03610309731580208, -0.005065832755990551, 0.0906344068212511, -0.04424699376951948, -0.037003972023946195, 0.08065449811638702, 0.01731687711568543, 0.07237513575566364, 0.22002426027906613, 0.07106460474669422, -0.11950458890786629, -0.045730221419911406, 0.06317745516335595, -0.09041957867011056, 0.10373405625864957], [-0.265187568856581, 0.08766000514693394, -0.089976210269336, 0.021745111416509928, 0.019002480840466546, -0.053596278642328604, 0.08714219100332922, 0.14346229846772018, -0.004745114155866776, -0.07085307620619789, -0.18690077643631176, 0.1976073004856582, 0.04399313536039066, 0.044311734933574556, 0.07586323359378046, -0.08290143816980514, 0.14561116337979496, 0.16088214105403154, 0.01925169845254451, -0.10164800378706745, -0.24129942388859166, 0.43436712428812524, -0.0789733469184094, -0.08099907872511539, -0.265187568856581, 0.04980574105423502, 0.01542500601427858, -0.15932389179856637, -0.053932273517463186, 0.20931794909553333, 0.06807528919162853, -0.10045109946758296, 0.013227470106692034, -0.12503300961525085, -0.10622361685259746, -0.013080736839030096, 0.0878546097975584, 0.0344173649825351, -0.05172516857427955, -0.043116349510401844, 0.16486242743501234, 0.16114691671733533, 0.11629984688717004, -0.04203256382673811, -0.017420454735624265, 0.11398812637551174, -0.05360103477367954, -0.08099907872511539, 0.0741670179240225, 0.03581027732631913, -0.16276797953192734, -0.028672841594188696, -0.07857938879956522, -0.04182780520291247], [-0.04360235451500243, 0.0584787688990483, -0.09425879684016918, 0.08971445018183824, -0.09428368173303199, 0.04523469436008217, 0.07894687607939459, -0.16475525188387505, -0.10710070342019179, 0.022223711906692086, -0.032892811930850266, -0.07605799376344158, 0.04487386504291967, -0.06641931239669832, 0.03604699440122315, -0.08260677444467474, -0.009727687954975701, 0.1611056631755171, 0.08894756343965216, -0.07520717719107697, -0.039885312113864445, 0.16079081566135398, 0.04721314293177661, 0.10857514444714524, -0.04360235451500243, 0.042004471264438126, 0.06753170022710903, -0.008845909985698673, 0.014452397229654659, 0.15344103610432577, 0.04028048102243445, 0.013454107821164712, -0.16727425820077493, -0.1734251781436699, -0.029863285023719467, 0.20101945748150027, -0.04926934847024424, -0.03748265663895095, -0.022413643951636055, -0.029032454035870025, 0.009149797036980501, 0.04649511125080154, 0.27739872364104873, -0.026787985405680198, 0.02488887720304463, 0.050685081280180164, 0.028052388601078608, 0.10857514444714524, 0.032186745645504264, 0.07364520861450533, -0.0005999439524581107, -0.034482371441699695, -0.10777669561154295, 0.10372878779289957], [0.1428874369624528, 0.07403701288793799, 0.07325975252630214, 0.028805966021127182, 0.10658110474330956, 0.11658660543286506, 0.12325622212691491, -0.10701731660645958, 0.05931072621117446, 0.025450574980015243, -0.1081649059250317, 0.18240103189027113, 0.1707764989858272, -0.014532850217890888, 0.1875045713796292, -0.044047437109893194, -0.14243784864797202, 0.19976826101341605, 0.11056687668939877, 0.054016034757129024, -0.11908005525892608, 0.26551988595722803, 0.0757305885536293, 0.34502699325685116, 0.1428874369624528, 0.0672893430830868, 0.08698838271447569, 0.10584338881020655, -0.10272076789111413, 0.0824175551528318, 0.07535794493656159, -0.0030974979712087886, -0.18874084871504354, -0.08168752745688805, -0.06797786887208666, 0.1476276904978388, -0.0296674850545706, -0.03761885304029829, -0.14820082300484075, -0.022173911529068197, -0.1054075336424251, 0.20939489124962535, 0.16726848475216038, 0.1425949741866988, -0.04771933427926447, 0.172090132563881, 0.12147200180727051, 0.34502699325685116, 0.012959146596824175, 0.09096178419726803, -0.07786873385897015, -0.07279025633012069, -0.043305825277244206, 0.22838752385871153], [0.061075077819008526, 0.039809516949501515, 0.016958460561901242, 0.02858737430974243, 0.052392271447257235, -0.06923121583005071, -0.06630881998921231, -0.08888585676408878, 0.03576728508909908, 0.01820374634690207, -0.15602491753546274, -0.037200146771930585, 0.21198462284843392, 0.09451761634478681, 0.06364877121359477, 0.03380048717932783, 0.05400153252097128, 0.16243025133411934, 0.1517439489651054, 0.24508609293155167, -0.06402128946498974, 0.15649663740709738, -0.029954736930551614, 0.004492140419269401, 0.061075077819008526, 0.05719096435709385, 0.02076387842701089, -0.020666321004192547, 0.0836194650898957, 0.14951008476059882, -0.09266784828612572, 0.004868245647438904, -0.14839846397689319, 0.039644948010176546, -0.1029166638528895, 0.04061278333527312, 0.036564538428196555, 0.018297153397460575, -0.0279247393733934, 0.09052267987764318, 0.06720585459695382, 0.03963985449110872, 0.09618266859712983, -0.032965500648632476, -0.10497416249834643, 0.2297573920374832, -0.012392240073125758, 0.004492140419269401, -0.03441338318915239, 0.05146281591972224, -0.10011867758907762, -0.021581351308245064, -0.1223064603309965, 0.2061371717110071], [0.006421125517038126, -0.04073311504523574, 0.12500330722248387, -0.10955059849987923, -0.20799824008443146, 0.008897603153164558, 0.15447789400894849, -0.08680664472648594, -0.040640677266554834, 0.05255691967575427, 0.10564201668710194, -0.0034728558570363438, -0.16493195671467628, -0.04024280167503442, -0.022683367084411103, 0.04633722517300894, -0.11696130340072682, 0.029809077676549722, -0.17158888530813504, 0.05625456401833095, 0.1912461249325846, -0.28682420104058565, -0.1256940204068061, -0.22992228300515594, 0.006421125517038126, -0.04011938952926947, 0.12486804600277163, -0.17746750626627314, -0.09206750963570551, -0.06025567671079671, 0.19338685459943597, 0.07750009016034606, 0.148228698495344, 0.04185146167309275, 0.10603930931240571, -0.05054266295604908, 0.16638884497270137, -0.05052488722757801, 0.12963329984136918, 0.037955012239989615, -0.04647516606983448, 0.14300524015996077, -0.0487223705904059, 0.17277347247056868, 0.12880747357139197, -0.21795164534765846, -0.11251544281077967, -0.22992228300515594, -0.001026522975992271, -0.07655803502847648, -0.023503190522342447, -0.13211890348151856, 0.014604033414839889, -0.3657572272548791], [-0.09894586139473638, -0.05360732365530696, 0.15020249358913457, -0.1357891308670035, -0.07854866030876921, -0.24456070661486135, 0.03913624984824406, 0.004996875339836092, 0.04848813576967819, 0.02970235925202268, 0.09869185886153631, 0.1393839640798377, -0.29081402091878833, -0.010781132529522043, -0.062312720721812195, 0.015974740854560576, 0.06072159169064176, -0.012071489690290357, -0.1658570981839089, -0.04677009811571464, -0.024156035011110428, -0.07973965319446287, -0.13404123443592858, -0.3237407503283789, -0.09894586139473638, -0.02370716239322714, -0.07975839227064727, 0.14922156298992623, -0.13439232511168958, -0.25593799198222833, 0.07709572889785647, 0.08329031889263734, 0.03701806525461994, 0.034277555151178016, 0.08953070588318544, 0.09100232518428673, -0.14962165764861377, -0.059780015643469187, 0.03099908787589212, 0.04205942788549194, 0.10761242098049818, -0.17898820305581692, -0.086947336363037, 0.0005744031348363303, 0.26470367015751056, -0.3410752977726659, -0.10864463848028279, -0.3237407503283789, -0.05865268557290702, -0.07352085378749369, 0.0019792050209501643, -0.08210935939172002, -0.05627388845608065, -0.17795292605499816], [-0.003744997981099421, -0.12494442721265661, 0.01213782421204565, 0.11011255598869184, 0.023441446522044638, -0.22898121207469282, -0.013626869823345072, 0.048311679078693257, 0.2323995678073435, 0.08503032764724522, 0.03229676026693013, 0.20973773176287117, -0.39534546397157005, 0.18162492838260375, -0.16784914405322665, 0.05400038424814656, 0.011955557290394474, 0.009613702033894271, -0.09859822426392142, 0.0613369449988797, -0.1453993627624006, -0.08351907266711839, -0.050881273647897456, -0.43790739806176016, -0.003744997981099421, -0.022413061391377734, 0.059950270803413544, 0.03343054467464914, -0.0004520670191639027, -0.265204732746611, -0.01769513094459745, 0.05856722122185046, 0.29126713479569205, 0.13226978312180007, 0.005347966415722569, 0.20569471417462423, -0.3362292608613774, 0.031790090288886054, 0.24114206797683124, 0.059928658114626644, -0.002620564130646887, -0.10139183626898135, -0.2043259983664543, 0.2247827350996877, 0.051963946970880866, -0.1067949200659468, -0.1395341152361871, -0.43790739806176016, -0.2013908603586764, 0.007978334472210356, 0.060789464553131625, 0.026810848599000722, -0.01619432803181889, -0.10190646643347208], [-0.08718961188559267, -0.06506333090540034, 0.046094023979193176, -0.17440558372398796, -0.11603238539809413, -0.16903843329272566, 0.05965325535097201, -0.002483838470726133, 0.12908069399934513, 0.24273573222766695, 0.028849993117731145, 0.38567841366765415, -0.46432597469680004, 0.0819980081948634, -0.014486030068465584, 0.07010620091431, 0.12769112173157196, -0.2341046194544844, -0.2398496870871647, 0.1371294604127039, -0.029732498510820434, -0.09356758871542345, -0.09941887408960437, -0.34903688320730664, -0.08718961188559267, -0.0010653990248732687, 0.049306734213960965, 0.07271574915585143, -0.08614599514860907, -0.005637664695323601, 0.08261579219131951, 0.02126602266695568, 0.1288155382433696, 0.05918205316810346, -0.015655488285239924, 0.2563170858395495, -0.3960684124897411, 0.019228858964349657, 0.12395015211496684, 0.11498631615203868, 0.10690005221933807, -0.2638790653084152, -0.3173617440749186, -0.03781179186828045, 0.03834934471643556, -0.044686533096702216, -0.17071849864274405, -0.34903688320730664, -0.12576180006011078, -0.044851364824929255, 0.10560951836809064, 0.04286074205030399, -0.09970339102498352, -0.11845048211215749], [-0.016254598608579543, -0.05971374037960699, 0.12178406186229801, -0.16289420481939548, 0.13077643257170635, 0.06965904742951479, -0.07417542047120816, -0.016526804381761226, -0.060636739086398696, -0.06233526809759892, 0.021438772959205647, -0.08633544715501326, 0.2501649044783462, 0.08052517137725675, 0.1896999628430566, 0.12491397991852599, 0.0344049944611515, -0.22275945259498597, -0.031495010646942476, -0.0672771658002176, -0.01466525703616687, -0.09864124021483932, 0.08407777616643873, 0.17767855817859549, -0.016254598608579543, 0.15122021092495994, -0.08614673419697881, 0.08044467532294877, 0.2669397126880848, 0.24081572730934472, -0.04085674024973371, 0.2242814794559583, 0.06959710685581975, 0.07006832369679673, -0.15623939292489356, 0.09026689849270375, 0.5343790889454, -0.0806749352623687, 0.09385165778590181, 0.007540861456562309, 0.1328638257811659, 0.2587503924946236, -0.0629448523899334, 0.10567039600643718, -0.06727355205614845, 0.3612063680116143, -0.09874947535710092, 0.17767855817859549, -0.4151970251065602, -0.06339538042103113, 0.44159200623513584, -0.17629482454168874, 0.2399121303625661, -0.2745317963714585], [0.019160388025096845, 0.14498029423908662, -0.13475766057274105, 0.15719895059894937, 0.06064943390650458, 0.2604807316659329, 0.02991446039611402, 0.06586877695133662, 0.06458292315299312, -0.028082559269172486, -0.09542094567995228, 0.13708471566339087, 0.5339028090291366, -0.15015998080632328, 0.036857699346818526, -0.07213039165757643, -0.02482536166846865, 0.352208632815886, 0.11349327466920256, 0.23834762803329798, 0.10261744155143412, 0.16467077451263915, -0.07364333056411053, 0.23395125647714599, 0.019160388025096845, 0.00270737855562907, 0.03359980908800876, 0.03347436084536788, 0.02214328559414322, 0.012820549461953656, 0.0006807527993404319, 0.03951461282516288, 0.046640903374398655, 0.07960921901745147, 0.03857637049624233, -0.07386888148722333, 0.20649388326203696, -0.0475197900887009, -0.030058496788778467, -0.11328333626342978, -0.1661746230407396, -0.1532804672635002, -0.10789036838097393, -0.22332656898147496, -0.026558419856741185, 0.03651830890809578, 0.03471013619446881, 0.23395125647714599, 0.2795777985774841, 0.05599136201046838, -0.3161462135072248, 0.2541175005447975, 0.09195370050861294, 0.398585503350577], [0.11027673402883688, -0.04530728001254237, 0.037723897257524626, 0.07861539009710601, -0.05160228611816166, 0.043834003799194804, -0.08460705366434923, -0.09253727176884763, -0.06918317374688285, 0.051828786911045824, -0.035522622151015745, -0.26646751332006224, 0.04811041930224573, 0.1152507536715826, -0.010366095086537404, 0.012509085526285553, 0.15856473468416166, -0.07097059146908517, 0.08313267932864114, 0.08009264834456892, 0.0035153908900325656, -0.08903404522456158, 0.15480338044881917, 0.26012209223746896, 0.11027673402883688, -0.07618676533542956, -0.01109822730315042, -0.0020801638762080058, 0.0700340068535543, 0.00039545567867383877, -0.08996510976306887, 0.014330927127282421, -0.1324269967652055, -0.10205677406473869, 0.015008181489271837, -0.11501433951912679, -0.00365709193202729, 0.11159230304080242, 0.009584844335938179, -0.010121320815830826, 0.09769508924076026, -0.06800940190450616, 0.1624312799216413, -0.10297916037837376, -0.12790243560694123, -0.0034397261690704787, 0.18880373995666347, 0.26012209223746896, 0.06102428360193465, 0.011206182872115942, -0.1193179089391054, 0.11649652252526128, 0.04703931038886742, 0.05938541686422442], [0.0817540282663904, -0.04257633659532269, -0.06906591563850993, -0.03010933708856409, 0.16130109957643166, -0.028340021343004486, -0.10875739603599165, 0.07063474446398794, 0.03583301565895917, -0.028553412548201717, 0.19326616331230323, -0.144209149640793, 0.05997835669899309, -0.12868890043608017, -0.13548292064717873, -0.046423347339868935, -0.08446176003209128, -0.2574445183578604, 0.05184340649174549, -0.23738391136731318, 0.050668248094386206, -0.13626397936952314, -0.0037065287650212596, -0.07455912951588801, 0.0817540282663904, -0.04734561003456389, -0.0496671649563151, -0.05811614807324446, -0.010555605326465583, 0.08396278540047435, -0.18777468061006597, -0.02838920553128261, -0.056119311595796945, 0.08037049688071712, 0.177850398957333, -0.093542576700271, 0.0100552459547753, -0.11832711805924973, -0.23519990555304834, -0.03155528785479805, -0.07352208253193789, -0.0678725792874533, 0.12036952789941785, 0.05394881230558881, -0.010827699551152467, -0.07620327187519725, -0.06292337636331671, -0.07455912951588801, 0.009584304351294922, 0.15332855101993398, -0.002462881209176477, -0.023273193384451003, -0.02832196825714544, 0.05862981673677674], [-0.013415251675667785, -0.013095344428741626, 0.046316731082074555, 0.07839382186799758, 0.06548858919984547, 0.09550335243477516, -0.01889109516091157, 0.15878787086963378, -0.19047062227192058, -0.1641540671436425, -0.0422648832671682, -0.2281930649186111, -0.24217539753123096, 0.06676953398209814, -0.05564912777841495, 0.08354718329942125, -0.07633312090499887, -0.05997992986502191, -0.1733654197682081, -0.03900130714403475, 0.052197045547834615, -0.08134516778230301, 0.026960022663349098, 0.01900407562229748, -0.013415251675667785, -0.02435414953406616, -0.07850405520571063, 0.10651782124363687, -0.08531728181846883, -0.08966036467510942, -0.04453552752188591, 0.08818179009585632, -0.16440733329415935, 0.06131951751569881, -0.06535390599887556, -0.12011562228291991, -0.18294677893131947, 0.11379570477059756, 0.1045063751351976, 0.1174661890471086, -0.06255194706168841, -0.05861364039876474, -0.03851445661381024, -0.07009489938279165, -0.05405166320126385, 0.013373090633155928, 0.021802831788842108, 0.01900407562229748, 0.022234416597543414, 0.049589682651240446, 0.06698170080039272, -0.02806315271352534, -0.07741877782531892, -0.08668527552755591], [0.0010411656539272619, -0.047506137180823416, -0.03891892681445266, -0.036471163692129875, -0.08548326492147755, 0.030375983032770303, -0.15329085272402324, -0.014746743651152671, -0.014052934347025968, -0.07082684535967335, 0.11051729842565076, -0.14552424675559508, -0.09154152999769861, -0.033664374224585354, -0.047784282902525614, -0.06573067288782411, 0.08086585903330297, -0.08368887368569207, 0.07933634565766552, -0.1171163563102651, 0.07199670593354451, -0.16836194419030184, 0.016107040697828753, -0.02369774842708724, 0.0010411656539272619, -0.06971769731236391, -0.006214831686689367, 0.03240549291104361, -0.016548478356469748, -0.08111100453566318, -0.11821548071929172, -0.11560332004923415, -0.06990638483789961, -0.03230680933399993, 0.14814325271292328, -0.2204240952500586, 0.0013976239525355886, -0.0475671590605535, -0.20528168258830315, -0.11726084474810371, 0.03818936440316368, -0.19069932271866483, 0.12361642586561951, -0.1657457266862387, -0.11137655011342308, -0.004058958454971471, 0.04196748988191514, -0.02369774842708724, 0.0439614190425977, -0.06594989974084448, -0.11802493345333638, 0.08041216815623047, 0.11347882753433547, -0.10713328558929437], [-0.050413126944255704, 0.011305093846459644, -0.1469617805588091, 0.01813148024941786, -0.16444850715447942, -0.1069440062712648, 0.05037043609515764, 0.08993145760414117, -0.08207768867790308, 0.004787974067718137, -0.024969796166493006, 0.005443575356944133, -0.278536340428166, 0.006620297982620197, -0.0011116426595265856, -0.028081008940564985, -0.30176729004291536, -0.1325342684174615, 0.024698089666285446, -0.09423840514932232, 0.04650201605419629, -0.0383142718427395, -0.023662284632861984, -0.06103278658343753, -0.050413126944255704, 0.01038528715722379, -0.12880592608466418, -0.04730099139943563, -0.014614081551747598, -0.1453575797786009, 0.00032959868776336564, -0.1951194780644976, -0.24517652949861124, 0.06678767029982291, -0.02163254734816326, -0.08096536365500805, -0.31335473928251756, 0.006601078390368587, 0.06433573927552536, -0.07672494834947263, -0.24819616089723984, 0.07754959644935575, -0.07555936498132147, -0.16584750455608274, 0.006107783092280698, 0.006188227579858002, -0.025710894811727395, -0.06103278658343753, 0.001761094817626297, 0.09585204312491978, -0.0030605386124274124, -0.10929940153047879, 0.10482661262791083, 0.008806604607551128], [0.13163232389722865, -0.016160143486288622, 0.0501740812852586, 0.025709058946114534, -0.011762365382437938, 0.12246339479090503, -0.02631033775581866, -0.0769501460429345, 0.0694471653891347, -0.09070980127005458, 0.0768891737791376, -0.05369190255903373, 0.06561097165879834, -0.05253971788921335, -0.07834244824801131, 0.04520123612062221, 0.18587397320703813, -0.13748793024719688, 0.14819853775881828, -0.059394040852354696, -0.03663604664272629, 0.011235243149781424, 0.13935924445036552, 0.20572755392503445, 0.13163232389722865, -0.06765440328272297, 0.04339866641827214, 0.1840955358038809, -0.029249409622879585, -0.06137720969391504, 0.03752167932524305, -0.24587837938003246, 0.055220217779286415, 0.005817216473904522, 0.010647958663468259, -0.10389467902819614, -0.015740601347484947, 0.1020786912373211, 0.10067867769560884, 0.005159528208886815, 0.14125981735707133, -0.07420129672798184, 0.06726207798480317, 0.2890032200279142, 0.0015437631849320173, -0.1241598638680272, 0.17382075810746042, 0.20572755392503445, 0.10149755459174908, -0.12266624973179521, 0.14995500961301264, 0.08829321143808798, 0.0816708779798896, 0.09232211869024162], [0.1245242368905501, 0.0012968386070699228, 0.0023859275643897163, -0.07248082195058173, 0.11731390974875852, -0.11692206622392833, -0.10791741882081347, -0.024537023900281736, -0.16166855159153765, 0.014360035813050813, 0.22443448477492703, -0.2576386321117329, 0.18683388188668792, -0.23691430344431724, -0.07045936293279151, -0.05762148882231311, -0.1324326249136224, -0.015059978267903404, 0.01816020057605848, -0.17169532476112986, 0.2469298920688544, -0.26945776590602555, 0.04846283436419168, 0.006293983011255431, 0.1245242368905501, -0.06289235206453962, -0.05916616279870233, -0.18852336086246926, 0.04849101333316331, -0.10171494996261078, -0.07650006037512296, -0.010187876963303585, 0.22898684546697878, 0.00878575435632731, 0.14856711075174467, -0.07443560313888704, 0.19080205869359804, -0.05953374267582014, -0.14735495370668333, -0.06845331719117484, -0.15572473892044136, 0.13031940330002195, -0.14431555001861782, -0.08475233947349145, -0.028915085113957645, -0.08275338785522539, 0.1017852344555814, 0.006293983011255431, 0.12642389077012384, -0.05937986976626741, 0.1426943180052439, 0.0365172072405981, 0.026814809467784705, 0.014528264676089055]], "intercept": [-0.26321616358348804, -0.31747815788074857, 0.2339801925879026, 0.03193255667602374, 0.407289548270891, 0.0882267948667739, -0.0725342024824994, -0.6525309486636928, -0.7368950355464242, -0.8312643601291542, -0.8759779404311443, 0.4036545573308553, 0.4522578131523835, 0.43433603508989843, 0.4669743150256425, 0.3519961649203457, 0.42816885382706554, 0.45107997696936786], "classes": ["claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash", "glm-5.3", "glm-5.3-flash", "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra", "kimi-k3", "qwen3.8-max"], "C": 0.05, "W": 1.0, "entropy_ref": 0.5692304418233374, "max_text_tokens": 256, "tok_per_char": 0.19383936500423704, "train_rows": 2699, "source": "Name-that-LLM experiments/exp060_glm_ll_cut.py: estopadilla\'s recipe at L256"}')


In [ ]:
from pathlib import Path
V1_TEST_CSV=os.path.join(DATA,'test.csv')
Path('/kaggle/temp/v1').mkdir(parents=True,exist_ok=True)
# GLM segment paths come from the pinned Kaggle input versions.
import json, os, shutil, subprocess, sys
from pathlib import Path
V1_DIR = Path('/kaggle/temp/v1')
V1_OUT = Path('/kaggle/working/v1')
V1_OUT.mkdir(parents=True, exist_ok=True)
GLM_SEGMENTS = {'jamesliebel/glm53-iq3xxs-segs-a-mirror': [['glm53_iq3xxs-seg00-of-19-L0-6.gguf', 12753355264], ['glm53_iq3xxs-seg01-of-19-L6-10.gguf', 15651714688], ['glm53_iq3xxs-seg02-of-19-L10-14.gguf', 14583346912], ['glm53_iq3xxs-seg03-of-19-L14-18.gguf', 14583346912], ['glm53_iq3xxs-seg04-of-19-L18-22.gguf', 14583346912], ['glm53_iq3xxs-seg05-of-19-L22-26.gguf', 14583346912], ['glm53_iq3xxs-seg06-of-19-L26-30.gguf', 14583346912], ['glm53_iq3xxs-seg07-of-19-L30-34.gguf', 14583346912], ['glm53_iq3xxs-seg08-of-19-L34-38.gguf', 14583346912], ['glm53_iq3xxs-seg09-of-19-L38-42.gguf', 14583346912]], 'jamesliebel/glm53-iq3xxs-segs-b-mirror': [['glm53_iq3xxs-seg10-of-19-L42-46.gguf', 14583346912], ['glm53_iq3xxs-seg11-of-19-L46-50.gguf', 14583346912], ['glm53_iq3xxs-seg12-of-19-L50-54.gguf', 14583346912], ['glm53_iq3xxs-seg13-of-19-L54-58.gguf', 14583346912], ['glm53_iq3xxs-seg14-of-19-L58-62.gguf', 14583346912], ['glm53_iq3xxs-seg15-of-19-L62-66.gguf', 14583346912], ['glm53_iq3xxs-seg16-of-19-L66-70.gguf', 14583346912], ['glm53_iq3xxs-seg17-of-19-L70-74.gguf', 14583346912], ['glm53_iq3xxs-seg18-of-19-L74-78.gguf', 16345433920]], 'jamesliebel/glm53-flash-iq3xxs-segs-mirror': [['flash_iq3xxs-seg00-of-06-L0-8.gguf', 14879704352], ['flash_iq3xxs-seg01-of-06-L8-16.gguf', 22864262976], ['flash_iq3xxs-seg02-of-06-L16-24.gguf', 21821175680], ['flash_iq3xxs-seg03-of-06-L24-32.gguf', 21821175680], ['flash_iq3xxs-seg04-of-06-L32-40.gguf', 21821175680], ['flash_iq3xxs-seg05-of-06-L40-45.gguf', 14407253632]]}

def pinned_dataset(owner, slug):
    namespaced = Path('/kaggle/input/datasets') / owner / slug
    flat = Path('/kaggle/input') / slug
    if namespaced.is_dir():
        return namespaced
    assert flat.is_dir(), (namespaced, flat)
    return flat

V1_FLASH, V1_GLM53 = [], []
for ref, files in GLM_SEGMENTS.items():
    owner, slug = ref.split('/')
    root = pinned_dataset(owner, slug)
    for filename, size in files:
        p = root / filename
        assert p.is_file() and p.stat().st_size == size, p
        (V1_FLASH if filename.startswith('flash_') else V1_GLM53).append(str(p))
assert len(V1_FLASH) == 6 and len(V1_GLM53) == 19
V1_FLASH.sort()
V1_GLM53.sort()
artifacts = Path('/kaggle/input/notebooks/estopadilla/ntl-glm-likelihood-train/ntl-bin')
if not (artifacts/'ntl-segrun').is_file():
    artifacts = Path('/kaggle/input/ntl-glm-likelihood-train/ntl-bin')
for filename in ('ntl-segrun','lib/libcublas.so.13','lib/libcublasLt.so.13','lib/libcudart.so.13'):
    assert (artifacts/filename).is_file(), artifacts/filename
V1_BIN = Path('/kaggle/temp/v1-bin')
V1_BIN.mkdir(exist_ok=True)
shutil.copyfile(artifacts/'ntl-segrun',V1_BIN/'ntl-segrun')
(V1_BIN/'ntl-segrun').chmod(0o755)
V1_ENV = dict(os.environ,LD_LIBRARY_PATH=str(artifacts/'lib')+':'+os.environ.get('LD_LIBRARY_PATH',''))
sys.path.insert(0,str(V1_DIR))
import segio
from v1_stage import PREFIX
smoke_texts = pd.read_csv(V1_TEST_CSV,dtype={'id':str}).text.head(3).fillna('').tolist()
smoke_root = V1_OUT
segio.write_texts(str(smoke_root/'smoke.bin'),smoke_texts)
smoke_results = {}
for tag, segments in (('flash',V1_FLASH),('glm53',V1_GLM53)):
    tokens = smoke_root/f'{tag}-smoke-tokens.bin'
    subprocess.run([str(V1_BIN/'ntl-segrun'),'tokenize','--model',segments[0],'--texts',str(smoke_root/'smoke.bin'),
                    '--out',str(tokens),'--max-text-tokens','160','--prefix',PREFIX],env=V1_ENV,check=True)
    smoke_results[tag] = str(tokens)
print('Verified 25 GLM segments, CUDA runtime and two tokenizers', flush=True)


In [ ]:
if not FAST_COMMIT:
    command=[sys.executable,str(V1_DIR/'v1_stage.py'),'--test-csv',V1_TEST_CSV,
             '--final-logits',FINAL_LOGITS,'--out',OUT,'--report',str(V1_OUT/'report.json'),
             '--bin',str(V1_BIN),'--pipeline',str(V1_DIR/'run_pipeline.py'),
             '--flash-segs',json.dumps(V1_FLASH),'--glm53-segs',json.dumps(V1_GLM53),
             '--model',str(V1_DIR/'ll_model.json'),'--work','/kaggle/temp/v1-work',
             '--outdir',str(V1_OUT),'--seq-batch','8','--ubatch-div','1']
    subprocess.run(command,env=V1_ENV,check=True)


In [ ]:
%%writefile apex_postprocess.py
"""Apex sibling residual and typography, with explicit GLM evidence routing."""
import json
import numpy as np
import pandas as pd
from scipy.special import softmax
from pngy_residual import Residual,features,refine

def aligned_logits(path,ids,labels):
    with np.load(path,allow_pickle=False) as z:
        assert z['ids'].astype(str).tolist()==list(ids), 'logits row order differs'
        saved=z['labels'].astype(str).tolist()
        assert set(saved)==set(labels) and len(saved)==len(labels)
        out=np.asarray(z['logits'],dtype=np.float64)[:,[saved.index(x) for x in labels]]
    assert out.shape==(len(ids),len(labels)) and np.isfinite(out).all()
    return out

def apply_arrays(base,evidence,labels,families,texts,current,strength=.10,corrected=False,penalty=0.,post=None):
    refined=refine(base,evidence,families,strength=strength)
    predicted=np.asarray(labels)[refined.argmax(1)]
    out=np.asarray(current,dtype=object).copy()
    if corrected:
        out=predicted.astype(object)
    else:
        keep=np.isin(out,['glm-5.3','glm-5.3-flash'])
        out[~keep]=predicted[~keep]
    probs=softmax(refined,axis=1)
    glm_winner = np.isin(out, ['glm-5.3', 'glm-5.3-flash'])
    if glm_winner.any():
        probs[glm_winner] = softmax(post[glm_winner], axis=1)
    allowed={'claude-opus-4-6','claude-opus-5','glm-5.3','glm-5.3-flash'}
    rules=[(' — ',np.array([x not in allowed for x in labels])),
           ('’',np.array([x.startswith(('claude-','glm-')) for x in labels])),
           ('—',np.array([x.startswith('gemini-') for x in labels]))]
    lookup={x:i for i,x in enumerate(labels)}
    for i,text in enumerate(texts):
        for marker,forbidden in rules:
            if marker in text and forbidden[lookup[out[i]]]:
                probs[i,forbidden]*=penalty
                if probs[i].sum()>0:
                    probs[i]/=probs[i].sum()
                    out[i]=labels[probs[i].argmax()]
    return out,refined

def apply_postprocessing(train,query,base_file,glm_file,output,strength=.10,corrected=False,penalty=0.):
    labels=sorted(train.model.unique()); ids=query.id.astype(str).tolist()
    current=pd.read_parquet(output)
    assert current.id.astype(str).tolist()==ids and current.model.isin(labels).all()
    base=aligned_logits(glm_file if corrected else base_file,ids,labels)
    post=aligned_logits(glm_file,ids,labels)
    model=Residual(labels).fit(train.text.fillna('').astype(str),train.model)
    x=features(query.text.fillna('').astype(str)); evidence=np.ones_like(base)
    for columns,estimator in model.models:
        prediction=estimator.predict_proba(x)
        for j,label in enumerate(estimator.classes_):evidence[:,labels.index(label)]=prediction[:,j]
    predictions,_=apply_arrays(base,evidence,labels,model.families,query.text.fillna('').astype(str).tolist(),
                               current.model.to_numpy(),strength,corrected,penalty,post=post)
    result=pd.DataFrame({'id':ids,'model':predictions})
    assert result.id.is_unique and result.model.isin(labels).all()
    result.to_parquet(output,index=False)
    assert pd.read_parquet(output).equals(result)
    print(json.dumps({'stage':'apex','strength':strength,'post_glm_evidence':corrected,
                      'forbidden_probability_multiplier':penalty,
                      'changed':int((predictions!=current.model.to_numpy()).sum())}),flush=True)
    return result


## Credits and references

### Ensemble lineage and inference

- **Alex Chilton** assembled the public attachments and reproduced the [scored 0.701 Apex release](https://www.kaggle.com/code/alexchilton/name-that-llm-0-702-all-public-what-failed?scriptVersionId=370092394). His [Llama 3.1 checkpoint attachment](https://www.kaggle.com/models/alexchilton/llama-3-1-8b-base) provides the original Meta base used by the Llama reader.
- **LumiBee** supplied the [Apex v44 sibling residual and typographic rules](https://www.kaggle.com/code/lumibee/ntl-kiro-v44-apex-grandmaster). The residual adjusts relative scores among models in the same family while preserving their total probability.
- **Ertuğrul Özer (`ozertuu`)** developed the Apex ensemble line. **`lbjyang`** made the [0.684 public mirror](https://www.kaggle.com/code/lbjyang/ntl-684-ozertuu-mirror). **`feiyu26`** combined that stack with the likelihood stage in the [0.696 graft](https://www.kaggle.com/code/feiyu26/ntl-0684-v1-graft?scriptVersionId=369151207), and **Areeb Ali** reproduced that graft with [input and mount-order checks](https://www.kaggle.com/code/areebbb/0-696-this-actually-works?scriptVersionId=369569661).
- **José Padilla (`estopadilla`)** built [the GLM likelihood stage and its runner](https://www.kaggle.com/code/estopadilla/0-689-real-glm-5-3-likelihood-on-2xt4) and [trained the likelihood classifier](https://www.kaggle.com/code/estopadilla/ntl-glm-likelihood-train). **`jamesliebel`** published the [GLM-5.3 segment A](https://www.kaggle.com/datasets/jamesliebel/glm53-iq3xxs-segs-a-mirror), [segment B](https://www.kaggle.com/datasets/jamesliebel/glm53-iq3xxs-segs-b-mirror) and [Flash segments](https://www.kaggle.com/datasets/jamesliebel/glm53-flash-iq3xxs-segs-mirror). The base model is from [Z.ai](https://huggingface.co/zai-org) and the runner uses [llama.cpp / ggml](https://github.com/ggml-org/llama.cpp).

### Trained readers and data

- **CHEN Han (`concyclics`)** developed the [multi-depth Qwen3-Embedding-8B expert](https://www.kaggle.com/code/concyclics/0-679?scriptVersionId=365653660) and supplied its five trained fold states.
- **Dhruv Tibarewal** supplied the [Qwen and Llama L24 adapters](https://www.kaggle.com/datasets/dhruvtibarewal/nlm-l24-adapters-v5), [matched-text pool](https://www.kaggle.com/datasets/dhruvtibarewal/nlm-matched-clean), and [offline wheels](https://www.kaggle.com/datasets/dhruvtibarewal/nlm-offline-wheels).
- **Nuri Dudak** supplied the [Qwen3-14B v4 adapters](https://www.kaggle.com/datasets/nuridudak/ntl-q14-lora-v4). **Nishant Dahal** released [two-stage LoRA weights](https://www.kaggle.com/datasets/nishantdahal/ntl-two-stage-lora). **`vnton37nguyn`** trained the [Qwen2.5 in-style adapter](https://www.kaggle.com/code/vnton37nguyn/ntl-train-lora-qwen25-instyle).
- **Lucas Boesen** supplied the [encoder extraction notebook and encoder outputs](https://www.kaggle.com/code/lucasboesen/name-that-llm-extract-encoders2). **`crtproduction`** published the [Open Pool](https://www.kaggle.com/datasets/crtproduction/name-that-llm-open-pool) and the flags identifying passages regenerated from or close to training prompts.
- [Qwen base models and tokenizers](https://www.kaggle.com/organizations/qwen-lm) are from the Qwen team. Adapter, corpus, checkpoint and library licenses remain with their respective releases. [Competition data and citation](https://www.kaggle.com/competitions/name-that-llm/data).

### Experiment reference

- **Flexonafft** proposed including Claude Fable in the [V46 GLM selection ranking](https://www.kaggle.com/code/flexonafft/ntl-kiro-v44-reproduction-residual-after-glm/versions). This variant uses a capped partial Fable contribution, keeping the original GLM likelihood work limits.

### Calibration reference

- **Ertuğrul Özer** explored passing likelihood logits to Apex in [V69–V70](https://www.kaggle.com/code/ozertuu/residual-sibling-gating-apex-better/versions). Here the newer probabilities are used solely when the last typographic stage considers a GLM winner.


In [ ]:
if not FAST_COMMIT:
    from apex_postprocess import apply_postprocessing
    train_df=pd.read_csv(os.path.join(DATA,'train.csv'))
    query_df=pd.read_csv(V1_TEST_CSV,dtype={'id':str})
    final=apply_postprocessing(train_df,query_df,FINAL_LOGITS,str(V1_OUT/'post-glm.npz'),OUT,
                              strength=0.1, corrected=False, penalty=0.15)
else:
    final=pd.read_parquet(OUT)
assert final.id.astype(str).tolist()==pd.read_csv(V1_TEST_CSV,dtype={'id':str}).id.tolist()
assert final.columns.tolist()==['id','model'] and final.id.is_unique
display(final.head())
